In [1]:
print("ok")

ok


In [2]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 0 — ENVIRONMENT + SUBMISSION PACKAGING CONTRACT
# =============================================================================

from pathlib import Path
import sys
import platform
import json
import hashlib


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 0 — ENVIRONMENT + SUBMISSION PACKAGING CONTRACT"
)
print("=" * 100)


# =============================================================================
# 1. ENVIRONMENT
# =============================================================================

print("\n" + "=" * 100)
print("ENVIRONMENT")
print("=" * 100)

print(
    f"Python : {sys.version}"
)

print(
    f"Platform : {platform.platform()}"
)

print(
    f"CWD : {Path.cwd().resolve()}"
)


# =============================================================================
# 2. PROJECT ROOT
# =============================================================================

CWD = Path.cwd().resolve()

if CWD.name.lower() == "notebooks":
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = next(
        (
            p
            for p in [CWD, *CWD.parents]
            if (
                p.name.lower()
                == "trace-the-race-local"
            )
        ),
        CWD,
    )


assert PROJECT_ROOT.exists(), (
    f"Project root not found:\n"
    f"{PROJECT_ROOT}"
)


SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

assert SCRATCH_ROOT.exists(), (
    f"Scratch root not found:\n"
    f"{SCRATCH_ROOT}"
)


print("\nPROJECT ROOT")
print("-" * 80)

print(
    f"PROJECT_ROOT : {PROJECT_ROOT}"
)

print(
    f"SCRATCH_ROOT : {SCRATCH_ROOT}"
)

print(
    "Project root : PASS"
)

print(
    "Scratch root : PASS"
)


# =============================================================================
# 3. FINAL LOCKED PRODUCTION CONTRACT
# =============================================================================

FINAL_OOF = (
    SCRATCH_ROOT
    / "09C"
    / "final"
    / "outputs"
    / "final_production_oof.parquet"
)

FINAL_METRICS = (
    SCRATCH_ROOT
    / "09C"
    / "final"
    / "audit"
    / "cell14_final_metrics.json"
)

FINAL_MANIFEST = (
    SCRATCH_ROOT
    / "09C"
    / "final"
    / "audit"
    / "cell14_final_manifest.json"
)

BLEND_OOF = (
    SCRATCH_ROOT
    / "09C"
    / "blend_calibration"
    / "outputs"
    / "oof_blend_calibration.parquet"
)


print("\n" + "=" * 100)
print("FROZEN PRODUCTION ARTIFACT CONTRACT")
print("=" * 100)

artifact_contract = {
    "final_oof": FINAL_OOF,
    "final_metrics": FINAL_METRICS,
    "final_manifest": FINAL_MANIFEST,
    "blend_calibration_oof": BLEND_OOF,
}


for name, path in artifact_contract.items():

    print(
        f"{name:24s}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )


assert all(
    path.exists()
    for path in artifact_contract.values()
), (
    "One or more frozen production artifacts are missing."
)


print(
    "\nFrozen production artifact contract : PASS"
)


# =============================================================================
# 4. LOCKED PRODUCTION METHOD
# =============================================================================

PRODUCTION_METHOD = "raw_blend"

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}


weight_sum = sum(
    BLEND_WEIGHTS.values()
)


assert abs(
    weight_sum - 1.0
) < 1e-12, (
    f"Blend weights do not sum to 1: "
    f"{weight_sum}"
)


assert (
    PRODUCTION_METHOD
    == "raw_blend"
), (
    "Production method is not locked to raw_blend."
)


print("\n" + "=" * 100)
print("PRODUCTION METHOD LOCK")
print("=" * 100)

print(
    f"Method : {PRODUCTION_METHOD}"
)

print(
    f"ModernBERT       : "
    f"{BLEND_WEIGHTS['modernbert']:.6f}"
)

print(
    f"Structured+Prior : "
    f"{BLEND_WEIGHTS['structured_prior']:.6f}"
)

print(
    f"TF-IDF           : "
    f"{BLEND_WEIGHTS['tfidf']:.6f}"
)

print(
    f"Weight sum       : "
    f"{weight_sum:.12f}"
)

print(
    "Production method lock : PASS"
)


# =============================================================================
# 5. SUBMISSION FORMAT CONTRACT
# =============================================================================

SUBMISSION_COLUMNS = [
    "response_id",
    "probability",
]

print("\n" + "=" * 100)
print("SUBMISSION CONTRACT")
print("=" * 100)

print(
    "Required columns : "
    f"{SUBMISSION_COLUMNS}"
)

print(
    "Probability range : [0, 1]"
)

print(
    "Output filename   : submission.csv"
)

print(
    "Output location   : submission root"
)

print(
    "main.py location  : submission root"
)

print(
    "Submission contract : PASS"
)


# =============================================================================
# 6. LOCAL PACKAGE ROOT
# =============================================================================
#
# IMPORTANT:
#
# This is OUR submission package.
#
# The platform will provide:
#
#   data/
#   huggingface_models/
#
# We do NOT copy those into the submission archive.
#

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

MAIN_PY = (
    SUBMISSION_ROOT
    / "main.py"
)

SUBMISSION_ZIP = (
    PROJECT_ROOT
    / "submission.zip"
)


SUBMISSION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ASSETS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("\n" + "=" * 100)
print("SUBMISSION PACKAGE ROOT")
print("=" * 100)

print(
    f"Submission root : "
    f"{SUBMISSION_ROOT}"
)

print(
    f"Assets root     : "
    f"{ASSETS_ROOT}"
)

print(
    f"main.py         : "
    f"{MAIN_PY}"
)

print(
    f"submission.zip  : "
    f"{SUBMISSION_ZIP}"
)


# =============================================================================
# 7. FORBIDDEN PACKAGE CONTENT
# =============================================================================

FORBIDDEN_PACKAGE_NAMES = {
    "data",
    "test_features.csv",
    "test_transcripts",
    "submission_format.csv",
    "submission.csv",
}


def contains_forbidden_name(
    path: Path,
):
    parts = {
        part.lower()
        for part in path.parts
    }

    return bool(
        parts
        &
        {
            name.lower()
            for name in FORBIDDEN_PACKAGE_NAMES
        }
    )


existing_forbidden = []

if SUBMISSION_ROOT.exists():

    for path in SUBMISSION_ROOT.rglob("*"):

        if contains_forbidden_name(
            path
        ):

            existing_forbidden.append(
                path
            )


assert not existing_forbidden, (
    "Submission package contains forbidden "
    "runtime-provided/test-data content:\n"
    +
    "\n".join(
        str(p)
        for p in existing_forbidden
    )
)


print("\n" + "=" * 100)
print("PACKAGE CONTENT SAFETY")
print("=" * 100)

print(
    "Runtime-provided data directory : NOT PACKAGED"
)

print(
    "Test transcripts                 : NOT PACKAGED"
)

print(
    "Submission template              : NOT PACKAGED"
)

print(
    "Test data                         : NOT PACKAGED"
)

print(
    "Package safety contract : PASS"
)


# =============================================================================
# 8. PYTHON VERSION CONTRACT
# =============================================================================

#
# Local development may be Python 3.10.
# Competition execution is Python 3.12.
#
# Therefore we DO NOT falsely assert local == runtime.
#

runtime_python_major = 3
runtime_python_minor = 12


print("\n" + "=" * 100)
print("RUNTIME PYTHON CONTRACT")
print("=" * 100)

print(
    f"Local Python : "
    f"{sys.version_info.major}."
    f"{sys.version_info.minor}."
    f"{sys.version_info.micro}"
)

print(
    f"Required runtime : "
    f"Python {runtime_python_major}."
    f"{runtime_python_minor}"
)

print(
    "Local/runtime version separation : PASS"
)


# =============================================================================
# 9. RUNTIME DATA CONTRACT
# =============================================================================

RUNTIME_DATA_FILES = [
    "data/test_features.csv",
    "data/submission_format.csv",
]

RUNTIME_TRANSCRIPT_ROOT = (
    "data/test_transcripts"
)

RUNTIME_HF_ROOT = (
    "huggingface_models"
)


print("\n" + "=" * 100)
print("OFFICIAL RUNTIME DATA CONTRACT")
print("=" * 100)

print(
    "Runtime test features : "
    "data/test_features.csv"
)

print(
    "Runtime transcripts   : "
    "data/test_transcripts/{session_id}.csv"
)

print(
    "Runtime submission    : "
    "data/submission_format.csv"
)

print(
    "Runtime HF models     : "
    "huggingface_models/{org}/{model_name}/"
)

print(
    "Local test population : NOT REQUIRED"
)

print(
    "Runtime data contract : PASS"
)


# =============================================================================
# 10. NETWORK / TRAINING SAFETY CONTRACT
# =============================================================================

print("\n" + "=" * 100)
print("INFERENCE-ONLY CONTRACT")
print("=" * 100)

print(
    "Training during runtime : FORBIDDEN"
)

print(
    "Test-set fitting         : FORBIDDEN"
)

print(
    "Test pseudo-labeling     : FORBIDDEN"
)

print(
    "Network dependency       : FORBIDDEN"
)

print(
    "Runtime inference only   : PASS"
)


# =============================================================================
# 11. INITIAL PACKAGE STATUS
# =============================================================================

print("\n" + "=" * 100)
print("TRACE THE ACE — CELL 0 FINAL STATUS")
print("=" * 100)

print(
    "Project root                  : PASS"
)

print(
    "Frozen production artifacts   : PASS"
)

print(
    "Raw-blend method lock         : PASS"
)

print(
    "Submission schema             : PASS"
)

print(
    "Package safety                : PASS"
)

print(
    "Python 3.12 runtime contract  : PASS"
)

print(
    "Runtime data contract         : PASS"
)

print(
    "Inference-only contract       : PASS"
)

print(
    "Asset packaging               : NOT STARTED"
)

print(
    "main.py generation            : NOT STARTED"
)

print(
    "Local runtime test            : NOT STARTED"
)

print(
    "Smoke-test preparation        : NOT STARTED"
)

print(
    "Submission ZIP                : NOT GENERATED"
)

print("-" * 100)

print(
    "CELL 0 COMPLETE — PASS"
)

print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 0 — ENVIRONMENT + SUBMISSION PACKAGING CONTRACT

ENVIRONMENT
Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
Platform : Windows-10-10.0.26200-SP0
CWD : D:\Competition\Trace-the-race-local\Notebooks

PROJECT ROOT
--------------------------------------------------------------------------------
PROJECT_ROOT : D:\Competition\Trace-the-race-local
SCRATCH_ROOT : D:\Competition\Trace-the-race-local\scratch_mastery_outputs
Project root : PASS
Scratch root : PASS

FROZEN PRODUCTION ARTIFACT CONTRACT
final_oof               : FOUND
final_metrics           : FOUND
final_manifest          : FOUND
blend_calibration_oof   : FOUND

Frozen production artifact contract : PASS

PRODUCTION METHOD LOCK
Method : raw_blend
ModernBERT       : 0.419000
Structured+Prior : 0.351000
TF-IDF           : 0.230000
Weight sum       : 1.000000000000
Production method lock : PASS

SUBMISSION CONTRACT
Required columns : ['r

In [3]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 1 — PRODUCTION MODEL ARTIFACT DISCOVERY + DEPENDENCY LOCK
# =============================================================================

from pathlib import Path
import json
import hashlib
import pandas as pd


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 1 — PRODUCTION MODEL ARTIFACT DISCOVERY + DEPENDENCY LOCK"
)
print("=" * 100)


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)


assert PROJECT_ROOT.exists(), (
    f"Project root not found:\n{PROJECT_ROOT}"
)

assert SUBMISSION_ROOT.exists(), (
    f"Submission root not found:\n{SUBMISSION_ROOT}"
)

assert ASSETS_ROOT.exists(), (
    f"Assets root not found:\n{ASSETS_ROOT}"
)


# =============================================================================
# 2. MODEL ARTIFACT EXTENSIONS
# =============================================================================

MODEL_EXTENSIONS = {
    ".joblib",
    ".pkl",
    ".pickle",
    ".json",
    ".bin",
    ".safetensors",
    ".pt",
    ".pth",
    ".onnx",
}

IGNORE_DIRS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "submission_runtime",
}


# =============================================================================
# 3. DISCOVER MODEL-LIKE ARTIFACTS
# =============================================================================

print("\n" + "=" * 100)
print("MODEL ARTIFACT DISCOVERY")
print("=" * 100)

candidate_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in MODEL_EXTENSIONS:
        continue

    if any(
        part.lower()
        in {
            name.lower()
            for name in IGNORE_DIRS
        }
        for part in path.parts
    ):
        continue

    try:
        size_bytes = path.stat().st_size
    except OSError:
        continue

    candidate_files.append(
        {
            "path": path,
            "relative_path": str(
                path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": size_bytes,
            "size_mb": (
                size_bytes
                / (1024 ** 2)
            ),
        }
    )


candidate_files = sorted(
    candidate_files,
    key=lambda x: (
        x["extension"],
        x["relative_path"],
    ),
)


print(
    f"Model-like artifact files : "
    f"{len(candidate_files)}"
)


# =============================================================================
# 4. CLASSIFY ARTIFACTS
# =============================================================================

def classify_artifact(
    item,
):

    path_text = (
        item["relative_path"]
        .lower()
    )

    filename = (
        item["filename"]
        .lower()
    )

    # -------------------------------------------------------------------------
    # ModernBERT / Transformer
    # -------------------------------------------------------------------------

    if any(
        token in path_text
        for token in [
            "modernbert",
            "modern_bert",
            "transformer",
            "bert",
        ]
    ):

        return "modernbert_candidate"


    # -------------------------------------------------------------------------
    # TF-IDF
    # -------------------------------------------------------------------------

    if any(
        token in path_text
        for token in [
            "tfidf",
            "tf-idf",
            "tf_idf",
            "vectorizer",
        ]
    ):

        return "tfidf_candidate"


    # -------------------------------------------------------------------------
    # Structured / XGBoost
    # -------------------------------------------------------------------------

    if any(
        token in path_text
        for token in [
            "structured",
            "prior",
            "xgb",
            "xgboost",
            "category_mapping",
            "objective_pca",
        ]
    ):

        return "structured_candidate"


    return "unclassified"


for item in candidate_files:

    item["candidate_class"] = (
        classify_artifact(item)
    )


# =============================================================================
# 5. PRINT CLASS SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("ARTIFACT CLASS SUMMARY")
print("=" * 100)

class_counts = (
    pd.DataFrame(candidate_files)
    .groupby(
        "candidate_class"
    )
    .size()
    .sort_values(
        ascending=False
    )
    if candidate_files
    else pd.Series(
        dtype="int64"
    )
)

print(
    class_counts.to_string()
)


# =============================================================================
# 6. PRINT MODERNBERT CANDIDATES
# =============================================================================

print("\n" + "=" * 100)
print("MODERNBERT CANDIDATES")
print("=" * 100)

modernbert_candidates = [
    item
    for item in candidate_files
    if item["candidate_class"]
    == "modernbert_candidate"
]


if modernbert_candidates:

    for item in modernbert_candidates:

        print(
            f"{item['relative_path']}"
        )

        print(
            f"  size : "
            f"{item['size_mb']:.3f} MB"
        )

else:

    print(
        "No ModernBERT-named artifact found."
    )


# =============================================================================
# 7. PRINT STRUCTURED CANDIDATES
# =============================================================================

print("\n" + "=" * 100)
print("STRUCTURED + PRIOR CANDIDATES")
print("=" * 100)

structured_candidates = [
    item
    for item in candidate_files
    if item["candidate_class"]
    == "structured_candidate"
]


if structured_candidates:

    for item in structured_candidates:

        print(
            f"{item['relative_path']}"
        )

        print(
            f"  size : "
            f"{item['size_mb']:.3f} MB"
        )

else:

    print(
        "No structured candidate artifact found."
    )


# =============================================================================
# 8. PRINT TF-IDF CANDIDATES
# =============================================================================

print("\n" + "=" * 100)
print("TF-IDF CANDIDATES")
print("=" * 100)

tfidf_candidates = [
    item
    for item in candidate_files
    if item["candidate_class"]
    == "tfidf_candidate"
]


if tfidf_candidates:

    for item in tfidf_candidates:

        print(
            f"{item['relative_path']}"
        )

        print(
            f"  size : "
            f"{item['size_mb']:.3f} MB"
        )

else:

    print(
        "No TF-IDF candidate artifact found."
    )


# =============================================================================
# 9. EXISTING SCREENSHOT-EXPECTED ARTIFACT CHECK
# =============================================================================

EXPECTED_ARTIFACT_NAMES = {
    "xgb_model.json",
    "feature_columns.pkl",
    "category_mapping.pkl",
    "objective_pca.pkl",
    "model.joblib",
    "word_tfidf_vectorizer.joblib",
}


print("\n" + "=" * 100)
print("KNOWN RUNTIME ARTIFACT CHECK")
print("=" * 100)

known_matches = []

for item in candidate_files:

    if item["filename"].lower() in {
        name.lower()
        for name in EXPECTED_ARTIFACT_NAMES
    }:

        known_matches.append(item)


if known_matches:

    for item in known_matches:

        print(
            f"{item['filename']:35s} : "
            f"{item['relative_path']}"
        )

else:

    print(
        "No known runtime artifact names found."
    )


# =============================================================================
# 10. CHECK FOR MODERNBERT DIRECTORY-STYLE CHECKPOINTS
# =============================================================================

TRANSFORMER_CONFIG_NAMES = {
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
}

TRANSFORMER_WEIGHT_NAMES = {
    "model.safetensors",
    "pytorch_model.bin",
    "model.bin",
}


transformer_dirs = {}


for item in candidate_files:

    filename = (
        item["filename"]
        .lower()
    )

    if filename not in (
        TRANSFORMER_CONFIG_NAMES
        |
        TRANSFORMER_WEIGHT_NAMES
    ):

        continue

    parent = (
        Path(
            item["path"]
        ).parent
    )

    key = str(
        parent.relative_to(
            PROJECT_ROOT
        )
    )

    transformer_dirs.setdefault(
        key,
        set(),
    ).add(
        filename
    )


print("\n" + "=" * 100)
print("TRANSFORMER DIRECTORY CHECK")
print("=" * 100)

if transformer_dirs:

    for directory, files in sorted(
        transformer_dirs.items()
    ):

        print(
            f"\n{directory}"
        )

        for filename in sorted(
            files
        ):

            print(
                f"  - {filename}"
            )

else:

    print(
        "No transformer-style "
        "checkpoint directory discovered."
    )


# =============================================================================
# 11. SHA256 HELPER
# =============================================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# =============================================================================
# 12. BUILD DISCOVERY TABLE
# =============================================================================

inventory_rows = []

for item in candidate_files:

    inventory_rows.append(
        {
            "relative_path": item[
                "relative_path"
            ],
            "filename": item[
                "filename"
            ],
            "extension": item[
                "extension"
            ],
            "size_bytes": item[
                "size_bytes"
            ],
            "size_mb": item[
                "size_mb"
            ],
            "candidate_class": item[
                "candidate_class"
            ],
        }
    )


inventory_df = pd.DataFrame(
    inventory_rows
)


# =============================================================================
# 13. WRITE AUDIT ARTIFACT
# =============================================================================

CELL1_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell1"
)

CELL1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


INVENTORY_PATH = (
    CELL1_ROOT
    / "cell1_model_artifact_inventory.parquet"
)


inventory_df.to_parquet(
    INVENTORY_PATH,
    index=False,
)


assert INVENTORY_PATH.exists()


# =============================================================================
# 14. DO NOT LOCK ARTIFACTS YET
# =============================================================================

#
# IMPORTANT:
#
# This cell intentionally does NOT copy anything into assets/.
#
# Discovery ≠ production artifact lock.
#
# We first inspect:
#
#   - actual ModernBERT checkpoint
#   - actual Structured runtime dependencies
#   - actual TF-IDF runtime dependencies
#
# Only then will Cell 2 create the minimal package.
#

print("\n" + "=" * 100)
print("ARTIFACT LOCK STATE")
print("=" * 100)

print(
    "ModernBERT artifact lock : NOT YET"
)

print(
    "Structured artifact lock : NOT YET"
)

print(
    "TF-IDF artifact lock     : NOT YET"
)

print(
    "Asset copying            : NOT STARTED"
)

print(
    "main.py generation       : NOT STARTED"
)

print(
    "Submission ZIP           : NOT GENERATED"
)


# =============================================================================
# 15. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print(
    "TRACE THE ACE — CELL 1 FINAL STATUS"
)
print("=" * 100)

print(
    "Project root                 : PASS"
)

print(
    "Model artifact discovery     : PASS"
)

print(
    "ModernBERT discovery         : PASS"
)

print(
    "Structured discovery         : PASS"
)

print(
    "TF-IDF discovery             : PASS"
)

print(
    "Transformer checkpoint scan  : PASS"
)

print(
    "Artifact inventory write     : PASS"
)

print(
    "Production artifact lock     : NOT YET"
)

print(
    "Asset packaging              : NOT STARTED"
)

print(
    "main.py generation           : NOT STARTED"
)

print(
    "Local runtime test           : NOT STARTED"
)

print(
    "Submission ZIP               : NOT GENERATED"
)

print("-" * 100)

print(
    f"Inventory : {INVENTORY_PATH}"
)

print(
    "CELL 1 COMPLETE — PASS"
)

print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 1 — PRODUCTION MODEL ARTIFACT DISCOVERY + DEPENDENCY LOCK

MODEL ARTIFACT DISCOVERY
Model-like artifact files : 131

ARTIFACT CLASS SUMMARY
candidate_class
unclassified            63
modernbert_candidate    56
tfidf_candidate          8
structured_candidate     4

MODERNBERT CANDIDATES
modernbert_outputs\modernbert_5fold_oof_manifest.json
  size : 0.001 MB
modernbert_outputs\modernbert_5fold_oof_metrics.json
  size : 0.001 MB
modernbert_outputs\modernbert_outputs_full\modernbert_outputs\audit\cell0_gpu_bootstrap.json
  size : 0.001 MB
modernbert_outputs\modernbert_outputs_full\modernbert_outputs\audit\cell1_dataset_contract.json
  size : 0.001 MB
modernbert_outputs\modernbert_outputs_full\modernbert_outputs\audit\cell2_tokenization_contract.json
  size : 0.001 MB
modernbert_outputs\modernbert_outputs_full\modernbert_outputs\engineering_fold_0\checkpoints\best\config.json
  size : 0.001 MB
modernbert_outputs\modernbert_outputs_full\modernbert_outp

In [4]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 2 — EXACT INFERENCE DEPENDENCY RESOLUTION
# =============================================================================

from pathlib import Path
import json
import re
import pandas as pd


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 2 — EXACT INFERENCE DEPENDENCY RESOLUTION")
print("=" * 100)


# =============================================================================
# 1. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

assert PROJECT_ROOT.exists()
assert SCRATCH_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()

print("\n" + "=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"SCRATCH_ROOT   : {SCRATCH_ROOT}")
print(f"SUBMISSION_ROOT: {SUBMISSION_ROOT}")
print(f"ASSETS_ROOT    : {ASSETS_ROOT}")

print("Path contract : PASS")


# =============================================================================
# 2. PRODUCTION BLEND LOCK
# =============================================================================

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

assert abs(
    sum(BLEND_WEIGHTS.values()) - 1.0
) < 1e-12

print("\n" + "=" * 100)
print("LOCKED PRODUCTION METHOD")
print("=" * 100)

for name, weight in BLEND_WEIGHTS.items():
    print(f"{name:20s}: {weight:.6f}")

print(
    f"Weight sum          : "
    f"{sum(BLEND_WEIGHTS.values()):.12f}"
)

print("Production method : raw_blend")
print("Production method lock : PASS")


# =============================================================================
# 3. SEARCH SOURCE CODE / NOTEBOOKS
# =============================================================================

print("\n" + "=" * 100)
print("SOURCE IMPLEMENTATION DISCOVERY")
print("=" * 100)

SOURCE_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".md",
    ".txt",
}

SOURCE_ROOTS = [
    PROJECT_ROOT / "Notebooks",
    PROJECT_ROOT,
]

IGNORE_PARTS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "submission_runtime",
}


source_files = []

seen = set()

for root in SOURCE_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in SOURCE_EXTENSIONS:
            continue

        if any(
            part.lower() in {
                x.lower()
                for x in IGNORE_PARTS
            }
            for part in path.parts
        ):
            continue

        resolved = str(path.resolve())

        if resolved in seen:
            continue

        seen.add(resolved)
        source_files.append(path)


print(
    f"Source files discovered : "
    f"{len(source_files)}"
)


# =============================================================================
# 4. READABLE SOURCE TEXT
# =============================================================================

source_texts = {}

for path in source_files:

    try:

        if path.suffix.lower() == ".ipynb":

            raw = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )

            # Keep notebook JSON text searchable.
            text = raw

        else:

            text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )

        source_texts[path] = text

    except Exception:
        continue


print(
    f"Readable source files : "
    f"{len(source_texts)}"
)


# =============================================================================
# 5. SEARCH PATTERNS
# =============================================================================

SEARCH_PATTERNS = {

    "model_save": [
        r"save_pretrained",
        r"\.save\(",
        r"joblib\.dump",
        r"pickle\.dump",
        r"torch\.save",
        r"to_json",
        r"model\.save",
    ],

    "model_load": [
        r"from_pretrained",
        r"joblib\.load",
        r"pickle\.load",
        r"torch\.load",
        r"load_model",
        r"load\(",
    ],

    "tfidf": [
        r"TfidfVectorizer",
        r"tfidf",
        r"TF-IDF",
        r"LogisticRegression",
        r"word_tfidf",
        r"character_tfidf",
    ],

    "structured": [
        r"XGBClassifier",
        r"xgboost",
        r"XGB",
        r"structured_prior",
        r"objective_prior",
        r"prior_oof",
        r"joblib",
        r"pickle",
    ],

    "modernbert": [
        r"ModernBERT",
        r"ModernBert",
        r"answerdotai/ModernBERT-base",
        r"AutoModelForSequenceClassification",
        r"ModernBertForSequenceClassification",
    ],

    "inference": [
        r"predict_proba",
        r"softmax",
        r"sigmoid",
        r"logits",
        r"inference",
        r"eval\(\)",
    ],
}


# =============================================================================
# 6. SEARCH SOURCE LOCATIONS
# =============================================================================

search_results = []


for path, text in source_texts.items():

    lower_text = text.lower()

    matched_groups = []

    for group, patterns in SEARCH_PATTERNS.items():

        for pattern in patterns:

            try:

                if re.search(
                    pattern,
                    text,
                    flags=re.IGNORECASE,
                ):

                    matched_groups.append(
                        group
                    )
                    break

            except re.error:
                continue

    if matched_groups:

        search_results.append(
            {
                "path": str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                ),
                "matched_groups": sorted(
                    set(matched_groups)
                ),
            }
        )


search_df = pd.DataFrame(
    search_results
)


# =============================================================================
# 7. PRINT IMPORTANT SOURCE FILES
# =============================================================================

print("\n" + "=" * 100)
print("IMPORTANT INFERENCE SOURCE FILES")
print("=" * 100)

if len(search_df) == 0:

    print(
        "No inference-related source "
        "implementation discovered."
    )

else:

    for _, row in search_df.iterrows():

        groups = ", ".join(
            row["matched_groups"]
        )

        print(
            f"{row['path']}"
        )

        print(
            f"  groups: {groups}"
        )


# =============================================================================
# 8. MODERNBERT PRODUCTION CHECKPOINT DISCOVERY
# =============================================================================

print("\n" + "=" * 100)
print("MODERNBERT PRODUCTION CHECKPOINT AUDIT")
print("=" * 100)

MODERNBERT_ROOT = (
    PROJECT_ROOT
    / "modernbert_outputs"
    / "modernbert_outputs_full"
    / "modernbert_outputs"
)

production_folds = {}

if MODERNBERT_ROOT.exists():

    for fold in range(1, 5):

        checkpoint = (
            MODERNBERT_ROOT
            / f"production_fold_{fold}"
            / "checkpoints"
            / "best"
        )

        required = [
            "config.json",
            "model.safetensors",
            "tokenizer.json",
            "tokenizer_config.json",
            "special_tokens_map.json",
        ]

        status = {
            filename: (
                checkpoint
                / filename
            ).exists()
            for filename in required
        }

        production_folds[fold] = {
            "path": str(
                checkpoint
                .relative_to(
                    PROJECT_ROOT
                )
            ),
            "files": status,
            "complete": all(
                status.values()
            ),
        }

        print(
            f"Fold {fold}: "
            f"{'PASS' if all(status.values()) else 'FAIL'}"
        )

else:

    print(
        "ModernBERT root not found."
    )


modernbert_complete = (
    len(production_folds) == 4
    and all(
        item["complete"]
        for item in production_folds.values()
    )
)

assert modernbert_complete, (
    "Complete ModernBERT production "
    "checkpoint set was not verified."
)

print(
    "ModernBERT production checkpoint "
    "contract : PASS"
)


# =============================================================================
# 9. STRUCTURED MODEL ARTIFACT SEARCH
# =============================================================================

print("\n" + "=" * 100)
print("STRUCTURED + PRIOR MODEL ARTIFACT AUDIT")
print("=" * 100)

STRUCTURED_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "structured_prior"
)

structured_artifacts = []

if STRUCTURED_ROOT.exists():

    for path in STRUCTURED_ROOT.rglob("*"):

        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix in {
            ".pkl",
            ".pickle",
            ".joblib",
            ".json",
            ".bin",
            ".model",
            ".ubj",
        }:

            structured_artifacts.append(
                path
            )


for path in structured_artifacts:

    print(
        str(
            path.relative_to(
                PROJECT_ROOT
            )
        )
    )


actual_structured_model_candidates = [
    p
    for p in structured_artifacts
    if p.suffix.lower()
    in {
        ".pkl",
        ".pickle",
        ".joblib",
        ".model",
        ".ubj",
        ".bin",
    }
]


print(
    f"\nStructured binary/model "
    f"candidates : "
    f"{len(actual_structured_model_candidates)}"
)


# =============================================================================
# 10. TF-IDF ARTIFACT SEARCH
# =============================================================================

print("\n" + "=" * 100)
print("TF-IDF ARTIFACT AUDIT")
print("=" * 100)

TFIDF_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

tfidf_artifacts = []

if TFIDF_ROOT.exists():

    for path in TFIDF_ROOT.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() in {
            ".pkl",
            ".pickle",
            ".joblib",
            ".json",
            ".model",
            ".bin",
        }:

            tfidf_artifacts.append(
                path
            )


for path in tfidf_artifacts:

    print(
        str(
            path.relative_to(
                PROJECT_ROOT
            )
        )
    )


tfidf_binary_candidates = [
    p
    for p in tfidf_artifacts
    if p.suffix.lower()
    in {
        ".pkl",
        ".pickle",
        ".joblib",
        ".model",
        ".bin",
    }
]


print(
    f"\nTF-IDF binary/model "
    f"candidates : "
    f"{len(tfidf_binary_candidates)}"
)


# =============================================================================
# 11. SOURCE-TO-ARTIFACT DEPENDENCY SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("DEPENDENCY RESOLUTION")
print("=" * 100)

dependency_status = {

    "modernbert": {
        "checkpoint_verified": modernbert_complete,
        "production_folds": len(
            production_folds
        ),
    },

    "structured_prior": {
        "binary_candidates": len(
            actual_structured_model_candidates
        ),
        "resolved": (
            len(
                actual_structured_model_candidates
            ) > 0
        ),
    },

    "tfidf": {
        "binary_candidates": len(
            tfidf_binary_candidates
        ),
        "resolved": (
            len(
                tfidf_binary_candidates
            ) > 0
        ),
    },
}


print(
    json.dumps(
        dependency_status,
        indent=2,
    )
)


# =============================================================================
# 12. HARD SAFETY GATE
# =============================================================================

structured_resolved = (
    dependency_status[
        "structured_prior"
    ]["resolved"]
)

tfidf_resolved = (
    dependency_status[
        "tfidf"
    ]["resolved"]
)


print("\n" + "=" * 100)
print("INFERENCE DEPENDENCY SAFETY GATE")
print("=" * 100)

print(
    "ModernBERT resolved       : "
    f"{modernbert_complete}"
)

print(
    "Structured+Prior resolved : "
    f"{structured_resolved}"
)

print(
    "TF-IDF resolved           : "
    f"{tfidf_resolved}"
)


# IMPORTANT:
# Do NOT assert these two yet if the source implementation itself
# reveals that the model is reconstructed from multiple artifacts.
#
# We only lock ModernBERT here.
#
# Structured and TF-IDF remain unresolved until the exact
# training/inference dependency chain is established.

if not structured_resolved:

    print(
        "\nWARNING:"
    )

    print(
        "Structured+Prior actual "
        "model artifact was NOT resolved."
    )

if not tfidf_resolved:

    print(
        "\nWARNING:"
    )

    print(
        "TF-IDF actual prediction "
        "model artifact was NOT resolved."
    )


# =============================================================================
# 13. WRITE CELL 2 AUDIT
# =============================================================================

CELL2_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell2"
)

CELL2_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


AUDIT_JSON = (
    CELL2_ROOT
    / "cell2_dependency_resolution.json"
)


audit_payload = {
    "cell": 2,
    "status": "PASS",
    "production_method": "raw_blend",
    "blend_weights": BLEND_WEIGHTS,
    "modernbert": dependency_status[
        "modernbert"
    ],
    "structured_prior": dependency_status[
        "structured_prior"
    ],
    "tfidf": dependency_status[
        "tfidf"
    ],
    "asset_copy_started": False,
    "main_py_generated": False,
    "submission_generated": False,
}


with open(
    AUDIT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        audit_payload,
        f,
        indent=2,
    )


assert AUDIT_JSON.exists()


# =============================================================================
# 14. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print(
    "TRACE THE ACE — CELL 2 FINAL STATUS"
)
print("=" * 100)

print(
    "Production method lock       : PASS"
)

print(
    "ModernBERT checkpoint audit   : "
    f"{'PASS' if modernbert_complete else 'FAIL'}"
)

print(
    "Structured+Prior dependency   : "
    f"{'PASS' if structured_resolved else 'UNRESOLVED'}"
)

print(
    "TF-IDF dependency             : "
    f"{'PASS' if tfidf_resolved else 'UNRESOLVED'}"
)

print(
    "Asset copying                 : NOT STARTED"
)

print(
    "main.py generation            : NOT STARTED"
)

print(
    "Local runtime test            : NOT STARTED"
)

print(
    "Submission ZIP                : NOT GENERATED"
)

print("-" * 100)

print(
    f"Audit JSON : {AUDIT_JSON}"
)

print(
    "CELL 2 COMPLETE — PASS"
)

print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 2 — EXACT INFERENCE DEPENDENCY RESOLUTION

PATH CONTRACT
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SCRATCH_ROOT   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
Path contract : PASS

LOCKED PRODUCTION METHOD
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Weight sum          : 1.000000000000
Production method : raw_blend
Production method lock : PASS

SOURCE IMPLEMENTATION DISCOVERY
Source files discovered : 21
Readable source files : 21

IMPORTANT INFERENCE SOURCE FILES
Notebooks\00_environment_and_paths.ipynb
  groups: model_load
Notebooks\01_data_inventory.ipynb
  groups: inference, model_load, model_save, structured
Notebooks\02_turn_parser.ipynb
  groups: inference, model_load, structured
Notebooks\03_data_integrity.ipynb
 

In [5]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 3 — FINAL INFERENCE ARTIFACT BUILD
#
# PURPOSE
# -------
# Build the actual inference-time artifacts required by main.py.
#
# Components:
#
#   1. Structured + Prior
#      27 frozen structured features
#      + full-training objective prior
#      + StandardScaler
#      + LogisticRegression
#
#   2. TF-IDF
#      objective_text + evidence_text
#      + full-training TfidfVectorizer
#      + LogisticRegression
#
# IMPORTANT
# ---------
# - No OOF predictions are used as model inputs.
# - No target is used in structured feature construction.
# - Training labels are used ONLY to fit the final models / priors.
# - No test data is accessed.
# - No ModernBERT artifact is copied here.
# - No submission is generated here.
# =============================================================================

from pathlib import Path
import gc
import json
import hashlib
import shutil

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 3 — FINAL INFERENCE ARTIFACT BUILD")
print("=" * 100)


# =============================================================================
# 1. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
).resolve()

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

STRUCTURED_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "structured_prior"
)

STRUCTURED_FEATURE_PATH = (
    STRUCTURED_ROOT
    / "outputs"
    / "structured_response_features.parquet"
)

EVIDENCE_PATH = (
    SCRATCH_ROOT
    / "03_evidence_pack"
    / "frozen"
    / "evidence_packs.parquet"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

CELL3_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell3"
)

CELL3_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ASSETS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists()
assert STRUCTURED_FEATURE_PATH.exists(), (
    f"Structured feature artifact missing:\n"
    f"{STRUCTURED_FEATURE_PATH}"
)

assert EVIDENCE_PATH.exists(), (
    f"Frozen evidence artifact missing:\n"
    f"{EVIDENCE_PATH}"
)

print("\n" + "=" * 100)
print("INPUT ARTIFACT CONTRACT")
print("=" * 100)

print(
    f"Structured features : {STRUCTURED_FEATURE_PATH}"
)

print(
    f"Frozen evidence     : {EVIDENCE_PATH}"
)

print(
    "Input artifact contract : PASS"
)


# =============================================================================
# 2. TRAINING POPULATION
# =============================================================================

EXPECTED_ROWS = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_OBJECTIVES = 398

FOLD_VALUES = [
    0,
    1,
    2,
    3,
    4,
]

FOLD_COUNTS = {
    0: 6958,
    1: 7050,
    2: 7023,
    3: 7081,
    4: 6960,
}


# =============================================================================
# 3. LOAD STRUCTURED FEATURES
# =============================================================================

print("\n" + "=" * 100)
print("STRUCTURED FEATURE LOAD")
print("=" * 100)

structured = pd.read_parquet(
    STRUCTURED_FEATURE_PATH
)

assert len(structured) == EXPECTED_ROWS

required_structured_identity = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

missing = [
    c
    for c in required_structured_identity
    if c not in structured.columns
]

assert not missing, (
    f"Missing structured columns:\n{missing}"
)

assert structured[
    "response_id"
].is_unique

assert structured[
    "session_id"
].notna().all()

assert structured[
    "objective_uid"
].notna().all()

assert structured[
    "target"
].isin([0, 1]).all()

assert structured[
    "fold"
].isin(FOLD_VALUES).all()

assert (
    structured["session_id"].nunique()
    == EXPECTED_SESSIONS
)

assert (
    structured["objective_uid"].nunique()
    == EXPECTED_OBJECTIVES
)

for fold, expected_count in FOLD_COUNTS.items():

    actual = int(
        (
            structured["fold"]
            == fold
        ).sum()
    )

    assert actual == expected_count, (
        f"Fold {fold} count mismatch: "
        f"{actual} != {expected_count}"
    )


print(
    f"Rows      : {len(structured):,}"
)

print(
    f"Sessions  : "
    f"{structured['session_id'].nunique():,}"
)

print(
    f"Objectives: "
    f"{structured['objective_uid'].nunique():,}"
)

print(
    "Structured feature population : PASS"
)


# =============================================================================
# 4. DISCOVER EXACT 27 STRUCTURED FEATURES
# =============================================================================

EXCLUDED_STRUCTURED_COLUMNS = {
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
}

STRUCTURED_FEATURE_COLUMNS = [
    c
    for c in structured.columns
    if c not in EXCLUDED_STRUCTURED_COLUMNS
]

assert len(
    STRUCTURED_FEATURE_COLUMNS
) == 27, (
    "Expected exactly 27 structured features, "
    f"found {len(STRUCTURED_FEATURE_COLUMNS)}"
)

non_numeric = [
    c
    for c in STRUCTURED_FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(
        structured[c]
    )
]

assert not non_numeric, (
    f"Non-numeric structured features:\n"
    f"{non_numeric}"
)

structured_matrix = structured[
    STRUCTURED_FEATURE_COLUMNS
].to_numpy(
    dtype=np.float64
)

assert np.isfinite(
    structured_matrix
).all()

print("\n" + "=" * 100)
print("STRUCTURED FEATURE CONTRACT")
print("=" * 100)

print(
    f"Structured feature count : "
    f"{len(STRUCTURED_FEATURE_COLUMNS)}"
)

print(
    "Numerical contract : PASS"
)

print(
    "Target-derived feature guard : PASS"
)


# =============================================================================
# 5. BUILD FULL-TRAINING OBJECTIVE PRIOR
#
# Exact Cell-8 formula:
#
#   (positive + alpha * global_prior)
#   /
#   (count + alpha)
#
# alpha = 20
#
# This is the prior used for inference on unseen/test responses.
# =============================================================================

PRIOR_ALPHA = 20.0

global_train_prior = float(
    structured["target"].mean()
)

assert (
    0.0
    <
    global_train_prior
    <
    1.0
)

objective_stats = (
    structured
    .groupby(
        "objective_uid",
        sort=False,
    )["target"]
    .agg(
        objective_train_count="size",
        objective_train_positive="sum",
    )
)

objective_stats[
    "objective_prior"
] = (
    objective_stats[
        "objective_train_positive"
    ]
    +
    PRIOR_ALPHA
    *
    global_train_prior
) / (
    objective_stats[
        "objective_train_count"
    ]
    +
    PRIOR_ALPHA
)

objective_stats[
    "objective_prior"
] = (
    objective_stats[
        "objective_prior"
    ]
    .astype(float)
    .clip(
        1e-6,
        1.0 - 1e-6,
    )
)

objective_stats[
    "objective_prior_logit"
] = np.log(
    objective_stats[
        "objective_prior"
    ]
    /
    (
        1.0
        -
        objective_stats[
            "objective_prior"
        ]
    )
)

assert len(
    objective_stats
) == EXPECTED_OBJECTIVES

assert np.isfinite(
    objective_stats[
        "objective_prior"
    ]
).all()

assert np.isfinite(
    objective_stats[
        "objective_prior_logit"
    ]
).all()

print("\n" + "=" * 100)
print("FULL-TRAINING OBJECTIVE PRIOR")
print("=" * 100)

print(
    f"Global training prior : "
    f"{global_train_prior:.12f}"
)

print(
    f"Objective count       : "
    f"{len(objective_stats):,}"
)

print(
    f"Prior alpha           : "
    f"{PRIOR_ALPHA:.1f}"
)

print(
    "Objective prior contract : PASS"
)


# =============================================================================
# 6. BUILD INNER-CROSS-FITTED TRAINING PRIOR
#
# Exact principle used by Cell 9:
#
# Every training row receives a prior calculated without using
# its own target.
#
# Deterministic assignment:
#   response_id sorted stable
#   inner_fold = position % 4
#
# This produces training features consistent with the OOF
# Structured + Prior implementation.
# =============================================================================

INNER_FOLDS = 4
RANDOM_STATE = 42

prior_work = structured[
    [
        "response_id",
        "objective_uid",
        "target",
    ]
].copy()

ordered_ids = (
    prior_work[
        "response_id"
    ]
    .astype(str)
    .sort_values(
        kind="stable"
    )
    .to_numpy()
)

inner_assignment = {
    response_id: (
        i % INNER_FOLDS
    )
    for i, response_id
    in enumerate(ordered_ids)
}

prior_work[
    "_inner_fold"
] = (
    prior_work[
        "response_id"
    ]
    .astype(str)
    .map(
        inner_assignment
    )
)

prior_work[
    "inner_objective_prior"
] = np.nan


for inner_valid_fold in range(
    INNER_FOLDS
):

    inner_train = prior_work.loc[
        prior_work[
            "_inner_fold"
        ]
        != inner_valid_fold
    ]

    inner_valid_mask = (
        prior_work[
            "_inner_fold"
        ]
        == inner_valid_fold
    )

    inner_global_prior = float(
        inner_train[
            "target"
        ].mean()
    )

    inner_stats = (
        inner_train
        .groupby(
            "objective_uid",
            sort=False,
        )["target"]
        .agg(
            count="size",
            positive="sum",
        )
    )

    inner_stats[
        "prior"
    ] = (
        inner_stats[
            "positive"
        ]
        +
        PRIOR_ALPHA
        *
        inner_global_prior
    ) / (
        inner_stats[
            "count"
        ]
        +
        PRIOR_ALPHA
    )

    valid_priors = (
        prior_work.loc[
            inner_valid_mask,
            "objective_uid",
        ]
        .map(
            inner_stats[
                "prior"
            ]
        )
        .fillna(
            inner_global_prior
        )
        .clip(
            1e-6,
            1.0 - 1e-6,
        )
    )

    prior_work.loc[
        inner_valid_mask,
        "inner_objective_prior",
    ] = valid_priors.to_numpy(
        dtype=np.float64
    )

    del inner_train
    del inner_stats
    del valid_priors


assert prior_work[
    "inner_objective_prior"
].notna().all()

assert np.isfinite(
    prior_work[
        "inner_objective_prior"
    ]
).all()

print("\n" + "=" * 100)
print("INNER-CROSS-FITTED TRAINING PRIOR")
print("=" * 100)

print(
    f"Inner folds : {INNER_FOLDS}"
)

print(
    "Own-target exclusion : PASS"
)

print(
    "Inner prior numerical contract : PASS"
)


# =============================================================================
# 7. FINAL STRUCTURED + PRIOR TRAINING MATRIX
# =============================================================================

structured_train = structured[
    [
        "response_id",
        "target",
    ]
    +
    STRUCTURED_FEATURE_COLUMNS
].copy()

structured_train = (
    structured_train
    .merge(
        prior_work[
            [
                "response_id",
                "inner_objective_prior",
            ]
        ],
        on="response_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(
    structured_train
) == EXPECTED_ROWS

assert structured_train[
    "inner_objective_prior"
].notna().all()

FINAL_STRUCTURED_COLUMNS = (
    STRUCTURED_FEATURE_COLUMNS
    +
    [
        "inner_objective_prior",
    ]
)

X_structured = (
    structured_train[
        FINAL_STRUCTURED_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)

y_structured = (
    structured_train[
        "target"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

assert X_structured.shape == (
    EXPECTED_ROWS,
    28,
)

assert np.isfinite(
    X_structured
).all()

assert np.isfinite(
    y_structured
).all()


# =============================================================================
# 8. FIT FINAL STRUCTURED + PRIOR MODEL
# =============================================================================

STRUCTURED_LOGISTIC_C = 1.0
STRUCTURED_MAX_ITER = 2000

structured_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(
                with_mean=True,
                with_std=True,
            ),
        ),
        (
            "logistic",
            LogisticRegression(
                C=STRUCTURED_LOGISTIC_C,
                max_iter=STRUCTURED_MAX_ITER,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

structured_model.fit(
    X_structured,
    y_structured,
)

structured_train_prediction = (
    structured_model
    .predict_proba(
        X_structured
    )[:, 1]
)

assert np.isfinite(
    structured_train_prediction
).all()

assert (
    (
        structured_train_prediction
        >= 0.0
    )
    &
    (
        structured_train_prediction
        <= 1.0
    )
).all()

print("\n" + "=" * 100)
print("STRUCTURED + PRIOR FINAL MODEL")
print("=" * 100)

print(
    "Model : StandardScaler + LogisticRegression"
)

print(
    f"C : {STRUCTURED_LOGISTIC_C}"
)

print(
    f"max_iter : {STRUCTURED_MAX_ITER}"
)

print(
    "Final fit population : "
    f"{len(structured_train):,}"
)

print(
    "Prediction contract : PASS"
)


# =============================================================================
# 9. LOAD FROZEN EVIDENCE
# =============================================================================

print("\n" + "=" * 100)
print("TF-IDF EVIDENCE LOAD")
print("=" * 100)

EVIDENCE_REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "objective_text",
    "evidence_text",
    "target",
]

evidence = pd.read_parquet(
    EVIDENCE_PATH,
    columns=EVIDENCE_REQUIRED_COLUMNS,
)

assert len(evidence) == EXPECTED_ROWS

assert evidence[
    "response_id"
].is_unique

assert evidence[
    "target"
].isin([0, 1]).all()

assert evidence[
    "fold"
].isin(FOLD_VALUES).all()

assert (
    set(
        evidence[
            "response_id"
        ].astype(str)
    )
    ==
    set(
        structured[
            "response_id"
        ].astype(str)
    )
)

assert evidence[
    "objective_text"
].notna().all()

assert evidence[
    "evidence_text"
].notna().all()

print(
    f"Evidence rows : {len(evidence):,}"
)

print(
    "Evidence population : PASS"
)


# =============================================================================
# 10. EXACT TF-IDF TEXT CONSTRUCTION
# =============================================================================

evidence[
    "objective_text"
] = (
    evidence[
        "objective_text"
    ]
    .astype(str)
    .str.strip()
)

evidence[
    "evidence_text"
] = (
    evidence[
        "evidence_text"
    ]
    .astype(str)
    .str.strip()
)

assert (
    evidence[
        "objective_text"
    ].str.len()
    > 0
).all()

assert (
    evidence[
        "evidence_text"
    ].str.len()
    > 0
).all()

evidence[
    "_tfidf_text"
] = (
    evidence[
        "objective_text"
    ]
    +
    "\nOBJECTIVE_EVIDENCE\n"
    +
    evidence[
        "evidence_text"
    ]
)

assert evidence[
    "_tfidf_text"
].notna().all()

assert (
    evidence[
        "_tfidf_text"
    ].str.len()
    > 0
).all()

print(
    "TF-IDF text construction : PASS"
)


# =============================================================================
# 11. FIT FINAL TF-IDF VECTORIZER
# =============================================================================

TFIDF_MAX_FEATURES = 150_000
TFIDF_NGRAM_RANGE = (
    1,
    2,
)
TFIDF_MIN_DF = 2
TFIDF_MAX_DF = 0.995
TFIDF_SUBLINEAR_TF = True
TFIDF_NORM = "l2"

tfidf_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    sublinear_tf=TFIDF_SUBLINEAR_TF,
    norm=TFIDF_NORM,
    dtype=np.float32,
)

X_tfidf = (
    tfidf_vectorizer.fit_transform(
        evidence[
            "_tfidf_text"
        ]
    )
)

assert X_tfidf.shape[0] == EXPECTED_ROWS
assert X_tfidf.shape[1] > 0

vocabulary_size = int(
    len(
        tfidf_vectorizer.vocabulary_
    )
)

assert vocabulary_size > 0

print("\n" + "=" * 100)
print("FINAL TF-IDF VECTORIZER")
print("=" * 100)

print(
    f"Rows       : {X_tfidf.shape[0]:,}"
)

print(
    f"Features   : {X_tfidf.shape[1]:,}"
)

print(
    f"Vocabulary : {vocabulary_size:,}"
)

print(
    "Vectorizer fit population : "
    f"{EXPECTED_ROWS:,}"
)

print(
    "TF-IDF vectorizer contract : PASS"
)


# =============================================================================
# 12. FIT FINAL TF-IDF LOGISTIC MODEL
# =============================================================================

TFIDF_LOGISTIC_C = 2.0
TFIDF_MAX_ITER = 1000

tfidf_classifier = LogisticRegression(
    C=TFIDF_LOGISTIC_C,
    max_iter=TFIDF_MAX_ITER,
    solver="lbfgs",
    random_state=RANDOM_STATE,
)

tfidf_classifier.fit(
    X_tfidf,
    evidence[
        "target"
    ].to_numpy(
        dtype=np.int64
    ),
)

tfidf_train_prediction = (
    tfidf_classifier
    .predict_proba(
        X_tfidf
    )[:, 1]
)

assert np.isfinite(
    tfidf_train_prediction
).all()

assert (
    (
        tfidf_train_prediction
        >= 0.0
    )
    &
    (
        tfidf_train_prediction
        <= 1.0
    )
).all()

print("\n" + "=" * 100)
print("TF-IDF FINAL MODEL")
print("=" * 100)

print(
    "Model : LogisticRegression"
)

print(
    f"C : {TFIDF_LOGISTIC_C}"
)

print(
    f"max_iter : {TFIDF_MAX_ITER}"
)

print(
    "Prediction contract : PASS"
)


# =============================================================================
# 13. SERIALIZATION PATHS
# =============================================================================

STRUCTURED_ASSET_ROOT = (
    ASSETS_ROOT
    / "structured_prior"
)

TFIDF_ASSET_ROOT = (
    ASSETS_ROOT
    / "tfidf"
)

STRUCTURED_ASSET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TFIDF_ASSET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STRUCTURED_MODEL_PATH = (
    STRUCTURED_ASSET_ROOT
    / "structured_prior_model.joblib"
)

STRUCTURED_PRIOR_PATH = (
    STRUCTURED_ASSET_ROOT
    / "objective_prior_stats.parquet"
)

STRUCTURED_FEATURE_SCHEMA_PATH = (
    STRUCTURED_ASSET_ROOT
    / "structured_feature_schema.json"
)

TFIDF_VECTORIZER_PATH = (
    TFIDF_ASSET_ROOT
    / "tfidf_vectorizer.joblib"
)

TFIDF_MODEL_PATH = (
    TFIDF_ASSET_ROOT
    / "tfidf_model.joblib"
)

TFIDF_CONFIG_PATH = (
    TFIDF_ASSET_ROOT
    / "tfidf_config.json"
)


# =============================================================================
# 14. SERIALIZE JOBLIB ARTIFACTS
# =============================================================================

import joblib


joblib.dump(
    structured_model,
    STRUCTURED_MODEL_PATH,
)

joblib.dump(
    tfidf_vectorizer,
    TFIDF_VECTORIZER_PATH,
)

joblib.dump(
    tfidf_classifier,
    TFIDF_MODEL_PATH,
)

assert STRUCTURED_MODEL_PATH.exists()
assert TFIDF_VECTORIZER_PATH.exists()
assert TFIDF_MODEL_PATH.exists()

print("\n" + "=" * 100)
print("MODEL SERIALIZATION")
print("=" * 100)

print(
    f"Structured model : "
    f"{STRUCTURED_MODEL_PATH}"
)

print(
    f"TF-IDF vectorizer: "
    f"{TFIDF_VECTORIZER_PATH}"
)

print(
    f"TF-IDF model     : "
    f"{TFIDF_MODEL_PATH}"
)

print(
    "Model serialization : PASS"
)


# =============================================================================
# 15. SERIALIZE OBJECTIVE PRIOR TABLE
# =============================================================================

prior_export = (
    objective_stats
    .reset_index()
    [
        [
            "objective_uid",
            "objective_train_count",
            "objective_train_positive",
            "objective_prior",
            "objective_prior_logit",
        ]
    ]
    .copy()
)

assert len(
    prior_export
) == EXPECTED_OBJECTIVES

prior_export.to_parquet(
    STRUCTURED_PRIOR_PATH,
    index=False,
)

assert STRUCTURED_PRIOR_PATH.exists()

print(
    "Objective prior table serialization : PASS"
)


# =============================================================================
# 16. SERIALIZE STRUCTURED FEATURE SCHEMA
# =============================================================================

structured_schema = {
    "model_type": (
        "StandardScaler+"
        "LogisticRegression"
    ),
    "structured_feature_columns": (
        STRUCTURED_FEATURE_COLUMNS
    ),
    "structured_feature_count": (
        len(
            STRUCTURED_FEATURE_COLUMNS
        )
    ),
    "prior_feature_columns": [
        "objective_prior",
        "objective_prior_logit",
    ],
    "training_prior_feature": (
        "inner_objective_prior"
    ),
    "prior_alpha": PRIOR_ALPHA,
    "logistic_C": STRUCTURED_LOGISTIC_C,
    "max_iter": STRUCTURED_MAX_ITER,
    "solver": "lbfgs",
    "random_state": RANDOM_STATE,
}

with open(
    STRUCTURED_FEATURE_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        structured_schema,
        f,
        indent=2,
    )

assert STRUCTURED_FEATURE_SCHEMA_PATH.exists()


# =============================================================================
# 17. SERIALIZE TF-IDF CONFIG
# =============================================================================

tfidf_config = {
    "text_source": (
        "objective_text + "
        "\\nOBJECTIVE_EVIDENCE\\n + "
        "evidence_text"
    ),
    "max_features": TFIDF_MAX_FEATURES,
    "ngram_range": list(
        TFIDF_NGRAM_RANGE
    ),
    "min_df": TFIDF_MIN_DF,
    "max_df": TFIDF_MAX_DF,
    "sublinear_tf": TFIDF_SUBLINEAR_TF,
    "norm": TFIDF_NORM,
    "dtype": "float32",
    "classifier": (
        "LogisticRegression"
    ),
    "classifier_C": TFIDF_LOGISTIC_C,
    "classifier_max_iter": TFIDF_MAX_ITER,
    "solver": "lbfgs",
    "random_state": RANDOM_STATE,
}

with open(
    TFIDF_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        tfidf_config,
        f,
        indent=2,
    )

assert TFIDF_CONFIG_PATH.exists()


# =============================================================================
# 18. RELOAD ARTIFACTS
# =============================================================================

print("\n" + "=" * 100)
print("SERIALIZATION RELOAD AUDIT")
print("=" * 100)

reloaded_structured_model = joblib.load(
    STRUCTURED_MODEL_PATH
)

reloaded_tfidf_vectorizer = joblib.load(
    TFIDF_VECTORIZER_PATH
)

reloaded_tfidf_model = joblib.load(
    TFIDF_MODEL_PATH
)

reloaded_prior = pd.read_parquet(
    STRUCTURED_PRIOR_PATH
)

with open(
    STRUCTURED_FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8",
) as f:

    reloaded_structured_schema = json.load(
        f
    )

with open(
    TFIDF_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    reloaded_tfidf_config = json.load(
        f
    )


# =============================================================================
# 19. RELOAD PREDICTION PARITY
# =============================================================================

reloaded_structured_prediction = (
    reloaded_structured_model
    .predict_proba(
        X_structured
    )[:, 1]
)

reloaded_tfidf_prediction = (
    reloaded_tfidf_model
    .predict_proba(
        X_tfidf
    )[:, 1]
)

assert np.allclose(
    structured_train_prediction,
    reloaded_structured_prediction,
    rtol=0.0,
    atol=1e-12,
)

assert np.allclose(
    tfidf_train_prediction,
    reloaded_tfidf_prediction,
    rtol=0.0,
    atol=1e-12,
)

assert len(
    reloaded_prior
) == EXPECTED_OBJECTIVES

assert (
    reloaded_structured_schema[
        "structured_feature_count"
    ]
    == 27
)

assert (
    reloaded_tfidf_config[
        "max_features"
    ]
    == TFIDF_MAX_FEATURES
)

print(
    "Structured model reload parity : PASS"
)

print(
    "TF-IDF model reload parity      : PASS"
)

print(
    "Prior table reload              : PASS"
)

print(
    "Configuration reload            : PASS"
)


# =============================================================================
# 20. ARTIFACT MANIFEST
# =============================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


runtime_artifacts = [
    STRUCTURED_MODEL_PATH,
    STRUCTURED_PRIOR_PATH,
    STRUCTURED_FEATURE_SCHEMA_PATH,
    TFIDF_VECTORIZER_PATH,
    TFIDF_MODEL_PATH,
    TFIDF_CONFIG_PATH,
]

artifact_manifest = []

for path in runtime_artifacts:

    assert path.exists()

    artifact_manifest.append(
        {
            "relative_path": str(
                path.relative_to(
                    SUBMISSION_ROOT
                )
            ),
            "size_bytes": int(
                path.stat().st_size
            ),
            "sha256": sha256_file(
                path
            ),
        }
    )


MANIFEST_PATH = (
    CELL3_ROOT
    / "cell3_runtime_artifact_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "cell": 3,
            "status": "PASS",
            "training_rows": EXPECTED_ROWS,
            "training_sessions": EXPECTED_SESSIONS,
            "training_objectives": EXPECTED_OBJECTIVES,
            "global_train_prior": (
                global_train_prior
            ),
            "structured_feature_count": 27,
            "structured_model": (
                "StandardScaler+"
                "LogisticRegression"
            ),
            "structured_logistic_C": (
                STRUCTURED_LOGISTIC_C
            ),
            "tfidf_model": (
                "TfidfVectorizer+"
                "LogisticRegression"
            ),
            "tfidf_max_features": (
                TFIDF_MAX_FEATURES
            ),
            "tfidf_logistic_C": (
                TFIDF_LOGISTIC_C
            ),
            "assets": artifact_manifest,
            "test_data_accessed": False,
            "prediction_submission_generated": False,
        },
        f,
        indent=2,
    )


# =============================================================================
# 21. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print("TRACE THE ACE — CELL 3 FINAL STATUS")
print("=" * 100)

print(
    "Structured feature population       : PASS"
)

print(
    "Full-training objective prior       : PASS"
)

print(
    "Inner-cross-fitted training prior  : PASS"
)

print(
    "Structured + Prior final model     : PASS"
)

print(
    "TF-IDF text construction            : PASS"
)

print(
    "TF-IDF final vectorizer             : PASS"
)

print(
    "TF-IDF final classifier              : PASS"
)

print(
    "Artifact serialization               : PASS"
)

print(
    "Artifact reload parity               : PASS"
)

print(
    "Runtime manifest                     : PASS"
)

print(
    "Test data accessed                   : NO"
)

print(
    "Submission generated                 : NO"
)

print(
    "main.py generated                    : NO"
)

print(
    "Submission ZIP generated             : NO"
)

print("-" * 100)

print(
    f"Manifest : {MANIFEST_PATH}"
)

print(
    "CELL 3 COMPLETE — PASS"
)

print("=" * 100)


# =============================================================================
# 22. MEMORY CLEANUP
# =============================================================================

del X_structured
del X_tfidf
del structured_train
del structured_matrix
del prior_work
del evidence
del structured
del tfidf_vectorizer
del tfidf_classifier
del structured_model
del reloaded_structured_model
del reloaded_tfidf_vectorizer
del reloaded_tfidf_model
del reloaded_prior

gc.collect()

print(
    "\nCell 3 memory cleanup : PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 3 — FINAL INFERENCE ARTIFACT BUILD

INPUT ARTIFACT CONTRACT
Structured features : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\structured_prior\outputs\structured_response_features.parquet
Frozen evidence     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet
Input artifact contract : PASS

STRUCTURED FEATURE LOAD
Rows      : 35,072
Sessions  : 22,821
Objectives: 398
Structured feature population : PASS

STRUCTURED FEATURE CONTRACT
Structured feature count : 27
Numerical contract : PASS
Target-derived feature guard : PASS

FULL-TRAINING OBJECTIVE PRIOR
Global training prior : 0.702469206204
Objective count       : 398
Prior alpha           : 20.0
Objective prior contract : PASS

INNER-CROSS-FITTED TRAINING PRIOR
Inner folds : 4
Own-target exclusion : PASS
Inner prior numerical contract : PASS

STRUCTURED + PRIOR FINAL MODEL
Model : StandardScaler + LogisticRegression
C :

In [6]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 4 — MODERNBERT PRODUCTION ARTIFACT PACKAGING + RELOAD LOCK
#
# PURPOSE
# -------
# Package ONLY the verified ModernBERT production checkpoints required
# for runtime inference.
#
# Verified production state:
#
#   Fold 0 -> reused / engineering fold
#   Fold 1 -> production checkpoint
#   Fold 2 -> production checkpoint
#   Fold 3 -> production checkpoint
#   Fold 4 -> production checkpoint
#
# Therefore runtime package contains folds 1–4 only.
#
# NO TRAINING
# NO OOF GENERATION
# NO TEST DATA
# NO SUBMISSION GENERATION
#
# This cell:
#   1. discovers the exact production checkpoint roots
#   2. verifies required checkpoint files
#   3. copies them into submission_runtime/assets/modernbert/
#   4. verifies copied files
#   5. reloads every checkpoint locally
#   6. performs a deterministic forward smoke test
#   7. records SHA-256 hashes + manifest
# =============================================================================

from pathlib import Path
import gc
import hashlib
import json
import shutil
import sys
import platform

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 4 — MODERNBERT PRODUCTION ARTIFACT "
    "PACKAGING + RELOAD LOCK"
)
print("=" * 100)


# =============================================================================
# 1. ENVIRONMENT
# =============================================================================

print("\n" + "=" * 100)
print("ENVIRONMENT")
print("=" * 100)

print(
    "Python :",
    sys.version,
)

print(
    "Platform :",
    platform.platform(),
)

print(
    "Torch :",
    torch.__version__,
)

print(
    "Transformers :",
    __import__("transformers").__version__,
)


# =============================================================================
# 2. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
).resolve()

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

MODERNBERT_ASSET_ROOT = (
    ASSETS_ROOT
    / "modernbert"
)

CELL4_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell4"
)

CELL4_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODERNBERT_ASSET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists(), (
    f"Project root missing:\n{PROJECT_ROOT}"
)

assert SUBMISSION_ROOT.exists(), (
    f"Submission root missing:\n{SUBMISSION_ROOT}"
)

assert ASSETS_ROOT.exists(), (
    f"Assets root missing:\n{ASSETS_ROOT}"
)

print(
    "PROJECT_ROOT   :",
    PROJECT_ROOT,
)

print(
    "SUBMISSION_ROOT:",
    SUBMISSION_ROOT,
)

print(
    "ASSETS_ROOT    :",
    ASSETS_ROOT,
)

print(
    "MODERNBERT_ASSET_ROOT:",
    MODERNBERT_ASSET_ROOT,
)

print(
    "Path contract : PASS"
)


# =============================================================================
# 3. LOCKED PRODUCTION METHOD
# =============================================================================

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

assert abs(
    sum(
        BLEND_WEIGHTS.values()
    )
    - 1.0
) < 1e-12

print("\n" + "=" * 100)
print("LOCKED PRODUCTION METHOD")
print("=" * 100)

for name, weight in BLEND_WEIGHTS.items():

    print(
        f"{name:20s}: {weight:.6f}"
    )

print(
    "Weight sum :",
    f"{sum(BLEND_WEIGHTS.values()):.12f}",
)

print(
    "Production method : raw_blend"
)

print(
    "Production method lock : PASS"
)


# =============================================================================
# 4. MODERNBERT SOURCE ROOT DISCOVERY
#
# The audited artifact inventory established:
#
# modernbert_outputs/
#   modernbert_outputs_full/
#       modernbert_outputs/
#           production_fold_1/
#           production_fold_2/
#           production_fold_3/
#           production_fold_4/
#
# Do NOT silently substitute another checkpoint.
# =============================================================================

MODERNBERT_ROOT_CANDIDATES = [

    (
        PROJECT_ROOT
        / "modernbert_outputs"
        / "modernbert_outputs_full"
        / "modernbert_outputs"
    ),

    (
        PROJECT_ROOT
        / "modernbert"
        / "modernbert_outputs"
        / "modernbert_outputs_full"
        / "modernbert_outputs"
    ),

    (
        PROJECT_ROOT
        / "modernbert_outputs_full"
        / "modernbert_outputs"
    ),

]


MODERNBERT_SOURCE_ROOT = next(
    (
        p
        for p in MODERNBERT_ROOT_CANDIDATES
        if p.exists()
        and p.is_dir()
    ),
    None,
)


assert MODERNBERT_SOURCE_ROOT is not None, (
    "Verified ModernBERT production root "
    "could not be discovered.\n\n"
    "Checked:\n"
    +
    "\n".join(
        str(p)
        for p in MODERNBERT_ROOT_CANDIDATES
    )
)

print("\n" + "=" * 100)
print("MODERNBERT SOURCE ROOT")
print("=" * 100)

print(
    "Source root :",
    MODERNBERT_SOURCE_ROOT,
)

print(
    "Source root discovery : PASS"
)


# =============================================================================
# 5. PRODUCTION FOLD CONTRACT
# =============================================================================

PRODUCTION_FOLDS = [
    1,
    2,
    3,
    4,
]

EXPECTED_MODEL_NAME = (
    "answerdotai/ModernBERT-base"
)

EXPECTED_MAX_LENGTH = 2048
EXPECTED_NUM_LABELS = 2

REQUIRED_CHECKPOINT_FILES = [
    "config.json",
    "model.safetensors",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
]


# =============================================================================
# 6. DISCOVER EXACT PRODUCTION CHECKPOINTS
# =============================================================================

print("\n" + "=" * 100)
print("PRODUCTION CHECKPOINT DISCOVERY")
print("=" * 100)

production_checkpoint_paths = {}

for fold in PRODUCTION_FOLDS:

    checkpoint = (
        MODERNBERT_SOURCE_ROOT
        / f"production_fold_{fold}"
        / "checkpoints"
        / "best"
    )

    assert checkpoint.exists(), (
        f"Production fold {fold} checkpoint missing:\n"
        f"{checkpoint}"
    )

    assert checkpoint.is_dir(), (
        f"Production fold {fold} checkpoint is not a directory:\n"
        f"{checkpoint}"
    )

    production_checkpoint_paths[
        fold
    ] = checkpoint

    print(
        f"Fold {fold}:",
        checkpoint,
    )


print(
    "Production fold discovery : PASS"
)


# =============================================================================
# 7. CHECKPOINT FILE CONTRACT
# =============================================================================

print("\n" + "=" * 100)
print("CHECKPOINT FILE CONTRACT")
print("=" * 100)

source_inventory = []

for fold, checkpoint in (
    production_checkpoint_paths.items()
):

    print(
        f"\nFold {fold}"
    )

    for filename in REQUIRED_CHECKPOINT_FILES:

        path = (
            checkpoint
            / filename
        )

        assert path.exists(), (
            f"Fold {fold} missing required "
            f"checkpoint file:\n{path}"
        )

        assert path.is_file(), (
            f"Fold {fold} required artifact "
            f"is not a file:\n{path}"
        )

        size_bytes = int(
            path.stat().st_size
        )

        assert size_bytes > 0, (
            f"Fold {fold} artifact is empty:\n"
            f"{path}"
        )

        source_inventory.append(
            {
                "fold": fold,
                "filename": filename,
                "source_path": str(path),
                "size_bytes": size_bytes,
            }
        )

        print(
            f"  {filename:28s}"
            f": PASS "
            f"({size_bytes / (1024**2):.3f} MB)"
        )


print(
    "\nRequired checkpoint files : PASS"
)


# =============================================================================
# 8. SOURCE CONFIG AUDIT
# =============================================================================

print("\n" + "=" * 100)
print("SOURCE CONFIG AUDIT")
print("=" * 100)

source_configs = {}

for fold, checkpoint in (
    production_checkpoint_paths.items()
):

    config = AutoConfig.from_pretrained(
        checkpoint,
        local_files_only=True,
    )

    config_model_type = str(
        config.model_type
    )

    num_labels = int(
        getattr(
            config,
            "num_labels",
            -1,
        )
    )

    source_configs[
        fold
    ] = {
        "model_type": config_model_type,
        "num_labels": num_labels,
        "hidden_size": int(
            getattr(
                config,
                "hidden_size",
                -1,
            )
        ),
        "num_hidden_layers": int(
            getattr(
                config,
                "num_hidden_layers",
                -1,
            )
        ),
        "num_attention_heads": int(
            getattr(
                config,
                "num_attention_heads",
                -1,
            )
        ),
    }

    print(
        f"Fold {fold}:"
    )

    print(
        "  model_type       :",
        config_model_type,
    )

    print(
        "  num_labels       :",
        num_labels,
    )

    print(
        "  hidden_size      :",
        source_configs[
            fold
        ]["hidden_size"],
    )

    print(
        "  hidden_layers    :",
        source_configs[
            fold
        ]["num_hidden_layers"],
    )

    print(
        "  attention_heads  :",
        source_configs[
            fold
        ]["num_attention_heads"],
    )

    assert (
        config_model_type
        == "modernbert"
    ), (
        f"Fold {fold} is not ModernBERT."
    )

    assert (
        num_labels
        == EXPECTED_NUM_LABELS
    ), (
        f"Fold {fold} num_labels mismatch."
    )


print(
    "\nModernBERT config contract : PASS"
)


# =============================================================================
# 9. TOKENIZER CONTRACT
#
# Each production checkpoint contains its own tokenizer artifacts.
# We verify all four, then use fold 1 as the canonical runtime tokenizer
# because the production tokenizer artifacts should be identical.
# =============================================================================

print("\n" + "=" * 100)
print("TOKENIZER CONTRACT")
print("=" * 100)

tokenizer_signatures = {}

for fold, checkpoint in (
    production_checkpoint_paths.items()
):

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True,
        local_files_only=True,
    )

    assert tokenizer.is_fast, (
        f"Fold {fold} tokenizer is not fast."
    )

    native_max_length = int(
        tokenizer.model_max_length
    )

    assert (
        tokenizer.pad_token_id
        is not None
    ), (
        f"Fold {fold} tokenizer has no pad token."
    )

    tokenizer_length = int(
        len(tokenizer)
    )

    tokenizer_vocab_size = int(
        tokenizer.vocab_size
    )

    assert (
        tokenizer_length
        >= tokenizer_vocab_size
    )

    tokenizer_signatures[
        fold
    ] = {
        "native_max_length": (
            native_max_length
        ),
        "tokenizer_length": (
            tokenizer_length
        ),
        "vocab_size": (
            tokenizer_vocab_size
        ),
        "pad_token_id": (
            tokenizer.pad_token_id
        ),
        "cls_token_id": (
            tokenizer.cls_token_id
        ),
        "sep_token_id": (
            tokenizer.sep_token_id
        ),
    }

    print(
        f"Fold {fold}: "
        f"native_max_length="
        f"{native_max_length}, "
        f"tokenizer_length="
        f"{tokenizer_length}, "
        f"vocab_size="
        f"{tokenizer_vocab_size}"
    )

    assert (
        native_max_length
        >= EXPECTED_MAX_LENGTH
    ), (
        f"Fold {fold} tokenizer native "
        f"max length is below {EXPECTED_MAX_LENGTH}."
    )


# Exact tokenizer signature consistency
reference_tokenizer_signature = (
    tokenizer_signatures[1]
)

for fold in PRODUCTION_FOLDS[1:]:

    assert (
        tokenizer_signatures[fold]
        ==
        reference_tokenizer_signature
    ), (
        f"Tokenizer signature mismatch "
        f"between fold 1 and fold {fold}."
    )


print(
    "Tokenizer consistency : PASS"
)

print(
    "Project max length :",
    EXPECTED_MAX_LENGTH,
)

print(
    "Tokenizer contract : PASS"
)


# =============================================================================
# 10. COPY PRODUCTION CHECKPOINTS
#
# We intentionally copy the complete `best` directory for each production
# fold. This avoids reconstructing a HuggingFace checkpoint from individual
# files and preserves the exact saved artifact.
# =============================================================================

print("\n" + "=" * 100)
print("MODERNBERT ASSET COPYING")
print("=" * 100)

copied_fold_paths = {}

for fold in PRODUCTION_FOLDS:

    source = (
        production_checkpoint_paths[
            fold
        ]
    )

    destination = (
        MODERNBERT_ASSET_ROOT
        / f"production_fold_{fold}"
    )

    if destination.exists():

        shutil.rmtree(
            destination
        )

    shutil.copytree(
        source,
        destination,
    )

    copied_fold_paths[
        fold
    ] = destination

    print(
        f"Fold {fold}: copied"
    )

print(
    "ModernBERT asset copying : PASS"
)


# =============================================================================
# 11. COPIED ARTIFACT EXISTENCE + SIZE AUDIT
# =============================================================================

print("\n" + "=" * 100)
print("COPIED ARTIFACT AUDIT")
print("=" * 100)

copied_inventory = []

for fold in PRODUCTION_FOLDS:

    checkpoint = (
        copied_fold_paths[
            fold
        ]
    )

    for filename in REQUIRED_CHECKPOINT_FILES:

        path = (
            checkpoint
            / filename
        )

        assert path.exists(), (
            f"Copied fold {fold} missing:\n"
            f"{path}"
        )

        assert path.is_file()

        size_bytes = int(
            path.stat().st_size
        )

        assert size_bytes > 0

        copied_inventory.append(
            {
                "fold": fold,
                "filename": filename,
                "path": str(path),
                "size_bytes": size_bytes,
            }
        )

print(
    "Copied artifact existence : PASS"
)


# =============================================================================
# 12. BYTE-LEVEL SOURCE → COPY PARITY
# =============================================================================

print("\n" + "=" * 100)
print("SOURCE → COPY PARITY")
print("=" * 100)


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


parity_rows = []

for fold in PRODUCTION_FOLDS:

    source_checkpoint = (
        production_checkpoint_paths[
            fold
        ]
    )

    copied_checkpoint = (
        copied_fold_paths[
            fold
        ]
    )

    for filename in REQUIRED_CHECKPOINT_FILES:

        source_file = (
            source_checkpoint
            / filename
        )

        copied_file = (
            copied_checkpoint
            / filename
        )

        source_hash = sha256_file(
            source_file
        )

        copied_hash = sha256_file(
            copied_file
        )

        assert (
            source_hash
            ==
            copied_hash
        ), (
            f"SHA-256 mismatch "
            f"fold={fold}, "
            f"file={filename}"
        )

        parity_rows.append(
            {
                "fold": fold,
                "filename": filename,
                "source_sha256": source_hash,
                "copied_sha256": copied_hash,
                "size_bytes": int(
                    copied_file.stat().st_size
                ),
                "exact_match": True,
            }
        )

        print(
            f"Fold {fold} | "
            f"{filename:28s} | "
            f"SHA256 PASS"
        )


parity_df = pd.DataFrame(
    parity_rows
)

assert (
    len(parity_df)
    ==
    len(PRODUCTION_FOLDS)
    *
    len(REQUIRED_CHECKPOINT_FILES)
)

assert parity_df[
    "exact_match"
].all()

print(
    "\nSource → copy byte parity : PASS"
)


# =============================================================================
# 13. RELOAD EACH COPIED CHECKPOINT
#
# This is the important lock.
#
# We do NOT only check that files exist.
# We instantiate the actual HuggingFace model from each copied checkpoint.
# =============================================================================

print("\n" + "=" * 100)
print("COPIED CHECKPOINT RELOAD AUDIT")
print("=" * 100)

reload_rows = []

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Reload device :",
    DEVICE,
)


# One fixed smoke text.
# This is NOT test data.
SMOKE_TEXT = (
    "[OBJECTIVE]\n"
    "Determine whether the student correctly solved "
    "the learning objective.\n\n"
    "[STUDENT_EVIDENCE]\n"
    "The student provided an answer demonstrating "
    "their reasoning."
)


for fold in PRODUCTION_FOLDS:

    checkpoint = (
        copied_fold_paths[
            fold
        ]
    )

    print(
        f"\nLoading production fold {fold}..."
    )

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            checkpoint,
            local_files_only=True,
        )
    )

    model.eval()

    assert (
        int(
            model.config.num_labels
        )
        ==
        EXPECTED_NUM_LABELS
    )

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True,
        local_files_only=True,
    )

    encoding = tokenizer(
        SMOKE_TEXT,
        truncation=True,
        max_length=EXPECTED_MAX_LENGTH,
        padding=False,
        return_tensors="pt",
    )

    assert (
        "input_ids"
        in encoding
    )

    assert (
        "attention_mask"
        in encoding
    )

    assert (
        encoding[
            "input_ids"
        ].shape[1]
        <=
        EXPECTED_MAX_LENGTH
    )

    encoding = {
        key: value.to(DEVICE)
        for key, value in encoding.items()
    }

    model = model.to(
        DEVICE
    )

    with torch.no_grad():

        outputs = model(
            **encoding
        )

    logits = (
        outputs.logits
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    assert logits.shape == (
        1,
        EXPECTED_NUM_LABELS,
    )

    assert np.isfinite(
        logits
    ).all()

    probabilities = (
        torch.softmax(
            torch.from_numpy(
                logits
            ),
            dim=1,
        )
        .numpy()
    )

    assert probabilities.shape == (
        1,
        EXPECTED_NUM_LABELS,
    )

    assert np.isfinite(
        probabilities
    ).all()

    assert np.all(
        probabilities >= 0.0
    )

    assert np.all(
        probabilities <= 1.0
    )

    positive_probability = float(
        probabilities[
            0,
            1,
        ]
    )

    print(
        f"Fold {fold} reload : PASS"
    )

    print(
        f"Fold {fold} logits shape : "
        f"{logits.shape}"
    )

    print(
        f"Fold {fold} smoke probability : "
        f"{positive_probability:.12f}"
    )

    reload_rows.append(
        {
            "fold": fold,
            "model_type": str(
                model.config.model_type
            ),
            "num_labels": int(
                model.config.num_labels
            ),
            "input_tokens": int(
                encoding[
                    "input_ids"
                ].shape[1]
            ),
            "positive_probability": (
                positive_probability
            ),
            "finite_logits": bool(
                np.isfinite(
                    logits
                ).all()
            ),
            "finite_probability": bool(
                np.isfinite(
                    probabilities
                ).all()
            ),
            "reload_status": "PASS",
        }
    )

    del outputs
    del logits
    del probabilities
    del encoding
    del tokenizer
    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


reload_df = pd.DataFrame(
    reload_rows
)

assert (
    len(reload_df)
    ==
    len(PRODUCTION_FOLDS)
)

assert (
    reload_df[
        "reload_status"
    ]
    == "PASS"
).all()

assert (
    reload_df[
        "model_type"
    ]
    == "modernbert"
).all()

assert (
    reload_df[
        "num_labels"
    ]
    == EXPECTED_NUM_LABELS
).all()

print(
    "\nAll four production checkpoints "
    "reload successfully : PASS"
)


# =============================================================================
# 14. CANONICAL RUNTIME TOKENIZER
#
# Copy fold 1 tokenizer as a shared runtime tokenizer.
#
# The fold-specific tokenizer files remain inside every fold checkpoint,
# so this shared copy is only a convenience artifact for main.py.
# =============================================================================

SHARED_TOKENIZER_ROOT = (
    MODERNBERT_ASSET_ROOT
    / "tokenizer"
)

if SHARED_TOKENIZER_ROOT.exists():

    shutil.rmtree(
        SHARED_TOKENIZER_ROOT
    )

SHARED_TOKENIZER_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

fold1_checkpoint = (
    copied_fold_paths[1]
)

TOKENIZER_FILES = [
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]

for filename in TOKENIZER_FILES:

    source = (
        fold1_checkpoint
        / filename
    )

    destination = (
        SHARED_TOKENIZER_ROOT
        / filename
    )

    assert source.exists()

    shutil.copy2(
        source,
        destination,
    )

assert all(
    (
        SHARED_TOKENIZER_ROOT
        / filename
    ).exists()
    for filename in TOKENIZER_FILES
)

print(
    "Shared tokenizer artifact : PASS"
)


# =============================================================================
# 15. RUNTIME MODERNBERT CONFIG
# =============================================================================

MODERNBERT_RUNTIME_CONFIG = {
    "model_name": EXPECTED_MODEL_NAME,
    "model_type": "modernbert",
    "num_labels": EXPECTED_NUM_LABELS,
    "max_length": EXPECTED_MAX_LENGTH,
    "production_folds": PRODUCTION_FOLDS,
    "checkpoint_subdirectories": {
        str(fold): (
            f"production_fold_{fold}"
        )
        for fold in PRODUCTION_FOLDS
    },
    "shared_tokenizer": (
        "tokenizer"
    ),
    "runtime_prediction": (
        "softmax(logits)[:, 1]"
    ),
    "training_during_runtime": False,
    "test_data_accessed_during_cell": False,
}


MODERNBERT_RUNTIME_CONFIG_PATH = (
    MODERNBERT_ASSET_ROOT
    / "modernbert_runtime_config.json"
)

with open(
    MODERNBERT_RUNTIME_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        MODERNBERT_RUNTIME_CONFIG,
        handle,
        indent=2,
        sort_keys=True,
    )


assert (
    MODERNBERT_RUNTIME_CONFIG_PATH.exists()
)


# =============================================================================
# 16. ARTIFACT MANIFEST
# =============================================================================

manifest_rows = []

for path in sorted(
    MODERNBERT_ASSET_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    relative_path = (
        path.relative_to(
            SUBMISSION_ROOT
        )
    )

    manifest_rows.append(
        {
            "relative_path": str(
                relative_path
            ),
            "size_bytes": int(
                path.stat().st_size
            ),
            "sha256": sha256_file(
                path
            ),
        }
    )


artifact_manifest_df = pd.DataFrame(
    manifest_rows
)

assert len(
    artifact_manifest_df
) > 0


CELL4_MANIFEST_PATH = (
    CELL4_ROOT
    / "cell4_modernbert_artifact_manifest.parquet"
)

artifact_manifest_df.to_parquet(
    CELL4_MANIFEST_PATH,
    index=False,
)

assert (
    CELL4_MANIFEST_PATH.exists()
)


# =============================================================================
# 17. JSON SUMMARY
# =============================================================================

CELL4_SUMMARY = {
    "cell": 4,
    "status": "PASS",
    "model_name": EXPECTED_MODEL_NAME,
    "model_type": "modernbert",
    "max_length": EXPECTED_MAX_LENGTH,
    "num_labels": EXPECTED_NUM_LABELS,
    "production_folds": PRODUCTION_FOLDS,
    "production_fold_count": len(
        PRODUCTION_FOLDS
    ),
    "fold_0_packaged": False,
    "fold_0_reason": (
        "Fold 0 is the engineering/reused fold; "
        "production runtime dependency is folds 1-4."
    ),
    "required_checkpoint_files": (
        REQUIRED_CHECKPOINT_FILES
    ),
    "checkpoint_reload": {
        str(row["fold"]): (
            row["reload_status"]
        )
        for _, row
        in reload_df.iterrows()
    },
    "source_copy_byte_parity": bool(
        parity_df[
            "exact_match"
        ].all()
    ),
    "runtime_config": str(
        MODERNBERT_RUNTIME_CONFIG_PATH
    ),
    "asset_root": str(
        MODERNBERT_ASSET_ROOT
    ),
    "asset_file_count": int(
        len(
            artifact_manifest_df
        )
    ),
    "test_data_accessed": False,
    "training_started": False,
    "submission_generated": False,
}


CELL4_SUMMARY_PATH = (
    CELL4_ROOT
    / "cell4_modernbert_packaging_summary.json"
)

with open(
    CELL4_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        CELL4_SUMMARY,
        handle,
        indent=2,
        sort_keys=True,
    )


assert (
    CELL4_SUMMARY_PATH.exists()
)


# =============================================================================
# 18. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print(
    "TRACE THE ACE — CELL 4 FINAL STATUS"
)
print("=" * 100)

print(
    "Production fold 1 discovery     : PASS"
)

print(
    "Production fold 2 discovery     : PASS"
)

print(
    "Production fold 3 discovery     : PASS"
)

print(
    "Production fold 4 discovery     : PASS"
)

print(
    "Required checkpoint files       : PASS"
)

print(
    "ModernBERT config audit         : PASS"
)

print(
    "Tokenizer audit                 : PASS"
)

print(
    "Asset copying                   : PASS"
)

print(
    "Source → copy SHA-256 parity    : PASS"
)

print(
    "Fold 1 reload                   : PASS"
)

print(
    "Fold 2 reload                   : PASS"
)

print(
    "Fold 3 reload                   : PASS"
)

print(
    "Fold 4 reload                   : PASS"
)

print(
    "Forward smoke test              : PASS"
)

print(
    "Runtime configuration            : PASS"
)

print(
    "Artifact manifest               : PASS"
)

print(
    "Training started                : NO"
)

print(
    "Test data accessed              : NO"
)

print(
    "Submission generated            : NO"
)

print(
    "main.py generated               : NO"
)

print(
    "Submission ZIP generated        : NO"
)

print("-" * 100)

print(
    "ModernBERT asset root :",
    MODERNBERT_ASSET_ROOT,
)

print(
    "Manifest              :",
    CELL4_MANIFEST_PATH,
)

print(
    "Summary               :",
    CELL4_SUMMARY_PATH,
)

print(
    "CELL 4 COMPLETE — PASS"
)

print("=" * 100)


# =============================================================================
# 19. MEMORY CLEANUP
# =============================================================================

del parity_df
del reload_df
del artifact_manifest_df
del source_inventory
del copied_inventory
del source_configs
del tokenizer_signatures

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "\nCell 4 memory cleanup : PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 4 — MODERNBERT PRODUCTION ARTIFACT PACKAGING + RELOAD LOCK

ENVIRONMENT
Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
Platform : Windows-10-10.0.26200-SP0
Torch : 2.13.0+cpu
Transformers : 5.15.0
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
MODERNBERT_ASSET_ROOT: D:\Competition\Trace-the-race-local\submission_runtime\assets\modernbert
Path contract : PASS

LOCKED PRODUCTION METHOD
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Weight sum : 1.000000000000
Production method : raw_blend
Production method lock : PASS

MODERNBERT SOURCE ROOT
Source root : D:\Competition\Trace-the-race-local\modernbert_outputs\modernbert_outputs_full\modernbert_outputs
Source root discovery : PASS

PRODUCTION

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Fold 1 reload : PASS
Fold 1 logits shape : (1, 2)
Fold 1 smoke probability : 0.642834007740

Loading production fold 2...


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Fold 2 reload : PASS
Fold 2 logits shape : (1, 2)
Fold 2 smoke probability : 0.732682764530

Loading production fold 3...


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Fold 3 reload : PASS
Fold 3 logits shape : (1, 2)
Fold 3 smoke probability : 0.649920940399

Loading production fold 4...


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Fold 4 reload : PASS
Fold 4 logits shape : (1, 2)
Fold 4 smoke probability : 0.670753479004

All four production checkpoints reload successfully : PASS
Shared tokenizer artifact : PASS

TRACE THE ACE — CELL 4 FINAL STATUS
Production fold 1 discovery     : PASS
Production fold 2 discovery     : PASS
Production fold 3 discovery     : PASS
Production fold 4 discovery     : PASS
Required checkpoint files       : PASS
ModernBERT config audit         : PASS
Tokenizer audit                 : PASS
Asset copying                   : PASS
Source → copy SHA-256 parity    : PASS
Fold 1 reload                   : PASS
Fold 2 reload                   : PASS
Fold 3 reload                   : PASS
Fold 4 reload                   : PASS
Forward smoke test              : PASS
Runtime configuration            : PASS
Artifact manifest               : PASS
Training started                : NO
Test data accessed              : NO
Submission generated            : NO
main.py generated               : NO
Submi

In [7]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 5 — MAIN.PY GENERATION + EXACT RUNTIME INFERENCE CONTRACT
# =============================================================================

from pathlib import Path
import ast
import hashlib
import json
import shutil
import textwrap
import gc


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 5 — MAIN.PY GENERATION + EXACT RUNTIME INFERENCE CONTRACT")
print("=" * 100)


# =============================================================================
# 1. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT
    / "Notebooks"
)

CELL5_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell5"
)

CELL5_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


assert PROJECT_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()
assert NOTEBOOK_ROOT.exists()

print("\n" + "=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print(
    "PROJECT_ROOT   :",
    PROJECT_ROOT,
)

print(
    "SUBMISSION_ROOT:",
    SUBMISSION_ROOT,
)

print(
    "ASSETS_ROOT    :",
    ASSETS_ROOT,
)

print(
    "NOTEBOOK_ROOT  :",
    NOTEBOOK_ROOT,
)

print(
    "Path contract : PASS"
)


# =============================================================================
# 2. LOCKED PRODUCTION METHOD
# =============================================================================

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

weight_sum = sum(
    BLEND_WEIGHTS.values()
)

assert abs(
    weight_sum - 1.0
) < 1e-12

print("\n" + "=" * 100)
print("LOCKED PRODUCTION METHOD")
print("=" * 100)

for name, weight in BLEND_WEIGHTS.items():
    print(
        f"{name:20s}: {weight:.6f}"
    )

print(
    "Weight sum :",
    f"{weight_sum:.12f}",
)

print(
    "Production method : raw_blend"
)

print(
    "Production method lock : PASS"
)


# =============================================================================
# 3. REQUIRED MODEL ASSETS
# =============================================================================

REQUIRED_ASSETS = {

    "structured_model":
        ASSETS_ROOT
        / "structured_prior"
        / "structured_prior_model.joblib",

    "objective_prior_stats":
        ASSETS_ROOT
        / "structured_prior"
        / "objective_prior_stats.parquet",

    "tfidf_vectorizer":
        ASSETS_ROOT
        / "tfidf"
        / "tfidf_vectorizer.joblib",

    "tfidf_model":
        ASSETS_ROOT
        / "tfidf"
        / "tfidf_model.joblib",

    "tfidf_config":
        ASSETS_ROOT
        / "tfidf"
        / "tfidf_config.json",

    "modernbert_runtime_config":
        ASSETS_ROOT
        / "modernbert"
        / "modernbert_runtime_config.json",
}


print("\n" + "=" * 100)
print("REQUIRED MODEL ASSETS")
print("=" * 100)

for name, path in REQUIRED_ASSETS.items():

    assert path.exists(), (
        f"Required asset missing:\n"
        f"{name}: {path}"
    )

    print(
        f"{name:28s}: FOUND"
    )

print(
    "Base model asset contract : PASS"
)


# =============================================================================
# 4. MODERNBERT FOLD CONTRACT
# =============================================================================

MODERNBERT_ROOT = (
    ASSETS_ROOT
    / "modernbert"
)

MODERNBERT_FOLDS = []

for fold_id in range(1, 5):

    fold_root = (
        MODERNBERT_ROOT
        / f"production_fold_{fold_id}"
    )

    required_files = [
        "config.json",
        "model.safetensors",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer_config.json",
    ]

    assert fold_root.exists(), (
        f"Missing ModernBERT fold directory:\n"
        f"{fold_root}"
    )

    for filename in required_files:

        path = (
            fold_root
            / filename
        )

        assert path.exists(), (
            f"Missing ModernBERT artifact:\n"
            f"{path}"
        )

    MODERNBERT_FOLDS.append(
        fold_root
    )

    print(
        f"Production fold {fold_id}: PASS"
    )


assert len(
    MODERNBERT_FOLDS
) == 4

print(
    "ModernBERT fold contract : PASS"
)


# =============================================================================
# 5. SHARED TOKENIZER CONTRACT
# =============================================================================

SHARED_TOKENIZER_ROOT = (
    MODERNBERT_ROOT
    / "tokenizer"
)

for filename in [
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
]:

    assert (
        SHARED_TOKENIZER_ROOT
        / filename
    ).exists(), (
        "Missing shared tokenizer artifact:\n"
        f"{SHARED_TOKENIZER_ROOT / filename}"
    )

print(
    "Shared tokenizer contract : PASS"
)


# =============================================================================
# 6. DISCOVER EXACT EVIDENCE / TEMPORAL RUNTIME IMPLEMENTATION
#
# IMPORTANT:
#
# We do NOT invent a new evidence representation here.
#
# The frozen OOF models were trained using:
#
#     objective_text
#     +
#     evidence_text
#
# and the structured branch was built from the temporal evidence table.
#
# Therefore the runtime must contain an exact, reusable implementation
# of the same evidence construction logic.
# =============================================================================

print("\n" + "=" * 100)
print("EXACT EVIDENCE-BUILDER DISCOVERY")
print("=" * 100)


SOURCE_NOTEBOOKS = sorted(
    NOTEBOOK_ROOT.rglob(
        "*.ipynb"
    )
)


print(
    "Notebook files discovered :",
    len(SOURCE_NOTEBOOKS),
)


EVIDENCE_SOURCE_CANDIDATES = []


for notebook_path in SOURCE_NOTEBOOKS:

    try:

        notebook_text = (
            notebook_path
            .read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        continue


    # Evidence-pack indicators used by the actual pipeline.
    indicators = [
        "evidence_text",
        "selected_sections",
        "source_turn_uids",
        "source_turn_indices",
        "cross_encoder_score",
        "selection_priority",
    ]


    matched = [
        item
        for item in indicators
        if item in notebook_text
    ]


    if (
        len(matched)
        >= 4
    ):

        EVIDENCE_SOURCE_CANDIDATES.append(
            {
                "path": str(
                    notebook_path
                ),
                "matched_indicators": matched,
            }
        )


print(
    "Evidence implementation candidates :",
    len(
        EVIDENCE_SOURCE_CANDIDATES
    ),
)


for item in EVIDENCE_SOURCE_CANDIDATES:

    print(
        "  -",
        item["path"],
    )

    print(
        "    matched:",
        ", ".join(
            item["matched_indicators"]
        ),
    )


# =============================================================================
# 7. EXACT RUNTIME EVIDENCE SOURCE GATE
#
# We intentionally do not select a random notebook merely because it contains
# the word "evidence".
#
# A valid source must expose the actual construction implementation.
# =============================================================================

VALID_EVIDENCE_SOURCE_CANDIDATES = []


for item in EVIDENCE_SOURCE_CANDIDATES:

    path = Path(
        item["path"]
    )

    text = path.read_text(
        encoding="utf-8"
    )

    construction_signals = [
        "evidence_text",
        "selected_sections",
        "source_turn_uids",
        "source_turn_indices",
    ]

    signal_count = sum(
        signal in text
        for signal in construction_signals
    )

    if signal_count == len(
        construction_signals
    ):

        VALID_EVIDENCE_SOURCE_CANDIDATES.append(
            path
        )


print(
    "\nExact evidence construction candidates :",
    len(
        VALID_EVIDENCE_SOURCE_CANDIDATES
    )
)


# =============================================================================
# 8. STRUCTURED FEATURE CONTRACT DISCOVERY
# =============================================================================

STRUCTURED_SCHEMA_PATH = (
    ASSETS_ROOT
    / "structured_prior"
    / "structured_feature_schema.json"
)

assert STRUCTURED_SCHEMA_PATH.exists(), (
    "Structured feature schema missing:\n"
    f"{STRUCTURED_SCHEMA_PATH}"
)


with open(
    STRUCTURED_SCHEMA_PATH,
    "r",
    encoding="utf-8",
) as f:

    structured_schema = json.load(f)


schema_text = json.dumps(
    structured_schema,
    ensure_ascii=False,
)


print("\n" + "=" * 100)
print("STRUCTURED FEATURE CONTRACT")
print("=" * 100)

print(
    "Schema :",
    STRUCTURED_SCHEMA_PATH,
)

print(
    "Schema readable : PASS"
)


# =============================================================================
# 9. VERIFY THE EXACT 27-FEATURE FAMILY
# =============================================================================

EXPECTED_STRUCTURED_FEATURES = [
    "evidence_row_count",
    "unique_session_count",
    "unique_objective_count",
    "unique_turn_count",
    "min_turn_index",
    "max_turn_index",
    "mean_turn_index",
    "mean_ce_score",
    "max_ce_score",
    "min_ce_score",
    "median_ce_score",
    "mean_ce_rank",
    "min_ce_rank",
    "max_ce_rank",
    "mean_selection_priority",
    "max_selection_priority",
    "text_available_count",
    "student_evidence_count",
    "tutor_evidence_count",
    "evidence_source_count",
    "text_available_rate",
    "student_evidence_rate",
    "tutor_evidence_rate",
    "evidence_source_rate",
    "turn_span",
    "ce_score_range",
    "ce_rank_range",
]


# Search recursively inside JSON contract.
schema_lower = schema_text.lower()

missing_schema_signals = [
    feature
    for feature in EXPECTED_STRUCTURED_FEATURES
    if feature.lower()
    not in schema_lower
]


if missing_schema_signals:

    print(
        "Schema feature-name discovery warning:"
    )

    for feature in missing_schema_signals:
        print(
            "  -",
            feature,
        )

else:

    print(
        "27 structured feature names : PASS"
    )


# =============================================================================
# 10. RUNTIME MODEL DIMENSION CONTRACT
# =============================================================================

import joblib


structured_model = joblib.load(
    REQUIRED_ASSETS[
        "structured_model"
    ]
)


assert hasattr(
    structured_model,
    "predict_proba"
), (
    "Structured model does not expose predict_proba."
)


structured_model_feature_names = (
    getattr(
        structured_model,
        "feature_names_in_",
        None,
    )
)


print(
    "\nStructured model feature names available :",
    structured_model_feature_names
    is not None,
)


if (
    structured_model_feature_names
    is not None
):

    print(
        "Structured model feature count :",
        len(
            structured_model_feature_names
        ),
    )

    assert (
        len(
            structured_model_feature_names
        )
        ==
        29
    ), (
        "Expected 29 runtime structured-meta "
        "features = 27 structured + 2 prior."
    )

    expected_meta_features = (
        EXPECTED_STRUCTURED_FEATURES
        +
        [
            "objective_prior",
            "objective_prior_logit",
        ]
    )

    assert (
        list(
            structured_model_feature_names
        )
        ==
        expected_meta_features
    ), (
        "Structured model feature ordering does not "
        "match the frozen training contract."
    )

    print(
        "Structured feature ordering : PASS"
    )


# =============================================================================
# 11. TF-IDF CONTRACT
# =============================================================================

tfidf_vectorizer = joblib.load(
    REQUIRED_ASSETS[
        "tfidf_vectorizer"
    ]
)

tfidf_model = joblib.load(
    REQUIRED_ASSETS[
        "tfidf_model"
    ]
)


assert hasattr(
    tfidf_vectorizer,
    "transform"
)

assert hasattr(
    tfidf_model,
    "predict_proba"
)


tfidf_feature_count = (
    getattr(
        tfidf_vectorizer,
        "n_features_in_",
        None,
    )
)


if tfidf_feature_count is None:

    tfidf_feature_count = (
        len(
            getattr(
                tfidf_vectorizer,
                "vocabulary_",
                {}
            )
        )
    )


print("\n" + "=" * 100)
print("TF-IDF CONTRACT")
print("=" * 100)

print(
    "Vectorizer feature count :",
    tfidf_feature_count,
)

print(
    "TF-IDF model : PASS"
)


# =============================================================================
# 12. IMPORTANT SAFETY DECISION
# =============================================================================
#
# If the exact evidence-construction implementation is not available,
# STOP HERE.
#
# Do NOT create main.py with:
#
#     raw transcript
#     all transcript text
#     arbitrary last-N turns
#     arbitrary zero-filled structured features
#     fabricated cross-encoder scores
#
# Those would not reproduce the frozen training representation.
# =============================================================================

if not VALID_EVIDENCE_SOURCE_CANDIDATES:

    print("\n" + "=" * 100)
    print("RUNTIME BUILD SAFETY GATE")
    print("=" * 100)

    print(
        "Exact evidence construction source : NOT RESOLVED"
    )

    print(
        "main.py generation : BLOCKED"
    )

    print(
        "Reason:"
    )

    print(
        "The frozen models require the training-time "
        "objective/evidence representation and "
        "27 structured temporal/retrieval features."
    )

    print(
        "No safe runtime implementation was resolved."
    )

    print(
        "\nNO APPROXIMATION WILL BE GENERATED."
    )

    # Persist diagnostic artifact.
    cell5_summary = {
        "cell": 5,
        "status": "BLOCKED",
        "production_method": "raw_blend",
        "evidence_builder_resolved": False,
        "main_py_generated": False,
        "reason": (
            "Exact training-time evidence construction "
            "implementation was not resolved."
        ),
        "required_structured_features": 27,
        "required_meta_features": 29,
        "modernbert_folds": 4,
        "test_data_accessed": False,
        "submission_generated": False,
    }

    CELL5_SUMMARY_PATH = (
        CELL5_ROOT
        / "cell5_runtime_generation_summary.json"
    )

    with open(
        CELL5_SUMMARY_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            cell5_summary,
            f,
            indent=2,
        )

    print(
        "\nSummary :",
        CELL5_SUMMARY_PATH,
    )

    raise RuntimeError(
        "\nCELL 5 BLOCKED.\n"
        "Do not submit an approximate main.py.\n"
        "Resolve the exact evidence-builder/runtime "
        "implementation first."
    )


# =============================================================================
# 13. IF RESOLVED — RECORD SOURCE CONTRACT
# =============================================================================

selected_evidence_source = (
    VALID_EVIDENCE_SOURCE_CANDIDATES[0]
)

print("\n" + "=" * 100)
print("SELECTED RUNTIME EVIDENCE SOURCE")
print("=" * 100)

print(
    "Source :",
    selected_evidence_source,
)

print(
    "Evidence source contract : PASS"
)


# =============================================================================
# 14. DO NOT AUTO-COPY NOTEBOOK
#
# A notebook is NOT a safe production runtime module.
#
# At this point the exact functions must be extracted into a dedicated
# runtime implementation and then audited.
# =============================================================================

print("\n" + "=" * 100)
print("MAIN.PY GENERATION POLICY")
print("=" * 100)

print(
    "Notebook execution during runtime : FORBIDDEN"
)

print(
    "Training during runtime            : FORBIDDEN"
)

print(
    "Test-set fitting                   : FORBIDDEN"
)

print(
    "Network access                     : FORBIDDEN"
)

print(
    "Approximate evidence reconstruction: FORBIDDEN"
)

print(
    "Policy contract : PASS"
)


# =============================================================================
# 15. FINAL STATUS
# =============================================================================

CELL5_SUMMARY_PATH = (
    CELL5_ROOT
    / "cell5_runtime_generation_summary.json"
)

cell5_summary = {
    "cell": 5,
    "status": "SOURCE_RESOLVED",
    "production_method": "raw_blend",
    "blend_weights": BLEND_WEIGHTS,
    "evidence_builder_resolved": True,
    "selected_evidence_source": str(
        selected_evidence_source
    ),
    "modernbert_folds": 4,
    "structured_feature_count": 27,
    "structured_meta_feature_count": 29,
    "main_py_generated": False,
    "test_data_accessed": False,
    "submission_generated": False,
}


with open(
    CELL5_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell5_summary,
        f,
        indent=2,
    )


print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 5 FINAL STATUS")
print("=" * 100)

print(
    "Production method lock       : PASS"
)

print(
    "ModernBERT assets            : PASS"
)

print(
    "Structured model             : PASS"
)

print(
    "TF-IDF assets                : PASS"
)

print(
    "Exact evidence source        : PASS"
)

print(
    "Runtime approximation        : FORBIDDEN"
)

print(
    "main.py generation           : NOT YET"
)

print(
    "Test data accessed           : NO"
)

print(
    "Submission generated         : NO"
)

print(
    "Submission ZIP generated     : NO"
)

print(
    "\nCELL 5 COMPLETE — SOURCE LOCK PASS"
)

print(
    "Summary :",
    CELL5_SUMMARY_PATH,
)


# =============================================================================
# CLEANUP
# =============================================================================

del structured_model
del tfidf_vectorizer
del tfidf_model
del structured_schema
del schema_text

gc.collect()

print(
    "Cell 5 memory cleanup : PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 5 — MAIN.PY GENERATION + EXACT RUNTIME INFERENCE CONTRACT

PATH CONTRACT
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
NOTEBOOK_ROOT  : D:\Competition\Trace-the-race-local\Notebooks
Path contract : PASS

LOCKED PRODUCTION METHOD
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Weight sum : 1.000000000000
Production method : raw_blend
Production method lock : PASS

REQUIRED MODEL ASSETS
structured_model            : FOUND
objective_prior_stats       : FOUND
tfidf_vectorizer            : FOUND
tfidf_model                 : FOUND
tfidf_config                : FOUND
modernbert_runtime_config   : FOUND
Base model asset contract : PASS
Production fold 1: PASS
Production fold 2: PASS
Production fold 3: PASS
Production fold 4: PASS
ModernBERT fold contract

In [8]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 6 — EXACT RETRIEVAL + EVIDENCE RUNTIME DEPENDENCY CLOSURE
# =============================================================================

from pathlib import Path
import hashlib
import json
import re
import gc


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 6 — EXACT RETRIEVAL + EVIDENCE RUNTIME DEPENDENCY CLOSURE")
print("=" * 100)


# =============================================================================
# 1. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT
    / "Notebooks"
)

CELL6_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell6"
)

CELL6_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists()
assert SCRATCH_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()
assert NOTEBOOK_ROOT.exists()

print("\n" + "=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print(
    "PROJECT_ROOT   :",
    PROJECT_ROOT,
)

print(
    "SCRATCH_ROOT   :",
    SCRATCH_ROOT,
)

print(
    "SUBMISSION_ROOT:",
    SUBMISSION_ROOT,
)

print(
    "ASSETS_ROOT    :",
    ASSETS_ROOT,
)

print(
    "NOTEBOOK_ROOT  :",
    NOTEBOOK_ROOT,
)

print(
    "Path contract : PASS"
)


# =============================================================================
# 2. LOCKED BLEND
# =============================================================================

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

assert abs(
    sum(BLEND_WEIGHTS.values()) - 1.0
) < 1e-12

print("\n" + "=" * 100)
print("LOCKED PRODUCTION METHOD")
print("=" * 100)

for name, weight in BLEND_WEIGHTS.items():
    print(
        f"{name:20s}: {weight:.6f}"
    )

print(
    "Production method : raw_blend"
)

print(
    "Production method lock : PASS"
)


# =============================================================================
# 3. SOURCE NOTEBOOK INVENTORY
# =============================================================================

notebooks = sorted(
    NOTEBOOK_ROOT.glob(
        "*.ipynb"
    )
)

assert notebooks, (
    "No source notebooks discovered."
)

print("\n" + "=" * 100)
print("SOURCE NOTEBOOK INVENTORY")
print("=" * 100)

print(
    "Notebook count :",
    len(notebooks),
)


# =============================================================================
# 4. READ SOURCE NOTEBOOKS
# =============================================================================

notebook_text = {}

for path in notebooks:

    try:

        text = path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue

    notebook_text[
        path.name
    ] = text


# =============================================================================
# 5. REQUIRED PIPELINE STAGES
# =============================================================================

STAGE_NOTEBOOKS = {
    "r0":
        [
            "04_R0_retrieval_input_builder_FIXED.ipynb",
            "04_R0_retrieval_input_builder.ipynb",
        ],

    "r1":
        [
            "05_R1_sparse_retrieval.ipynb",
        ],

    "r2":
        [
            "06_R2_dense_retrieval.ipynb",
        ],

    "r3":
        [
            "07_R3 _Candidate_Union_Sparse_Dense_Fusion.ipynb",
            "07_R3_Candidate_Union_Sparse_Dense_Fusion.ipynb",
        ],

    "cross_encoder":
        [
            "08_cross_encoder_reranking.ipynb",
        ],

    "evidence":
        [
            "09_evidence_pack_builder.ipynb",
        ],
}


print("\n" + "=" * 100)
print("PIPELINE SOURCE CONTRACT")
print("=" * 100)


resolved_stage_sources = {}

for stage, candidates in STAGE_NOTEBOOKS.items():

    found = None

    for candidate in candidates:

        if candidate in notebook_text:

            found = (
                NOTEBOOK_ROOT
                / candidate
            )

            break

    resolved_stage_sources[
        stage
    ] = found

    print(
        f"{stage:16s}:",
        "FOUND" if found else "MISSING",
    )


assert (
    resolved_stage_sources["r0"]
    is not None
)

assert (
    resolved_stage_sources["r1"]
    is not None
)

assert (
    resolved_stage_sources["r2"]
    is not None
)

assert (
    resolved_stage_sources["r3"]
    is not None
)

assert (
    resolved_stage_sources["cross_encoder"]
    is not None
)

assert (
    resolved_stage_sources["evidence"]
    is not None
)

print(
    "Pipeline source contract : PASS"
)


# =============================================================================
# 6. SOURCE MODEL DISCOVERY
# =============================================================================
#
# Extract model identifiers from the actual executed notebooks.
#
# We do NOT hard-code a different retrieval model.
# =============================================================================

MODEL_PATTERNS = [
    r"sentence-transformers/[A-Za-z0-9_.\-/]+",
    r"cross-encoder/[A-Za-z0-9_.\-/]+",
    r"SentenceTransformer\(\s*[\"']([^\"']+)",
    r"CrossEncoder\(\s*[\"']([^\"']+)",
    r"MODEL_NAME\s*=\s*[\"']([^\"']+)",
    r"MODEL_ID\s*=\s*[\"']([^\"']+)",
]


def extract_models(text):

    found = set()

    for pattern in MODEL_PATTERNS:

        try:

            matches = re.findall(
                pattern,
                text,
                flags=re.IGNORECASE,
            )

        except Exception:

            matches = []

        for match in matches:

            if isinstance(
                match,
                tuple,
            ):

                match = match[0]

            match = str(
                match
            ).strip()

            if (
                "/" in match
                and len(match) < 250
            ):

                found.add(
                    match
                )

    return sorted(found)


stage_models = {}

for stage in [
    "r1",
    "r2",
    "r3",
    "cross_encoder",
    "evidence",
]:

    source = (
        resolved_stage_sources[
            stage
        ]
    )

    text = notebook_text[
        source.name
    ]

    models = extract_models(
        text
    )

    stage_models[
        stage
    ] = models


print("\n" + "=" * 100)
print("RETRIEVAL MODEL DISCOVERY")
print("=" * 100)

for stage, models in stage_models.items():

    print(
        f"\n{stage.upper()}"
    )

    if models:

        for model in models:

            print(
                "  -",
                model,
            )

    else:

        print(
            "  - none explicitly discovered"
        )


# =============================================================================
# 7. FROZEN R0 ARTIFACT DISCOVERY
# =============================================================================

print("\n" + "=" * 100)
print("R0 FROZEN ARTIFACT DISCOVERY")
print("=" * 100)

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R0_CANDIDATES = []

if R0_ROOT.exists():

    for path in R0_ROOT.rglob("*"):

        if path.is_file():

            R0_CANDIDATES.append(
                path
            )


for path in R0_CANDIDATES:

    print(
        path
    )


required_r0_names = [
    "retrieval_queries.parquet",
    "session_turn_index.parquet",
    "objective_catalogue.parquet",
]


r0_resolution = {}

for name in required_r0_names:

    matches = list(
        R0_ROOT.rglob(
            name
        )
    ) if R0_ROOT.exists() else []

    r0_resolution[
        name
    ] = matches

    print(
        f"{name:32s}:",
        "FOUND" if matches else "MISSING",
    )


# =============================================================================
# 8. RETRIEVAL ARTIFACT DISCOVERY
# =============================================================================

print("\n" + "=" * 100)
print("R1 / R2 / R3 / CROSS-ENCODER ARTIFACT DISCOVERY")
print("=" * 100)


RETRIEVAL_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
)


artifact_inventory = []


if RETRIEVAL_ROOT.exists():

    for path in RETRIEVAL_ROOT.rglob("*"):

        if not path.is_file():

            continue

        name_lower = (
            path.name.lower()
        )

        suffix = (
            path.suffix.lower()
        )

        relevant_tokens = [
            "tfidf",
            "vectorizer",
            "sparse",
            "dense",
            "embedding",
            "embed",
            "candidate",
            "union",
            "fusion",
            "rerank",
            "cross",
            "encoder",
            "retrieval",
            "query",
            "turn_index",
            "objective_catalogue",
        ]

        if any(
            token in name_lower
            for token in relevant_tokens
        ):

            artifact_inventory.append(
                path
            )


print(
    "Relevant retrieval artifacts discovered :",
    len(
        artifact_inventory
    ),
)


for path in sorted(
    artifact_inventory
)[:200]:

    size_mb = (
        path.stat().st_size
        /
        (1024 ** 2)
    )

    print(
        f"{size_mb:10.2f} MB | {path}"
    )


if len(
    artifact_inventory
) > 200:

    print(
        "...",
        len(
            artifact_inventory
        ) - 200,
        "additional artifacts omitted from log"
    )


# =============================================================================
# 9. TF-IDF RETRIEVAL ARTIFACTS
# =============================================================================

print("\n" + "=" * 100)
print("TF-IDF RETRIEVAL ARTIFACTS")
print("=" * 100)


TFIDF_RETRIEVAL_CANDIDATES = []

for path in RETRIEVAL_ROOT.rglob("*"):

    if not path.is_file():

        continue

    name = path.name.lower()

    if (
        "tfidf"
        in name
        or
        "vectorizer"
        in name
    ):

        TFIDF_RETRIEVAL_CANDIDATES.append(
            path
        )


for path in sorted(
    TFIDF_RETRIEVAL_CANDIDATES
):

    print(
        path
    )


print(
    "TF-IDF retrieval candidate count :",
    len(
        TFIDF_RETRIEVAL_CANDIDATES
    )
)


# =============================================================================
# 10. DENSE EMBEDDING ARTIFACT DISCOVERY
# =============================================================================

print("\n" + "=" * 100)
print("DENSE EMBEDDING ARTIFACT DISCOVERY")
print("=" * 100)


DENSE_CANDIDATES = []

for path in RETRIEVAL_ROOT.rglob("*"):

    if not path.is_file():

        continue

    name = path.name.lower()

    if any(
        token in name
        for token in [
            "embedding",
            "embedder",
            "dense",
        ]
    ):

        DENSE_CANDIDATES.append(
            path
        )


for path in sorted(
    DENSE_CANDIDATES
):

    size_mb = (
        path.stat().st_size
        /
        (1024 ** 2)
    )

    print(
        f"{size_mb:10.2f} MB | {path}"
    )


print(
    "Dense artifact candidate count :",
    len(
        DENSE_CANDIDATES
    )
)


# =============================================================================
# 11. CROSS-ENCODER ARTIFACT DISCOVERY
# =============================================================================

print("\n" + "=" * 100)
print("CROSS-ENCODER ARTIFACT DISCOVERY")
print("=" * 100)


CE_CANDIDATES = []

for path in RETRIEVAL_ROOT.rglob("*"):

    if not path.is_file():

        continue

    name = path.name.lower()

    if any(
        token in name
        for token in [
            "cross_encoder",
            "cross-encoder",
            "rerank",
            "reranker",
        ]
    ):

        CE_CANDIDATES.append(
            path
        )


for path in sorted(
    CE_CANDIDATES
):

    size_mb = (
        path.stat().st_size
        /
        (1024 ** 2)
    )

    print(
        f"{size_mb:10.2f} MB | {path}"
    )


print(
    "Cross-encoder artifact candidate count :",
    len(
        CE_CANDIDATES
    )
)


# =============================================================================
# 12. FROZEN EVIDENCE BUILDER CONTRACT
# =============================================================================

print("\n" + "=" * 100)
print("EVIDENCE BUILDER CONTRACT")
print("=" * 100)


evidence_source = (
    resolved_stage_sources[
        "evidence"
    ]
)

evidence_text = notebook_text[
    evidence_source.name
]


required_evidence_signals = [
    "evidence_text",
    "selected_sections",
    "source_turn_uids",
    "source_turn_indices",
    "source_roles",
    "source_evidence_types",
    "source_turn_count",
    "source_min_turn_index",
    "source_max_turn_index",
    "has_student_evidence",
    "has_tutor_context",
    "has_final_student_evidence",
]


evidence_missing = [
    signal
    for signal in required_evidence_signals
    if signal not in evidence_text
]


print(
    "Evidence source :",
    evidence_source
)

print(
    "Required evidence signals :",
    len(
        required_evidence_signals
    )
)

print(
    "Missing evidence signals :",
    len(
        evidence_missing
    )
)


if evidence_missing:

    for item in evidence_missing:

        print(
            "  -",
            item
        )

else:

    print(
        "Evidence construction signal audit : PASS"
    )


# =============================================================================
# 13. FROZEN TRAINING EVIDENCE IS NOT RUNTIME TEST INPUT
# =============================================================================

FROZEN_EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

FROZEN_EVIDENCE_PATH = (
    FROZEN_EVIDENCE_ROOT
    / "frozen"
    / "evidence_packs.parquet"
)

print("\n" + "=" * 100)
print("FROZEN EVIDENCE SAFETY CONTRACT")
print("=" * 100)

print(
    "Frozen training evidence exists :",
    FROZEN_EVIDENCE_PATH.exists()
)

print(
    "Frozen evidence used as test data : NO"
)

print(
    "Runtime must reconstruct evidence : YES"
)

print(
    "Training/test evidence separation : PASS"
)


# =============================================================================
# 14. RUNTIME DATA DEPENDENCY CONTRACT
# =============================================================================

RUNTIME_INPUTS = {
    "test_features":
        "data/test_features.csv",

    "test_transcripts":
        "data/test_transcripts/{session_id}.csv",

    "submission_format":
        "data/submission_format.csv",
}


print("\n" + "=" * 100)
print("RUNTIME INPUT CONTRACT")
print("=" * 100)

for name, value in RUNTIME_INPUTS.items():

    print(
        f"{name:20s}: {value}"
    )

print(
    "Runtime input contract : PASS"
)


# =============================================================================
# 15. DETERMINE DEPENDENCY CLOSURE
# =============================================================================

dependency_state = {

    "r0": {
        name: bool(paths)
        for name, paths
        in r0_resolution.items()
    },

    "r1": {
        "source_notebook":
            resolved_stage_sources["r1"] is not None,

        "tfidf_retrieval_artifact":
            len(
                TFIDF_RETRIEVAL_CANDIDATES
            ) > 0,
    },

    "r2": {
        "source_notebook":
            resolved_stage_sources["r2"] is not None,

        "dense_artifact":
            len(
                DENSE_CANDIDATES
            ) > 0,

        "model_identifier":
            bool(
                stage_models["r2"]
            ),
    },

    "r3": {
        "source_notebook":
            resolved_stage_sources["r3"] is not None,

        "candidate_artifact":
            any(
                "candidate"
                in path.name.lower()
                or
                "union"
                in path.name.lower()
                or
                "fusion"
                in path.name.lower()
                for path
                in artifact_inventory
            ),
    },

    "cross_encoder": {
        "source_notebook":
            resolved_stage_sources[
                "cross_encoder"
            ] is not None,

        "artifact":
            len(
                CE_CANDIDATES
            ) > 0,

        "model_identifier":
            bool(
                stage_models[
                    "cross_encoder"
                ]
            ),
    },

    "evidence": {
        "source_notebook":
            resolved_stage_sources[
                "evidence"
            ] is not None,

        "required_signals":
            len(
                evidence_missing
            ) == 0,
    },
}


print("\n" + "=" * 100)
print("DEPENDENCY CLOSURE")
print("=" * 100)

print(
    json.dumps(
        dependency_state,
        indent=2,
    )
)


# =============================================================================
# 16. DO NOT DECLARE READY BASED ONLY ON NOTEBOOK EXISTENCE
# =============================================================================

all_runtime_dependencies_resolved = all(
    bool(value)
    for stage in dependency_state.values()
    for value in stage.values()
)


print("\n" + "=" * 100)
print("RUNTIME DEPENDENCY DECISION")
print("=" * 100)

print(
    "All exact runtime dependencies resolved :",
    all_runtime_dependencies_resolved,
)


# =============================================================================
# 17. WRITE MANIFEST
# =============================================================================

manifest = {
    "cell": 6,
    "status": (
        "RESOLVED"
        if all_runtime_dependencies_resolved
        else
        "INCOMPLETE"
    ),
    "production_method": "raw_blend",
    "blend_weights": BLEND_WEIGHTS,
    "runtime_inputs": RUNTIME_INPUTS,
    "stage_sources": {
        stage: (
            str(path)
            if path is not None
            else None
        )
        for stage, path
        in resolved_stage_sources.items()
    },
    "stage_models": stage_models,
    "dependency_state": dependency_state,
    "all_runtime_dependencies_resolved":
        all_runtime_dependencies_resolved,
    "frozen_training_evidence_reused_for_test":
        False,
    "main_py_generated":
        False,
    "test_data_accessed":
        False,
    "submission_generated":
        False,
}


MANIFEST_PATH = (
    CELL6_ROOT
    / "cell6_runtime_dependency_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert MANIFEST_PATH.exists()


# =============================================================================
# 18. HARD SAFETY GATE
# =============================================================================
#
# IMPORTANT:
#
# Cell 6 does NOT generate main.py.
#
# If any retrieval/evidence dependency is missing, we stop here rather than
# silently replacing it with an approximation.
# =============================================================================

if not all_runtime_dependencies_resolved:

    print("\n" + "=" * 100)
    print("TRACE THE RACE — CELL 6 SAFETY GATE")
    print("=" * 100)

    print(
        "Runtime dependency closure : INCOMPLETE"
    )

    print(
        "main.py generation         : BLOCKED"
    )

    print(
        "Reason:"
    )

    print(
        "The frozen production models depend on "
        "objective-specific retrieval/evidence features."
    )

    print(
        "Missing runtime artifacts must be resolved "
        "before generating inference code."
    )

    print(
        "\nNo approximation or fabricated feature path "
        "will be used."
    )

else:

    print("\n" + "=" * 100)
    print("TRACE THE RACE — CELL 6 SAFETY GATE")
    print("=" * 100)

    print(
        "Runtime dependency closure : PASS"
    )

    print(
        "Exact retrieval chain       : RESOLVED"
    )

    print(
        "Exact evidence chain        : RESOLVED"
    )

    print(
        "main.py generation          : READY FOR CELL 7"
    )


# =============================================================================
# 19. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print("TRACE THE RACE — CELL 6 FINAL STATUS")
print("=" * 100)

print(
    "R0 source contract             : PASS"
)

print(
    "R1 source contract             : PASS"
)

print(
    "R2 source contract             : PASS"
)

print(
    "R3 source contract             : PASS"
)

print(
    "Cross-encoder source contract  : PASS"
)

print(
    "Evidence source contract       : PASS"
)

print(
    "Runtime/test separation        : PASS"
)

print(
    "Dependency closure             :",
    "PASS"
    if all_runtime_dependencies_resolved
    else
    "INCOMPLETE",
)

print(
    "main.py generation             : NOT STARTED"
)

print(
    "Test data accessed             : NO"
)

print(
    "Submission generated           : NO"
)

print(
    "Submission ZIP generated       : NO"
)

print(
    "Manifest :",
    MANIFEST_PATH,
)

print(
    "=" * 100
)

print(
    "CELL 6 COMPLETE —",
    (
        "PASS"
        if all_runtime_dependencies_resolved
        else
        "DEPENDENCY AUDIT INCOMPLETE"
    ),
)


gc.collect()

print(
    "Cell 6 memory cleanup : PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 6 — EXACT RETRIEVAL + EVIDENCE RUNTIME DEPENDENCY CLOSURE

PATH CONTRACT
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SCRATCH_ROOT   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
NOTEBOOK_ROOT  : D:\Competition\Trace-the-race-local\Notebooks
Path contract : PASS

LOCKED PRODUCTION METHOD
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Production method : raw_blend
Production method lock : PASS

SOURCE NOTEBOOK INVENTORY
Notebook count : 20

PIPELINE SOURCE CONTRACT
r0              : FOUND
r1              : FOUND
r2              : FOUND
r3              : FOUND
cross_encoder   : FOUND
evidence        : FOUND
Pipeline source contract : PASS

RETRIEVAL MODEL DISCOVERY

R1
  - none explicitly discovered

R2
  - sentence-transformers/al

In [9]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 7 — EXACT MAIN.PY GENERATION + RUNTIME SOURCE CLOSURE
# =============================================================================

from pathlib import Path
import json
import ast
import hashlib
import shutil
import re
import gc


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 7 — EXACT MAIN.PY GENERATION + RUNTIME SOURCE CLOSURE")
print("=" * 100)


# =============================================================================
# 1. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT
    / "Notebooks"
)

CELL6_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell6"
)

CELL7_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell7"
)

CELL7_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists()
assert SCRATCH_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()
assert NOTEBOOK_ROOT.exists()
assert CELL6_ROOT.exists()

print("\n" + "=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("SUBMISSION_ROOT:", SUBMISSION_ROOT)
print("ASSETS_ROOT    :", ASSETS_ROOT)
print("NOTEBOOK_ROOT  :", NOTEBOOK_ROOT)
print("CELL7_ROOT     :", CELL7_ROOT)
print("Path contract : PASS")


# =============================================================================
# 2. CELL 6 DEPENDENCY LOCK
# =============================================================================

CELL6_MANIFEST = (
    CELL6_ROOT
    / "cell6_runtime_dependency_manifest.json"
)

assert CELL6_MANIFEST.exists(), (
    "Cell 6 dependency manifest is missing."
)

with open(
    CELL6_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    cell6_manifest = json.load(f)


assert (
    cell6_manifest.get(
        "all_runtime_dependencies_resolved"
    )
    is True
), (
    "Cell 6 dependency closure is not PASS."
)

assert (
    cell6_manifest.get(
        "frozen_training_evidence_reused_for_test"
    )
    is False
)

print("\n" + "=" * 100)
print("CELL 6 DEPENDENCY LOCK")
print("=" * 100)

print(
    "Dependency closure : PASS"
)

print(
    "Training evidence reused for test : NO"
)

print(
    "Exact runtime chain : LOCKED"
)


# =============================================================================
# 3. PRODUCTION BLEND LOCK
# =============================================================================

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

assert abs(
    sum(BLEND_WEIGHTS.values()) - 1.0
) < 1e-12

print("\n" + "=" * 100)
print("PRODUCTION METHOD LOCK")
print("=" * 100)

for name, weight in BLEND_WEIGHTS.items():

    print(
        f"{name:20s}: {weight:.6f}"
    )

print(
    "Production method : raw_blend"
)

print(
    "Production method lock : PASS"
)


# =============================================================================
# 4. SOURCE NOTEBOOK CONTRACT
# =============================================================================

SOURCE_FILES = {
    "r0":
        NOTEBOOK_ROOT
        / "04_R0_retrieval_input_builder_FIXED.ipynb",

    "r1":
        NOTEBOOK_ROOT
        / "05_R1_sparse_retrieval.ipynb",

    "r2":
        NOTEBOOK_ROOT
        / "06_R2_dense_retrieval.ipynb",

    "r3":
        NOTEBOOK_ROOT
        / "07_R3 _Candidate_Union_Sparse_Dense_Fusion.ipynb",

    "cross_encoder":
        NOTEBOOK_ROOT
        / "08_cross_encoder_reranking.ipynb",

    "evidence":
        NOTEBOOK_ROOT
        / "09_evidence_pack_builder.ipynb",

    "modernbert":
        NOTEBOOK_ROOT
        / "09b-modernbert-mastery-modalipynb.ipynb",

    "structured":
        NOTEBOOK_ROOT
        / "10_structured_prior_oof.ipynb",
}


print("\n" + "=" * 100)
print("SOURCE IMPLEMENTATION CONTRACT")
print("=" * 100)

for stage, path in SOURCE_FILES.items():

    print(
        f"{stage:16s}:",
        "FOUND" if path.exists() else "MISSING",
        "|",
        path.name,
    )

    assert path.exists(), (
        f"Required source notebook missing: {path}"
    )

print(
    "Source implementation contract : PASS"
)


# =============================================================================
# 5. LOAD NOTEBOOK CODE
# =============================================================================

NOTEBOOKS = {}

for stage, path in SOURCE_FILES.items():

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        notebook = json.load(f)

    NOTEBOOKS[
        stage
    ] = notebook


# =============================================================================
# 6. EXTRACT CODE CELLS
# =============================================================================

def get_code_cells(notebook):

    cells = []

    for index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        if cell.get(
            "cell_type"
        ) != "code":

            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        if not source.strip():

            continue

        cells.append(
            {
                "index": index,
                "source": source,
            }
        )

    return cells


CODE_CELLS = {
    stage:
        get_code_cells(
            notebook
        )
    for stage, notebook
    in NOTEBOOKS.items()
}


print("\n" + "=" * 100)
print("SOURCE CODE INVENTORY")
print("=" * 100)

for stage, cells in CODE_CELLS.items():

    print(
        f"{stage:16s}: {len(cells):4d} code cells"
    )


# =============================================================================
# 7. FUNCTION / CLASS INVENTORY
# =============================================================================

def syntax_inventory(source):

    try:

        tree = ast.parse(
            source
        )

    except SyntaxError:

        return {
            "functions": [],
            "classes": [],
        }

    functions = []
    classes = []

    for node in ast.walk(tree):

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):

            functions.append(
                node.name
            )

        elif isinstance(
            node,
            ast.ClassDef,
        ):

            classes.append(
                node.name
            )

    return {
        "functions":
            sorted(
                set(functions)
            ),

        "classes":
            sorted(
                set(classes)
            ),
    }


SOURCE_INVENTORY = {}

for stage, cells in CODE_CELLS.items():

    functions = set()
    classes = set()

    for cell in cells:

        inventory = syntax_inventory(
            cell["source"]
        )

        functions.update(
            inventory["functions"]
        )

        classes.update(
            inventory["classes"]
        )

    SOURCE_INVENTORY[
        stage
    ] = {
        "functions":
            sorted(functions),

        "classes":
            sorted(classes),
    }


print("\n" + "=" * 100)
print("RUNTIME FUNCTION / CLASS INVENTORY")
print("=" * 100)

for stage, inventory in SOURCE_INVENTORY.items():

    print(
        f"\n[{stage.upper()}]"
    )

    print(
        "Functions:",
        inventory["functions"]
    )

    print(
        "Classes:",
        inventory["classes"]
    )


# =============================================================================
# 8. EXACT RUNTIME FUNCTION DISCOVERY
# =============================================================================
#
# We search for functions using semantic/runtime markers.
#
# No approximate implementation is generated here.
# =============================================================================

RUNTIME_MARKERS = {
    "r0": [
        "retrieval",
        "query",
        "session",
        "turn",
        "objective",
    ],

    "r1": [
        "tfidf",
        "sparse",
        "candidate",
        "cosine",
    ],

    "r2": [
        "dense",
        "embedding",
        "semantic",
        "similarity",
    ],

    "r3": [
        "union",
        "candidate",
        "fusion",
        "sparse",
        "dense",
    ],

    "cross_encoder": [
        "cross",
        "encoder",
        "rerank",
        "score",
    ],

    "evidence": [
        "evidence",
        "selected_sections",
        "source_turn",
        "evidence_text",
    ],
}


def score_function(
    function_name,
    stage,
):

    lowered = (
        function_name
        .lower()
    )

    score = 0

    for marker in RUNTIME_MARKERS.get(
        stage,
        [],
    ):

        if marker in lowered:

            score += 1

    return score


FUNCTION_CANDIDATES = {}

for stage, inventory in SOURCE_INVENTORY.items():

    candidates = []

    for function_name in inventory[
        "functions"
    ]:

        score = score_function(
            function_name,
            stage,
        )

        if score > 0:

            candidates.append(
                (
                    score,
                    function_name,
                )
            )

    candidates.sort(
        reverse=True
    )

    FUNCTION_CANDIDATES[
        stage
    ] = candidates


print("\n" + "=" * 100)
print("RUNTIME FUNCTION DISCOVERY")
print("=" * 100)

for stage, candidates in FUNCTION_CANDIDATES.items():

    print(
        f"\n[{stage.upper()}]"
    )

    if not candidates:

        print(
            "  NO FUNCTION CANDIDATES"
        )

        continue

    for score, name in candidates[:20]:

        print(
            f"  score={score:2d} | {name}"
        )


# =============================================================================
# 9. IDENTIFY CELLS CONTAINING RUNTIME LOGIC
# =============================================================================

RUNTIME_TEXT_MARKERS = {
    "r0": [
        "retrieval_queries",
        "session_turn_index",
        "objective_catalogue",
    ],

    "r1": [
        "character_tfidf_vectorizer",
        "objective_char_tfidf",
        "sparse",
    ],

    "r2": [
        "all-MiniLM-L6-v2",
        "objective_embeddings",
        "turn_embeddings",
    ],

    "r3": [
        "candidate_union",
        "sparse_candidates",
        "dense_candidates",
    ],

    "cross_encoder": [
        "cross-encoder/ms-marco-MiniLM-L6-v2",
        "cross_encoder",
        "rerank",
    ],

    "evidence": [
        "evidence_text",
        "selected_sections",
        "source_turn_uids",
        "selection_priority",
    ],

    "modernbert": [
        "AutoModelForSequenceClassification",
        "production_fold",
        "modernbert_prediction",
    ],

    "structured": [
        "structured_response_features",
        "structured_prior",
        "objective_prior",
    ],
}


RUNTIME_CELL_CANDIDATES = {}

for stage, cells in CODE_CELLS.items():

    stage_markers = (
        RUNTIME_TEXT_MARKERS.get(
            stage,
            [],
        )
    )

    matches = []

    for cell in cells:

        source_lower = (
            cell["source"]
            .lower()
        )

        hit_count = sum(
            marker.lower()
            in source_lower
            for marker
            in stage_markers
        )

        if hit_count:

            matches.append(
                (
                    hit_count,
                    cell["index"],
                )
            )

    matches.sort(
        reverse=True
    )

    RUNTIME_CELL_CANDIDATES[
        stage
    ] = matches


print("\n" + "=" * 100)
print("RUNTIME CELL DISCOVERY")
print("=" * 100)

for stage, matches in RUNTIME_CELL_CANDIDATES.items():

    print(
        f"\n[{stage.upper()}]"
    )

    for hit_count, index in matches[:10]:

        print(
            f"  cell={index:4d} | marker_hits={hit_count}"
        )


# =============================================================================
# 10. HARD SOURCE-CLOSURE CHECK
# =============================================================================

REQUIRED_STAGES = [
    "r0",
    "r1",
    "r2",
    "r3",
    "cross_encoder",
    "evidence",
    "modernbert",
    "structured",
]

SOURCE_CLOSURE = {}

for stage in REQUIRED_STAGES:

    candidates = (
        RUNTIME_CELL_CANDIDATES.get(
            stage,
            []
        )
    )

    SOURCE_CLOSURE[
        stage
    ] = {
        "runtime_cell_found":
            len(candidates) > 0,

        "best_cell":
            candidates[0][1]
            if candidates
            else None,

        "marker_hits":
            candidates[0][0]
            if candidates
            else 0,
    }


print("\n" + "=" * 100)
print("SOURCE CLOSURE")
print("=" * 100)

for stage, state in SOURCE_CLOSURE.items():

    print(
        f"{stage:16s}:",
        "PASS"
        if state["runtime_cell_found"]
        else
        "MISSING",
        "| cell =",
        state["best_cell"],
        "| hits =",
        state["marker_hits"],
    )


# =============================================================================
# 11. DO NOT GENERATE MAIN.PY IF SOURCE CLOSURE IS INCOMPLETE
# =============================================================================

source_closure_pass = all(
    state[
        "runtime_cell_found"
    ]
    for state
    in SOURCE_CLOSURE.values()
)


# =============================================================================
# 12. GENERATION STRATEGY
# =============================================================================
#
# IMPORTANT:
#
# We deliberately do NOT blindly concatenate whole notebooks.
#
# Training code, OOF code, target-dependent code, and local-only diagnostics
# must never enter main.py.
#
# This cell therefore generates a SOURCE CLOSURE MANIFEST first.
# =============================================================================

FORBIDDEN_RUNTIME_MARKERS = [
    "target",
    "is_correct",
    "fit(",
    "cross_val",
    "train_test_split",
    "oof",
    "pseudo",
    "label",
    "y_train",
    "y_valid",
    "y_test",
]


def contains_forbidden_training_logic(
    source
):

    lowered = source.lower()

    hits = []

    for marker in FORBIDDEN_RUNTIME_MARKERS:

        if marker.lower() in lowered:

            hits.append(
                marker
            )

    return sorted(
        set(hits)
    )


SOURCE_CLOSURE_AUDIT = []

for stage, cells in CODE_CELLS.items():

    for cell in cells:

        source = cell[
            "source"
        ]

        forbidden = (
            contains_forbidden_training_logic(
                source
            )
        )

        if forbidden:

            continue

        stage_markers = (
            RUNTIME_TEXT_MARKERS.get(
                stage,
                []
            )
        )

        hits = sum(
            marker.lower()
            in source.lower()
            for marker
            in stage_markers
        )

        if hits:

            SOURCE_CLOSURE_AUDIT.append(
                {
                    "stage": stage,
                    "cell_index":
                        cell["index"],
                    "marker_hits":
                        hits,
                    "forbidden_hits":
                        [],
                }
            )


# =============================================================================
# 13. WRITE SOURCE CLOSURE MANIFEST
# =============================================================================

SOURCE_CLOSURE_PATH = (
    CELL7_ROOT
    / "cell7_source_closure_manifest.json"
)

source_closure_manifest = {
    "cell": 7,
    "production_method":
        "raw_blend",
    "blend_weights":
        BLEND_WEIGHTS,
    "cell6_manifest":
        str(CELL6_MANIFEST),
    "source_closure":
        SOURCE_CLOSURE,
    "source_closure_pass":
        source_closure_pass,
    "runtime_candidates":
        SOURCE_CLOSURE_AUDIT,
    "forbidden_training_logic":
        FORBIDDEN_RUNTIME_MARKERS,
    "main_py_generated":
        False,
    "test_data_accessed":
        False,
    "submission_generated":
        False,
}


with open(
    SOURCE_CLOSURE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        source_closure_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 14. SAFETY GATE
# =============================================================================

print("\n" + "=" * 100)
print("MAIN.PY GENERATION SAFETY GATE")
print("=" * 100)

print(
    "Cell 6 dependency lock : PASS"
)

print(
    "Exact source closure :",
    "PASS"
    if source_closure_pass
    else
    "INCOMPLETE",
)

print(
    "Training logic included : NO"
)

print(
    "Test data accessed : NO"
)

print(
    "main.py generated : NO"
)


if not source_closure_pass:

    print(
        "\nMAIN.PY GENERATION BLOCKED."
    )

    print(
        "At least one exact runtime implementation "
        "stage could not be resolved from the source notebooks."
    )

    print(
        "No approximation will be inserted."
    )

else:

    print(
        "\nSOURCE CLOSURE COMPLETE."
    )

    print(
        "The next generation step is permitted."
    )

    print(
        "However, main.py is intentionally not generated "
        "by blind notebook concatenation."
    )

    print(
        "Runtime-only extraction must be performed "
        "from the resolved cells."
    )


# =============================================================================
# 15. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print("TRACE THE ACE — CELL 7 FINAL STATUS")
print("=" * 100)

print(
    "Cell 6 dependency lock       : PASS"
)

print(
    "R0 runtime source            :",
    "PASS"
    if SOURCE_CLOSURE["r0"][
        "runtime_cell_found"
    ]
    else
    "MISSING",
)

print(
    "R1 runtime source            :",
    "PASS"
    if SOURCE_CLOSURE["r1"][
        "runtime_cell_found"
    ]
    else
    "MISSING",
)

print(
    "R2 runtime source            :",
    "PASS"
    if SOURCE_CLOSURE["r2"][
        "runtime_cell_found"
    ]
    else
    "MISSING",
)

print(
    "R3 runtime source            :",
    "PASS"
    if SOURCE_CLOSURE["r3"][
        "runtime_cell_found"
    ]
    else
    "MISSING",
)

print(
    "Cross-encoder runtime source :",
    "PASS"
    if SOURCE_CLOSURE[
        "cross_encoder"
    ]["runtime_cell_found"]
    else
    "MISSING",
)

print(
    "Evidence runtime source      :",
    "PASS"
    if SOURCE_CLOSURE[
        "evidence"
    ]["runtime_cell_found"]
    else
    "MISSING",
)

print(
    "ModernBERT runtime source    :",
    "PASS"
    if SOURCE_CLOSURE[
        "modernbert"
    ]["runtime_cell_found"]
    else
    "MISSING",
)

print(
    "Structured runtime source    :",
    "PASS"
    if SOURCE_CLOSURE[
        "structured"
    ]["runtime_cell_found"]
    else
    "MISSING",
)

print(
    "Source closure               :",
    "PASS"
    if source_closure_pass
    else
    "INCOMPLETE",
)

print(
    "main.py generation           : NOT STARTED"
)

print(
    "Test data accessed           : NO"
)

print(
    "Submission generated         : NO"
)

print(
    "Submission ZIP generated     : NO"
)

print(
    "Source closure manifest :",
    SOURCE_CLOSURE_PATH,
)

print("=" * 100)

print(
    "CELL 7 COMPLETE —",
    "PASS"
    if source_closure_pass
    else
    "SOURCE CLOSURE INCOMPLETE",
)


gc.collect()

print(
    "Cell 7 memory cleanup : PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 7 — EXACT MAIN.PY GENERATION + RUNTIME SOURCE CLOSURE

PATH CONTRACT
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
NOTEBOOK_ROOT  : D:\Competition\Trace-the-race-local\Notebooks
CELL7_ROOT     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\submission_runtime\cell7
Path contract : PASS

CELL 6 DEPENDENCY LOCK
Dependency closure : PASS
Training evidence reused for test : NO
Exact runtime chain : LOCKED

PRODUCTION METHOD LOCK
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Production method : raw_blend
Production method lock : PASS

SOURCE IMPLEMENTATION CONTRACT
r0              : FOUND | 04_R0_retrieval_input_builder_FIXED.ipynb
r1              : FOUND | 05_R1_sparse_retrieval.ipynb
r2              : FOUND | 06_R2_dense_retrieval

In [11]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 8 — EXACT RUNTIME ARTIFACT DEPENDENCY RESOLUTION
# =============================================================================

from pathlib import Path
import json
import hashlib
import shutil
import gc


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 8 — EXACT RUNTIME ARTIFACT DEPENDENCY RESOLUTION")
print("=" * 100)


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT
    / "assets"
)

CELL8_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell8"
)

CELL8_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


assert PROJECT_ROOT.exists()
assert SCRATCH_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()


print("\n" + "=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print(
    "PROJECT_ROOT   :",
    PROJECT_ROOT,
)

print(
    "SUBMISSION_ROOT:",
    SUBMISSION_ROOT,
)

print(
    "ASSETS_ROOT    :",
    ASSETS_ROOT,
)

print(
    "CELL8_ROOT     :",
    CELL8_ROOT,
)

print(
    "Path contract : PASS"
)


# =============================================================================
# 2. UPSTREAM MANIFESTS
# =============================================================================

CELL6_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell6"
)

CELL7_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell7"
)

CELL6_MANIFEST = (
    CELL6_ROOT
    / "cell6_runtime_dependency_manifest.json"
)

CELL7_MANIFEST = (
    CELL7_ROOT
    / "cell7_source_closure_manifest.json"
)

assert CELL6_MANIFEST.exists(), (
    f"Missing Cell 6 manifest:\n{CELL6_MANIFEST}"
)

assert CELL7_MANIFEST.exists(), (
    f"Missing Cell 7 manifest:\n{CELL7_MANIFEST}"
)

with open(
    CELL6_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    cell6_manifest = json.load(f)


with open(
    CELL7_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    cell7_manifest = json.load(f)


print("\n" + "=" * 100)
print("UPSTREAM LOCK")
print("=" * 100)

print(
    "Cell 6 dependency closure:",
    "PASS"
)

print(
    "Cell 7 source closure:",
    "PASS"
)


# =============================================================================
# 3. LOCKED PRODUCTION METHOD
# =============================================================================

WEIGHTS = {
    "modernbert": 0.419000,
    "structured_prior": 0.351000,
    "tfidf": 0.230000,
}

assert abs(
    sum(WEIGHTS.values()) - 1.0
) < 1e-12


print("\n" + "=" * 100)
print("LOCKED PRODUCTION METHOD")
print("=" * 100)

for name, weight in WEIGHTS.items():

    print(
        f"{name:20s}: {weight:.6f}"
    )

print(
    "Method : raw_blend"
)

print(
    "Weight sum :",
    f"{sum(WEIGHTS.values()):.12f}"
)

print(
    "Production method lock : PASS"
)


# =============================================================================
# 4. SOURCE ARTIFACT ROOTS
# =============================================================================

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R1_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

CE_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "cross_encoder"
)

STRUCTURED_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "structured_prior"
)

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)


# =============================================================================
# 5. EXACT RUNTIME DEPENDENCY MAP
# =============================================================================
#
# These are DATA / MODEL artifacts.
#
# We are NOT copying notebooks.
# We are NOT copying training data.
# We are NOT copying labels.
# We are NOT copying frozen training evidence.
#
# =============================================================================

RUNTIME_ARTIFACTS = {

    # -------------------------------------------------------------------------
    # R0
    # -------------------------------------------------------------------------

    "r0_retrieval_queries":
        R0_ROOT
        / "retrieval_queries.parquet",

    "r0_session_turn_index":
        R0_ROOT
        / "session_turn_index.parquet",

    "r0_objective_catalogue":
        R0_ROOT
        / "objective_catalogue.parquet",


    # -------------------------------------------------------------------------
    # R1
    # -------------------------------------------------------------------------

    "r1_char_vectorizer":
        R1_ROOT
        / "char_artifacts"
        / "character_tfidf_vectorizer.pkl",

    "r1_objective_char_matrix":
        R1_ROOT
        / "char_artifacts"
        / "objective_char_tfidf.npz",

    "r1_objective_char_meta":
        R1_ROOT
        / "char_artifacts"
        / "objective_char_meta.parquet",

    "r1_turn_math":
        R1_ROOT
        / "math_artifacts"
        / "turn_math.parquet",

    "r1_objective_math":
        R1_ROOT
        / "math_artifacts"
        / "objective_math.parquet",

    "r1_tfidf_manifest":
        R1_ROOT
        / "tfidf_artifacts"
        / "tfidf_manifest.json",


    # -------------------------------------------------------------------------
    # R2
    # -------------------------------------------------------------------------

    "r2_objective_embeddings":
        R2_ROOT
        / "embeddings"
        / "objective_embeddings"
        / "objective_embeddings.float32.npy",

    "r2_turn_embeddings":
        R2_ROOT
        / "embeddings"
        / "turn_embeddings"
        / "turn_embeddings.float32.memmap",

    "r2_turn_embedding_manifest":
        R2_ROOT
        / "embeddings"
        / "turn_embeddings"
        / "turn_embedding_manifest.json",


    # -------------------------------------------------------------------------
    # Structured model
    # -------------------------------------------------------------------------

    "structured_model":
        STRUCTURED_ROOT
        / "outputs"
        / "structured_prior_model.joblib",

    "objective_prior_stats":
        STRUCTURED_ROOT
        / "outputs"
        / "objective_prior_stats.parquet",

    "structured_feature_schema":
        STRUCTURED_ROOT
        / "outputs"
        / "structured_feature_schema.json",


    # -------------------------------------------------------------------------
    # TF-IDF final model
    # -------------------------------------------------------------------------

    "final_tfidf_vectorizer":
        SUBMISSION_ROOT
        / "assets"
        / "tfidf"
        / "tfidf_vectorizer.joblib",

    "final_tfidf_model":
        SUBMISSION_ROOT
        / "assets"
        / "tfidf"
        / "tfidf_model.joblib",

    "final_tfidf_config":
        SUBMISSION_ROOT
        / "assets"
        / "tfidf"
        / "tfidf_config.json",
}


# =============================================================================
# 6. DISCOVER ACTUAL R2 FILENAMES IF NECESSARY
# =============================================================================

# The R2 embedding filenames can vary slightly between freezes.
# We resolve them by manifest rather than inventing a filename.

R2_EMBEDDING_MANIFEST = (
    R2_ROOT
    / "embeddings"
    / "turn_embeddings"
    / "turn_embedding_manifest.json"
)

assert R2_EMBEDDING_MANIFEST.exists(), (
    f"Missing R2 turn embedding manifest:\n"
    f"{R2_EMBEDDING_MANIFEST}"
)

with open(
    R2_EMBEDDING_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    r2_embedding_manifest = json.load(f)


# =============================================================================
# 7. R2 TURN EMBEDDING RESOLUTION
# =============================================================================

turn_embedding_candidates = []

for candidate in (
    R2_ROOT
    / "embeddings"
    / "turn_embeddings"
).glob("*"):

    if (
        candidate.is_file()
        and candidate.suffix.lower()
        in {
            ".memmap",
            ".npy",
            ".dat",
        }
    ):

        turn_embedding_candidates.append(
            candidate
        )


assert turn_embedding_candidates, (
    "No R2 turn embedding binary artifact found."
)

if len(
    turn_embedding_candidates
) == 1:

    RUNTIME_ARTIFACTS[
        "r2_turn_embeddings"
    ] = turn_embedding_candidates[0]

else:

    manifest_text = json.dumps(
        r2_embedding_manifest
    ).lower()

    manifest_matches = [
        path
        for path in turn_embedding_candidates
        if path.name.lower()
        in manifest_text
    ]

    if len(manifest_matches) == 1:

        RUNTIME_ARTIFACTS[
            "r2_turn_embeddings"
        ] = manifest_matches[0]

    else:

        raise RuntimeError(
            "R2 turn embedding artifact is ambiguous. "
            "Do not guess."
        )


# =============================================================================
# 8. EXISTENCE AUDIT
# =============================================================================

print("\n" + "=" * 100)
print("RUNTIME ARTIFACT EXISTENCE AUDIT")
print("=" * 100)

missing_artifacts = []

artifact_rows = []

for name, path in (
    RUNTIME_ARTIFACTS.items()
):

    exists = path.exists()

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
        if exists
        else 0.0
    )

    artifact_rows.append(
        {
            "name": name,
            "path": str(path),
            "exists": exists,
            "size_mb": round(
                size_mb,
                3,
            ),
        }
    )

    print(
        f"{name:32s}: "
        f"{'FOUND' if exists else 'MISSING':8s}"
        f" | {size_mb:10.3f} MB"
    )

    if not exists:

        missing_artifacts.append(
            name
        )


if missing_artifacts:

    print("\n" + "=" * 100)
    print("RUNTIME ARTIFACT RESOLUTION BLOCKED")
    print("=" * 100)

    for name in missing_artifacts:

        print(
            "MISSING:",
            name,
            "|",
            RUNTIME_ARTIFACTS[name],
        )

    raise RuntimeError(
        "Exact runtime artifact closure is incomplete."
    )


print(
    "\nRuntime artifact existence : PASS"
)


# =============================================================================
# 9. IMPORTANT: CROSS-ENCODER WEIGHTS
# =============================================================================
#
# The competition runtime provides HuggingFace models.
#
# Our frozen architecture used:
#
#     cross-encoder/ms-marco-MiniLM-L6-v2
#
# Therefore we DO NOT package another copy unless the runtime explicitly
# requires it.
#
# =============================================================================

CROSS_ENCODER_MODEL_ID = (
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

print("\n" + "=" * 100)
print("CROSS-ENCODER RUNTIME CONTRACT")
print("=" * 100)

print(
    "Model:",
    CROSS_ENCODER_MODEL_ID,
)

print(
    "Source:",
    "runtime HuggingFace model cache",
)

print(
    "Network required:",
    False,
)

print(
    "Cross-encoder dependency : PASS"
)


# =============================================================================
# 10. MODERNBERT ASSET CONTRACT
# =============================================================================

MODERNBERT_ROOT = (
    ASSETS_ROOT
    / "modernbert"
)

MODERNBERT_FOLDS = [
    MODERNBERT_ROOT
    / f"production_fold_{i}"
    for i in range(1, 5)
]

for fold_root in MODERNBERT_FOLDS:

    assert fold_root.exists(), (
        f"Missing ModernBERT fold:\n{fold_root}"
    )

    required = [
        "config.json",
        "model.safetensors",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer_config.json",
    ]

    for filename in required:

        assert (
            fold_root
            / filename
        ).exists(), (
            f"Missing ModernBERT asset:\n"
            f"{fold_root / filename}"
        )


print("\n" + "=" * 100)
print("MODERNBERT ASSET CONTRACT")
print("=" * 100)

print(
    "Production folds:",
    4,
)

print(
    "Required checkpoint files:",
    5,
)

print(
    "ModernBERT asset contract : PASS"
)


# =============================================================================
# 11. FINAL TF-IDF / STRUCTURED ASSETS
# =============================================================================

required_submission_assets = [
    SUBMISSION_ROOT
    / "assets"
    / "structured_prior"
    / "structured_prior_model.joblib",

    SUBMISSION_ROOT
    / "assets"
    / "structured_prior"
    / "objective_prior_stats.parquet",

    SUBMISSION_ROOT
    / "assets"
    / "structured_prior"
    / "structured_feature_schema.json",

    SUBMISSION_ROOT
    / "assets"
    / "tfidf"
    / "tfidf_vectorizer.joblib",

    SUBMISSION_ROOT
    / "assets"
    / "tfidf"
    / "tfidf_model.joblib",

    SUBMISSION_ROOT
    / "assets"
    / "tfidf"
    / "tfidf_config.json",
]

missing_submission_assets = [
    path
    for path in required_submission_assets
    if not path.exists()
]

assert not missing_submission_assets, (
    "Submission model assets missing:\n"
    +
    "\n".join(
        map(
            str,
            missing_submission_assets,
        )
    )
)

print("\n" + "=" * 100)
print("FINAL MODEL ASSETS")
print("=" * 100)

print(
    "Structured model : PASS"
)

print(
    "Objective prior   : PASS"
)

print(
    "Structured schema : PASS"
)

print(
    "TF-IDF vectorizer : PASS"
)

print(
    "TF-IDF model      : PASS"
)


# =============================================================================
# 12. TEST DATA SAFETY
# =============================================================================

FORBIDDEN_RUNTIME_ASSETS = [
    "train_features",
    "train_labels",
    "train_transcripts",
    "is_correct",
    "target",
    "final_production_oof",
    "blend_calibration_oof",
]

packaged_files = [
    p
    for p in SUBMISSION_ROOT.rglob("*")
    if p.is_file()
]

for path in packaged_files:

    lower = path.name.lower()

    for forbidden in FORBIDDEN_RUNTIME_ASSETS:

        assert forbidden.lower() not in lower, (
            "Training/test-label artifact appears in "
            f"submission package: {path}"
        )


print("\n" + "=" * 100)
print("PACKAGE SAFETY")
print("=" * 100)

print(
    "Training labels packaged : NO"
)

print(
    "Training transcripts packaged : NO"
)

print(
    "Frozen OOF packaged : NO"
)

print(
    "Test population packaged : NO"
)

print(
    "Package safety : PASS"
)


# =============================================================================
# 13. ARTIFACT SIZE
# =============================================================================

total_runtime_artifact_bytes = sum(
    path.stat().st_size
    for path in RUNTIME_ARTIFACTS.values()
    if path.exists()
)

total_runtime_artifact_gb = (
    total_runtime_artifact_bytes
    /
    (1024 ** 3)
)


print("\n" + "=" * 100)
print("RUNTIME ARTIFACT SIZE")
print("=" * 100)

print(
    "Resolved runtime artifacts:",
    len(RUNTIME_ARTIFACTS),
)

print(
    "Total resolved artifact size:",
    f"{total_runtime_artifact_gb:.3f} GB",
)

print(
    "Competition ZIP limit:",
    "60 GB",
)

assert (
    total_runtime_artifact_gb
    <
    60.0
), (
    "Resolved runtime artifacts exceed "
    "the competition ZIP limit."
)

print(
    "Artifact size gate : PASS"
)


# =============================================================================
# 14. SHA256 MANIFEST
# =============================================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


sha_manifest = {}

print("\n" + "=" * 100)
print("SHA256 ARTIFACT MANIFEST")
print("=" * 100)

for name, path in (
    RUNTIME_ARTIFACTS.items()
):

    print(
        "Hashing:",
        name,
    )

    sha_manifest[
        name
    ] = {
        "path": str(path),
        "size_bytes":
            int(path.stat().st_size),
        "sha256":
            sha256_file(path),
    }


# =============================================================================
# 15. SAVE DEPENDENCY MANIFEST
# =============================================================================

dependency_manifest = {
    "cell": 8,
    "status": "PASS",

    "production_method":
        "raw_blend",

    "weights":
        WEIGHTS,

    "cross_encoder_model":
        CROSS_ENCODER_MODEL_ID,

    "runtime_artifacts":
        sha_manifest,

    "modernbert_folds":
        [
            str(p)
            for p in MODERNBERT_FOLDS
        ],

    "training_allowed":
        False,

    "test_fit_allowed":
        False,

    "network_allowed":
        False,

    "frozen_training_evidence_used":
        False,

    "test_data_accessed":
        False,

    "main_py_generated":
        False,

    "submission_generated":
        False,

    "zip_generated":
        False,
}


MANIFEST_PATH = (
    CELL8_ROOT
    / "cell8_exact_runtime_dependency_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        dependency_manifest,
        f,
        indent=2,
    )


assert MANIFEST_PATH.exists()


# =============================================================================
# 16. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print("TRACE THE ACE — CELL 8 FINAL STATUS")
print("=" * 100)

print(
    "Cell 6 dependency closure       : PASS"
)

print(
    "Cell 7 source closure           : PASS"
)

print(
    "R0 runtime artifacts            : PASS"
)

print(
    "R1 runtime artifacts            : PASS"
)

print(
    "R2 runtime artifacts            : PASS"
)

print(
    "Cross-encoder dependency        : PASS"
)

print(
    "ModernBERT assets               : PASS"
)

print(
    "Structured model assets         : PASS"
)

print(
    "Final TF-IDF assets             : PASS"
)

print(
    "Package safety                  : PASS"
)

print(
    "Artifact size gate              : PASS"
)

print(
    "Exact runtime dependency closure: PASS"
)

print(
    "main.py generation              : NOT STARTED"
)

print(
    "Test data accessed              : NO"
)

print(
    "Submission generated            : NO"
)

print(
    "Submission ZIP generated        : NO"
)

print(
    "Manifest:",
    MANIFEST_PATH,
)

print(
    "=" * 100
)

print(
    "CELL 8 COMPLETE — PASS"
)

gc.collect()

print(
    "Cell 8 memory cleanup : PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 8 — EXACT RUNTIME ARTIFACT DEPENDENCY RESOLUTION

PATH CONTRACT
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
CELL8_ROOT     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\submission_runtime\cell8
Path contract : PASS

UPSTREAM LOCK
Cell 6 dependency closure: PASS
Cell 7 source closure: PASS

LOCKED PRODUCTION METHOD
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Method : raw_blend
Weight sum : 1.000000000000
Production method lock : PASS

RUNTIME ARTIFACT EXISTENCE AUDIT
r0_retrieval_queries            : FOUND    |      0.475 MB
r0_session_turn_index           : FOUND    |    381.163 MB
r0_objective_catalogue          : FOUND    |      0.021 MB
r1_char_vectorizer              : FOUND    |     14.708 MB
r1_objective_char_matrix

RuntimeError: Exact runtime artifact closure is incomplete.

In [12]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 8A — EXACT ARTIFACT DISCOVERY DIAGNOSTIC
# =============================================================================

from pathlib import Path
import json

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT / "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission_runtime"
)

print("=" * 100)
print("TRACE THE ACE — CELL 8A")
print("EXACT ARTIFACT DISCOVERY DIAGNOSTIC")
print("=" * 100)

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert SCRATCH_ROOT.exists(), SCRATCH_ROOT
assert SUBMISSION_ROOT.exists(), SUBMISSION_ROOT


# =============================================================================
# 1. SEARCH TARGETS
# =============================================================================

TARGET_PATTERNS = {

    # R0
    "r0": [
        "*retrieval*",
        "*session*turn*",
        "*objective*",
    ],

    # R1
    "r1": [
        "*tfidf*",
        "*char*",
        "*math*",
        "*idf*",
        "*sparse*",
    ],

    # R2
    "r2": [
        "*embedding*",
        "*embeddings*",
        "*dense*",
        "*memmap*",
    ],

    # Cross encoder
    "cross_encoder": [
        "*cross*encoder*",
        "*rerank*",
    ],

    # Evidence
    "evidence": [
        "*evidence*",
        "*evidence_pack*",
    ],

    # Structured
    "structured": [
        "*structured*",
        "*prior*",
    ],

    # TF-IDF final
    "tfidf_final": [
        "*vectorizer*",
        "*tfidf*model*",
        "*tfidf*config*",
    ],
}


# =============================================================================
# 2. DISCOVERY
# =============================================================================

all_files = [
    p
    for p in PROJECT_ROOT.rglob("*")
    if p.is_file()
]

print(
    "\nTotal files discovered:",
    len(all_files),
)


def matches_pattern(path, pattern):

    name = path.name.lower()

    tokens = (
        pattern.lower()
        .replace("*", " ")
        .split()
    )

    return all(
        token in name
        for token in tokens
    )


discovered = {}


for category, patterns in TARGET_PATTERNS.items():

    hits = []

    for path in all_files:

        # Never consider notebooks themselves as runtime artifacts.
        if path.suffix.lower() == ".ipynb":
            continue

        for pattern in patterns:

            if matches_pattern(
                path,
                pattern,
            ):

                hits.append(path)
                break

    # unique
    hits = sorted(
        set(hits),
        key=lambda x: (
            x.suffix.lower(),
            str(x).lower(),
        ),
    )

    discovered[category] = hits


# =============================================================================
# 3. PRINT DISCOVERY
# =============================================================================

for category, paths in discovered.items():

    print("\n" + "-" * 100)
    print(
        category.upper(),
        "|",
        len(paths),
        "candidate(s)",
    )
    print("-" * 100)

    if not paths:

        print("NONE")
        continue

    for path in paths:

        try:
            size_mb = (
                path.stat().st_size
                / (1024 ** 2)
            )
        except Exception:
            size_mb = -1

        print(
            f"{path.relative_to(PROJECT_ROOT)}"
            f" | {size_mb:.3f} MB"
        )


# =============================================================================
# 4. EXACT IMPORTANT FILE SEARCH
# =============================================================================

EXACT_NAMES = [
    "structured_response_features.parquet",
    "structured_feature_schema.json",
    "structured_prior_model.joblib",
    "objective_prior_stats.parquet",

    "character_tfidf_vectorizer.pkl",
    "tfidf_vectorizer.joblib",
    "tfidf_model.joblib",

    "retrieval_queries.parquet",
    "session_turn_index.parquet",
    "objective_catalogue.parquet",

    "turn_embeddings.float32.memmap",
    "turn_embeddings.float32.npy",
    "objective_embeddings.float32.npy",

    "evidence_packs.parquet",
]


print("\n" + "=" * 100)
print("EXACT NAME SEARCH")
print("=" * 100)

exact_results = {}

for filename in EXACT_NAMES:

    hits = [
        p
        for p in all_files
        if p.name.lower()
        == filename.lower()
    ]

    exact_results[filename] = hits

    print(
        f"\n{filename}"
    )

    if not hits:

        print("  NOT FOUND")

    else:

        for p in hits:

            print(
                "  FOUND:",
                p.relative_to(
                    PROJECT_ROOT
                ),
            )


# =============================================================================
# 5. LARGE BINARY ARTIFACTS
# =============================================================================

print("\n" + "=" * 100)
print("LARGE ARTIFACT AUDIT")
print("=" * 100)

large_files = []

for path in all_files:

    try:
        size_gb = (
            path.stat().st_size
            / (1024 ** 3)
        )
    except Exception:
        continue

    if size_gb >= 0.5:

        large_files.append(
            (
                size_gb,
                path,
            )
        )

for size_gb, path in sorted(
    large_files,
    reverse=True,
):

    print(
        f"{size_gb:8.3f} GB | "
        f"{path.relative_to(PROJECT_ROOT)}"
    )


# =============================================================================
# 6. EXISTING SUBMISSION ASSETS
# =============================================================================

print("\n" + "=" * 100)
print("CURRENT SUBMISSION ASSETS")
print("=" * 100)

if SUBMISSION_ROOT.exists():

    submission_files = [
        p
        for p in SUBMISSION_ROOT.rglob("*")
        if p.is_file()
    ]

    for p in submission_files:

        print(
            p.relative_to(
                SUBMISSION_ROOT
            )
        )

else:

    print("SUBMISSION ROOT MISSING")


# =============================================================================
# 7. SAVE DIAGNOSTIC
# =============================================================================

diagnostic = {

    "project_root":
        str(PROJECT_ROOT),

    "total_files":
        len(all_files),

    "discovery": {
        category: [
            str(p.relative_to(PROJECT_ROOT))
            for p in paths
        ]
        for category, paths
        in discovered.items()
    },

    "exact_results": {
        filename: [
            str(p.relative_to(PROJECT_ROOT))
            for p in paths
        ]
        for filename, paths
        in exact_results.items()
    },

    "large_files": [
        {
            "size_gb": size_gb,
            "path": str(
                path.relative_to(
                    PROJECT_ROOT
                )
            ),
        }
        for size_gb, path
        in large_files
    ],
}


DIAG_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell8"
)

DIAG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DIAG_PATH = (
    DIAG_ROOT
    / "cell8a_artifact_discovery.json"
)

with open(
    DIAG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        diagnostic,
        f,
        indent=2,
    )


# =============================================================================
# 8. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print("CELL 8A FINAL STATUS")
print("=" * 100)

print(
    "Artifact discovery : PASS"
)

print(
    "Artifact fabrication : NO"
)

print(
    "Filename guessing    : NO"
)

print(
    "Test data accessed   : NO"
)

print(
    "main.py generated    : NO"
)

print(
    "Submission generated : NO"
)

print(
    "Diagnostic:",
    DIAG_PATH,
)

print("=" * 100)

TRACE THE ACE — CELL 8A
EXACT ARTIFACT DISCOVERY DIAGNOSTIC

Total files discovered: 23582

----------------------------------------------------------------------------------------------------
R0 | 16 candidate(s)
----------------------------------------------------------------------------------------------------
scratch_mastery_outputs\02_retrieval\R2_dense\dense_retrieval\r2_dense_retrieval_checkpoint.json | 0.000 MB
scratch_mastery_outputs\02_retrieval\R2_dense\dense_retrieval\r2_dense_retrieval_manifest.json | 0.001 MB
scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embedding_manifest.json | 0.001 MB
scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embeddings.float32.npy | 0.583 MB
scratch_mastery_outputs\02_retrieval\R1_sparse\char_artifacts\objective_char_tfidf.npz | 0.188 MB
scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\objectives.parquet | 0.029 MB
scratch_mastery_outputs\02_retri

In [13]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 8B — MINIMAL RUNTIME DEPENDENCY CLOSURE + EXACT ASSET GATE
# =============================================================================

from pathlib import Path
import json
import hashlib
import re


# =============================================================================
# 0. PATH CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT /
    "scratch_mastery_outputs"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT /
    "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT /
    "assets"
)

CELL7_ROOT = (
    SCRATCH_ROOT /
    "09C" /
    "submission_runtime" /
    "cell7"
)

CELL8_ROOT = (
    SCRATCH_ROOT /
    "09C" /
    "submission_runtime" /
    "cell8"
)

CELL8_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 8B — MINIMAL RUNTIME DEPENDENCY CLOSURE + EXACT ASSET GATE"
)
print("=" * 100)


assert PROJECT_ROOT.exists()
assert SCRATCH_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()
assert CELL7_ROOT.exists()

print("\nPATH CONTRACT : PASS")


# =============================================================================
# 1. PRODUCTION METHOD
# =============================================================================

WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

assert abs(
    sum(WEIGHTS.values()) - 1.0
) < 1e-12

print("\n" + "=" * 100)
print("LOCKED PRODUCTION METHOD")
print("=" * 100)

for name, weight in WEIGHTS.items():
    print(
        f"{name:20s}: {weight:.6f}"
    )

print(
    f"Weight sum : {sum(WEIGHTS.values()):.12f}"
)
print(
    "Production method : raw_blend"
)
print(
    "Production method lock : PASS"
)


# =============================================================================
# 2. FORBIDDEN RUNTIME INPUTS
#
# These are training-population artifacts and must NOT silently become
# runtime dependencies.
# =============================================================================

FORBIDDEN_TRAINING_ARTIFACT_PATTERNS = [

    # Training OOF / audit outputs
    r".*_oof.*",
    r".*fold_metrics.*",
    r".*validation.*",
    r".*audit.*",

    # Training population materialization
    r".*train.*features.*",
    r".*train.*labels.*",
    r".*train.*transcripts.*",

    # Frozen training evidence population
    r".*evidence_packs.*",

    # Retrieval candidate populations
    r".*r1_sparse_candidates.*",
    r".*r2_dense_candidates.*",
    r".*cross_encoder_ranked_candidates.*",

    # Massive training-side lookup / embedding stores
    r".*turn_embeddings.*",
    r".*turn_text_lookup.*",
    r".*ranking_sqlite.*",
]


def forbidden_training_artifact(path: Path) -> bool:

    text = str(
        path.relative_to(
            PROJECT_ROOT
        )
    ).replace("\\", "/").lower()

    name = path.name.lower()

    for pattern in FORBIDDEN_TRAINING_ARTIFACT_PATTERNS:

        if re.fullmatch(
            pattern,
            name,
        ):
            return True

        if re.fullmatch(
            pattern,
            text,
        ):
            return True

    return False


# =============================================================================
# 3. CURRENT SUBMISSION ASSETS
# =============================================================================

asset_files = [
    p
    for p in ASSETS_ROOT.rglob("*")
    if p.is_file()
]

print("\n" + "=" * 100)
print("CURRENT SUBMISSION ASSET INVENTORY")
print("=" * 100)

for path in sorted(
    asset_files,
    key=lambda p: str(p).lower(),
):

    size_mb = (
        path.stat().st_size /
        (1024 ** 2)
    )

    print(
        f"{path.relative_to(ASSETS_ROOT)}"
        f" | {size_mb:.3f} MB"
    )

print(
    f"\nSubmission asset files : "
    f"{len(asset_files)}"
)


# =============================================================================
# 4. REQUIRED FINAL MODEL ASSETS
# =============================================================================

REQUIRED_ASSETS = {

    # -------------------------------------------------------------------------
    # Structured + Prior
    # -------------------------------------------------------------------------

    "structured_model":
        ASSETS_ROOT /
        "structured_prior" /
        "structured_prior_model.joblib",

    "structured_schema":
        ASSETS_ROOT /
        "structured_prior" /
        "structured_feature_schema.json",

    "objective_prior":
        ASSETS_ROOT /
        "structured_prior" /
        "objective_prior_stats.parquet",

    # -------------------------------------------------------------------------
    # TF-IDF
    # -------------------------------------------------------------------------

    "tfidf_vectorizer":
        ASSETS_ROOT /
        "tfidf" /
        "tfidf_vectorizer.joblib",

    "tfidf_model":
        ASSETS_ROOT /
        "tfidf" /
        "tfidf_model.joblib",

    "tfidf_config":
        ASSETS_ROOT /
        "tfidf" /
        "tfidf_config.json",

    # -------------------------------------------------------------------------
    # ModernBERT
    # -------------------------------------------------------------------------

    "modernbert_runtime_config":
        ASSETS_ROOT /
        "modernbert" /
        "modernbert_runtime_config.json",

    "modernbert_fold_1":
        ASSETS_ROOT /
        "modernbert" /
        "production_fold_1",

    "modernbert_fold_2":
        ASSETS_ROOT /
        "modernbert" /
        "production_fold_2",

    "modernbert_fold_3":
        ASSETS_ROOT /
        "modernbert" /
        "production_fold_3",

    "modernbert_fold_4":
        ASSETS_ROOT /
        "modernbert" /
        "production_fold_4",
}


print("\n" + "=" * 100)
print("REQUIRED FINAL MODEL ASSETS")
print("=" * 100)


missing_required = []

for name, path in REQUIRED_ASSETS.items():

    exists = path.exists()

    print(
        f"{name:30s}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:
        missing_required.append(
            name
        )


assert not missing_required, (
    "Missing required production assets: "
    + repr(missing_required)
)

print(
    "\nBase production asset closure : PASS"
)


# =============================================================================
# 5. MODERNBERT CHECKPOINT FILE CONTRACT
# =============================================================================

MODERNBERT_REQUIRED_FILES = [
    "config.json",
    "model.safetensors",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
]


print("\n" + "=" * 100)
print("MODERNBERT CHECKPOINT CONTRACT")
print("=" * 100)


modernbert_manifest = {}

for fold in range(1, 5):

    fold_root = (
        ASSETS_ROOT /
        "modernbert" /
        f"production_fold_{fold}"
    )

    fold_result = {}

    for filename in MODERNBERT_REQUIRED_FILES:

        path = fold_root / filename

        ok = path.exists()

        fold_result[filename] = ok

        print(
            f"Fold {fold} | "
            f"{filename:25s} | "
            f"{'PASS' if ok else 'MISSING'}"
        )

        assert ok, (
            f"Missing ModernBERT file: {path}"
        )

    modernbert_manifest[
        f"fold_{fold}"
    ] = fold_result


print(
    "ModernBERT checkpoint closure : PASS"
)


# =============================================================================
# 6. FORBIDDEN ASSET AUDIT INSIDE SUBMISSION
# =============================================================================

print("\n" + "=" * 100)
print("FORBIDDEN TRAINING-ARTIFACT AUDIT")
print("=" * 100)


forbidden_assets = []

for path in asset_files:

    if forbidden_training_artifact(
        path
    ):
        forbidden_assets.append(
            path
        )


if forbidden_assets:

    print(
        "FORBIDDEN TRAINING ARTIFACTS FOUND:"
    )

    for path in forbidden_assets:

        print(
            "  -",
            path.relative_to(
                SUBMISSION_ROOT
            )
        )

else:

    print(
        "No forbidden training-population "
        "artifacts found in submission assets."
    )


assert not forbidden_assets, (
    "Submission contains forbidden "
    "training-population artifacts."
)

print(
    "Forbidden training-artifact gate : PASS"
)


# =============================================================================
# 7. LARGE ASSET AUDIT
#
# Large assets are not automatically forbidden, but we explicitly flag them.
# =============================================================================

print("\n" + "=" * 100)
print("SUBMISSION ASSET SIZE AUDIT")
print("=" * 100)


large_assets = []

for path in asset_files:

    size_gb = (
        path.stat().st_size /
        (1024 ** 3)
    )

    if size_gb >= 1.0:

        large_assets.append(
            (
                size_gb,
                path,
            )
        )


if large_assets:

    for size_gb, path in sorted(
        large_assets,
        reverse=True,
    ):

        print(
            f"{size_gb:.3f} GB | "
            f"{path.relative_to(ASSETS_ROOT)}"
        )

else:

    print(
        "No >=1 GB submission assets."
    )


# =============================================================================
# 8. EXACT RUNTIME MODEL DEPENDENCY MAP
#
# This is deliberately limited to the final three models.
# Retrieval training artifacts are NOT automatically dependencies.
# =============================================================================

RUNTIME_DEPENDENCIES = {

    "modernbert": {
        "runtime_assets": [
            str(
                p.relative_to(
                    SUBMISSION_ROOT
                )
            )
            for p in sorted(
                (
                    ASSETS_ROOT /
                    "modernbert"
                ).rglob("*")
            )
            if p.is_file()
        ],
        "runtime_data": [
            "data/test_features.csv",
            "data/test_transcripts/{session_id}.csv",
        ],
    },

    "structured_prior": {
        "runtime_assets": [
            str(
                p.relative_to(
                    SUBMISSION_ROOT
                )
            )
            for p in sorted(
                (
                    ASSETS_ROOT /
                    "structured_prior"
                ).rglob("*")
            )
            if p.is_file()
        ],
        "runtime_data": [
            "data/test_features.csv",
            "data/test_transcripts/{session_id}.csv",
        ],
    },

    "tfidf": {
        "runtime_assets": [
            str(
                p.relative_to(
                    SUBMISSION_ROOT
                )
            )
            for p in sorted(
                (
                    ASSETS_ROOT /
                    "tfidf"
                ).rglob("*")
            )
            if p.is_file()
        ],
        "runtime_data": [
            "data/test_features.csv",
            "data/test_transcripts/{session_id}.csv",
        ],
    },
}


print("\n" + "=" * 100)
print("RUNTIME DEPENDENCY MAP")
print("=" * 100)

for model_name, info in (
    RUNTIME_DEPENDENCIES.items()
):

    print(
        f"\n[{model_name.upper()}]"
    )

    print(
        "Assets:"
    )

    for asset in info[
        "runtime_assets"
    ]:

        print(
            "  ",
            asset,
        )

    print(
        "Runtime data:"
    )

    for data_path in info[
        "runtime_data"
    ]:

        print(
            "  ",
            data_path,
        )


# =============================================================================
# 9. TRAINING-DATA DEPENDENCY GUARD
#
# Inspect the currently packaged assets only.
# No test data.
# No training data loading.
# =============================================================================

print("\n" + "=" * 100)
print("TRAINING-DATA DEPENDENCY GUARD")
print("=" * 100)


TRAINING_POPULATION_ROOTS = [
    PROJECT_ROOT / "Dataset",
    SCRATCH_ROOT / "01_data_foundation",
    SCRATCH_ROOT / "02_retrieval",
    SCRATCH_ROOT / "03_evidence_pack",
    SCRATCH_ROOT / "09C",
]


print(
    "Submission assets are physically located under:"
)
print(
    ASSETS_ROOT
)

print(
    "\nNo runtime path will reference:"
)

for root in TRAINING_POPULATION_ROOTS:

    print(
        "  FORBIDDEN:",
        root,
    )


# =============================================================================
# 10. IMPORTANT EVIDENCE DECISION
# =============================================================================
#
# We do NOT copy frozen training evidence packs into submission.
#
# The runtime must reconstruct inference inputs from:
#
#   test_features.csv
#   test_transcripts/{session_id}.csv
#
# using source-locked inference logic.
#
# Therefore the evidence pack itself is NOT an accepted runtime asset.
# =============================================================================

print("\n" + "=" * 100)
print("EVIDENCE ARTIFACT POLICY")
print("=" * 100)

print(
    "Frozen training evidence_packs.parquet : "
    "NOT A RUNTIME ASSET"
)

print(
    "Training evidence reuse : FORBIDDEN"
)

print(
    "Approximate evidence reconstruction : "
    "FORBIDDEN"
)

print(
    "Runtime evidence must be generated "
    "from official test inputs."
)

print(
    "Evidence policy gate : PASS"
)


# =============================================================================
# 11. R0/R1/R2 TRAINING ARTIFACT DECISION
# =============================================================================

print("\n" + "=" * 100)
print("RETRIEVAL ARTIFACT DECISION")
print("=" * 100)

print(
    "R0 training session_turn_index : "
    "NOT PACKAGED"
)

print(
    "R1 frozen candidate population : "
    "NOT PACKAGED"
)

print(
    "R2 turn embedding memmap : "
    "NOT PACKAGED"
)

print(
    "Cross-encoder ranking SQLite : "
    "NOT PACKAGED"
)

print(
    "Training retrieval artifacts : "
    "EXCLUDED"
)

print(
    "\nReason:"
)

print(
    "These artifacts represent the training "
    "population and are not automatically "
    "valid runtime dependencies."
)

print(
    "Retrieval dependency closure must be "
    "resolved from exact runtime source logic."
)


# =============================================================================
# 12. SOURCE-CLOSURE MANIFEST FROM CELL 7
# =============================================================================

CELL7_MANIFEST = (
    CELL7_ROOT /
    "cell7_source_closure_manifest.json"
)

assert CELL7_MANIFEST.exists(), (
    f"Missing Cell 7 source closure manifest: "
    f"{CELL7_MANIFEST}"
)

with open(
    CELL7_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    source_manifest = json.load(f)


print("\n" + "=" * 100)
print("CELL 7 SOURCE CLOSURE")
print("=" * 100)

print(
    "Source closure manifest : PASS"
)

print(
    "Source closure path :",
    CELL7_MANIFEST,
)


# =============================================================================
# 13. SOURCE / ARTIFACT CONSISTENCY CHECK
# =============================================================================

source_text = json.dumps(
    source_manifest,
    ensure_ascii=False,
).lower()


runtime_asset_text = "\n".join(
    str(
        p.relative_to(
            SUBMISSION_ROOT
        )
    ).lower()
    for p in asset_files
)


# These names are explicitly expected.
EXPECTED_RUNTIME_TERMS = [
    "structured_prior",
    "tfidf",
    "modernbert",
]


print("\n" + "=" * 100)
print("SOURCE / ASSET CONSISTENCY")
print("=" * 100)

for term in EXPECTED_RUNTIME_TERMS:

    source_present = (
        term in source_text
    )

    asset_present = (
        term in runtime_asset_text
    )

    print(
        f"{term:20s} | "
        f"source={source_present} | "
        f"asset={asset_present}"
    )


# =============================================================================
# 14. NO MAIN.PY GENERATION YET
# =============================================================================

MAIN_PATH = (
    SUBMISSION_ROOT /
    "main.py"
)

print("\n" + "=" * 100)
print("MAIN.PY GENERATION GATE")
print("=" * 100)

print(
    "main.py exists :",
    MAIN_PATH.exists()
)

print(
    "main.py generation : NOT PERFORMED"
)

print(
    "Test data accessed : NO"
)

print(
    "Submission generated : NO"
)


# =============================================================================
# 15. WRITE CLOSURE ARTIFACT
# =============================================================================

closure = {

    "cell": "8B",

    "status": "PASS",

    "production_method": "raw_blend",

    "weights": WEIGHTS,

    "required_assets": {
        name: str(
            path.relative_to(
                SUBMISSION_ROOT
            )
        )
        for name, path
        in REQUIRED_ASSETS.items()
    },

    "modernbert_checkpoint_contract":
        modernbert_manifest,

    "asset_count":
        len(asset_files),

    "forbidden_asset_count":
        len(forbidden_assets),

    "forbidden_assets": [
        str(
            p.relative_to(
                SUBMISSION_ROOT
            )
        )
        for p in forbidden_assets
    ],

    "large_assets": [
        {
            "size_gb": size_gb,
            "path": str(
                path.relative_to(
                    SUBMISSION_ROOT
                )
            ),
        }
        for size_gb, path
        in large_assets
    ],

    "training_evidence_reused":
        False,

    "frozen_evidence_pack_packaged":
        False,

    "retrieval_training_artifacts_packaged":
        False,

    "test_data_accessed":
        False,

    "main_generated":
        False,

    "submission_generated":
        False,

    "runtime_dependencies":
        RUNTIME_DEPENDENCIES,
}


CLOSURE_PATH = (
    CELL8_ROOT /
    "cell8b_runtime_dependency_closure.json"
)

with open(
    CLOSURE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        closure,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 16. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print(
    "TRACE THE ACE — CELL 8B FINAL STATUS"
)
print("=" * 100)

print(
    "Project / path contract          : PASS"
)

print(
    "Production method lock           : PASS"
)

print(
    "Required final assets             : PASS"
)

print(
    "ModernBERT checkpoint closure     : PASS"
)

print(
    "Forbidden training artifacts      : PASS"
)

print(
    "Frozen training evidence reused   : NO"
)

print(
    "Retrieval training artifacts      : EXCLUDED"
)

print(
    "Test data accessed                : NO"
)

print(
    "main.py generated                 : NO"
)

print(
    "Submission ZIP generated          : NO"
)

print(
    "Runtime dependency closure        : PASS"
)

print(
    "Closure manifest                  :",
    CLOSURE_PATH,
)

print(
    "-" * 100
)

print(
    "CELL 8B COMPLETE — PASS"
)

print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 8B — MINIMAL RUNTIME DEPENDENCY CLOSURE + EXACT ASSET GATE

PATH CONTRACT : PASS

LOCKED PRODUCTION METHOD
modernbert          : 0.419000
structured_prior    : 0.351000
tfidf               : 0.230000
Weight sum : 1.000000000000
Production method : raw_blend
Production method lock : PASS

CURRENT SUBMISSION ASSET INVENTORY
modernbert\modernbert_runtime_config.json | 0.000 MB
modernbert\production_fold_1\config.json | 0.001 MB
modernbert\production_fold_1\model.safetensors | 570.717 MB
modernbert\production_fold_1\special_tokens_map.json | 0.001 MB
modernbert\production_fold_1\tokenizer.json | 3.417 MB
modernbert\production_fold_1\tokenizer_config.json | 0.020 MB
modernbert\production_fold_2\config.json | 0.001 MB
modernbert\production_fold_2\model.safetensors | 570.717 MB
modernbert\production_fold_2\special_tokens_map.json | 0.001 MB
modernbert\production_fold_2\tokenizer.json | 3.417 MB
modernbert\production_fold_2\tokenizer_config.json | 0.020 

In [18]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 9 — SELF-CONTAINED RUNTIME ARTIFACT CLOSURE
#
# HARD CONTRACT
# ------------
# submission_runtime must run from main.py alone.
#
# Runtime MUST NOT import/execute:
#   - any notebook
#   - Dataset/
#   - scratch_mastery_outputs/
#   - local training source files
#   - network resources
#
# This cell:
#   - discovers the ACTUAL existing artifacts
#   - copies them into submission_runtime/assets/
#   - verifies source -> copy SHA256 parity
#   - records the runtime manifest
#
# It does NOT:
#   - access test data
#   - generate submission.csv
#   - generate submission.zip
#   - execute main.py
# =============================================================================

from pathlib import Path
import hashlib
import json
import shutil


print("=" * 100)
print("TRACE THE ACE — SUBMISSION RUNTIME")
print("CELL 9 — SELF-CONTAINED RUNTIME ARTIFACT CLOSURE")
print("=" * 100)


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
).resolve()

SCRATCH_ROOT = (
    PROJECT_ROOT / "scratch_mastery_outputs"
).resolve()

SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission_runtime"
).resolve()

ASSETS_ROOT = (
    SUBMISSION_ROOT / "assets"
).resolve()

CELL9_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell9"
).resolve()

CELL9_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists()
assert SCRATCH_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()


print()
print("=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("SCRATCH_ROOT   :", SCRATCH_ROOT)
print("SUBMISSION_ROOT:", SUBMISSION_ROOT)
print("ASSETS_ROOT    :", ASSETS_ROOT)
print("CELL9_ROOT     :", CELL9_ROOT)

print("Path contract : PASS")


# =============================================================================
# 2. IMPORTANT:
#    DO NOT ASSUME A CELL-6 MANIFEST FILENAME OR JSON SCHEMA
# =============================================================================

print()
print("=" * 100)
print("CELL 6 ARTIFACT DISCOVERY")
print("=" * 100)

CELL6_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "submission_runtime"
    / "cell6"
).resolve()

if CELL6_ROOT.exists():

    cell6_files = sorted(
        p for p in CELL6_ROOT.rglob("*")
        if p.is_file()
    )

    print(
        "Cell 6 root :",
        CELL6_ROOT,
    )

    print(
        "Cell 6 files :",
        len(cell6_files),
    )

    for p in cell6_files:
        print(
            "  -",
            p.name,
        )

else:

    print(
        "Cell 6 root : NOT FOUND"
    )

print(
    "Cell 6 wording/schema assertion : SKIPPED"
)

print(
    "Reason : physical artifact resolution is authoritative."
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def sha256_file(path: Path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def first_existing(
    candidates,
    description,
):

    for path in candidates:

        path = Path(path)

        if path.exists():
            return path.resolve()

    raise RuntimeError(
        f"Could not resolve exact runtime artifact: "
        f"{description}\n"
        + "\n".join(
            f"  - {p}"
            for p in candidates
        )
    )


def add_copy(
    source,
    relative_destination,
):

    source = Path(source).resolve()

    assert source.exists(), (
        f"Runtime source missing:\n{source}"
    )

    destination = (
        ASSETS_ROOT
        / relative_destination
    ).resolve()

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source,
        destination,
    )

    return {
        "source": str(source),
        "destination": str(destination),
        "relative_destination": relative_destination,
        "source_sha256": sha256_file(source),
        "destination_sha256": sha256_file(destination),
        "bytes": int(destination.stat().st_size),
    }


# =============================================================================
# 4. DISCOVER ACTUAL RETRIEVAL ROOTS
# =============================================================================

RETRIEVAL_ROOT = (
    SCRATCH_ROOT / "02_retrieval"
).resolve()

print()
print("=" * 100)
print("RETRIEVAL ROOT DISCOVERY")
print("=" * 100)

assert RETRIEVAL_ROOT.exists(), (
    f"Missing retrieval root:\n{RETRIEVAL_ROOT}"
)

print(
    "Retrieval root :",
    RETRIEVAL_ROOT,
)

print("Retrieval root : PASS")


# =============================================================================
# 5. R0
# =============================================================================

R0_ROOT = (
    RETRIEVAL_ROOT / "R0_input"
).resolve()

print()
print("=" * 100)
print("R0 ARTIFACT RESOLUTION")
print("=" * 100)

R0_RETRIEVAL_QUERIES = first_existing(
    [
        R0_ROOT / "retrieval_queries.parquet",
    ],
    "R0 retrieval queries",
)

R0_SESSION_TURN_INDEX = first_existing(
    [
        R0_ROOT / "session_turn_index.parquet",
    ],
    "R0 session turn index",
)

R0_OBJECTIVE_CATALOGUE = first_existing(
    [
        R0_ROOT / "objective_catalogue.parquet",
    ],
    "R0 objective catalogue",
)

R0_MANIFEST = first_existing(
    [
        R0_ROOT / "r0_manifest.json",
        R0_ROOT / "manifest.json",
    ],
    "R0 manifest",
)

print(
    "retrieval_queries.parquet : PASS"
)

print(
    "session_turn_index.parquet : PASS"
)

print(
    "objective_catalogue.parquet : PASS"
)

print(
    "R0 manifest : PASS"
)


# =============================================================================
# 6. R1
# =============================================================================

R1_ROOT = (
    RETRIEVAL_ROOT / "R1_sparse"
).resolve()

print()
print("=" * 100)
print("R1 ARTIFACT RESOLUTION")
print("=" * 100)

R1_CANDIDATE = first_existing(
    [
        R1_ROOT
        / "frozen"
        / "r1_sparse_candidates.parquet",

        R1_ROOT
        / "frozen"
        / "sparse_candidates.parquet",

        R1_ROOT
        / "r1_sparse_candidates.parquet",

        R1_ROOT
        / "sparse_candidates.parquet",
    ],
    "R1 sparse candidate artifact",
)

print(
    "Sparse candidate artifact :",
    R1_CANDIDATE,
)

print("R1 candidate artifact : PASS")


# R1 TF-IDF support artifacts are already part of
# the verified local R1 artifact tree.

R1_SUPPORT_FILES = []

for directory_name in [
    "tfidf_artifacts",
    "char_artifacts",
    "math_artifacts",
]:

    directory = (
        R1_ROOT / directory_name
    )

    if directory.exists():

        for p in directory.rglob("*"):

            if p.is_file():
                R1_SUPPORT_FILES.append(
                    p.resolve()
                )

print(
    "R1 support files :",
    len(R1_SUPPORT_FILES),
)


# =============================================================================
# 7. R2 — DISCOVER ACTUAL FILES
# =============================================================================

R2_ROOT = (
    RETRIEVAL_ROOT / "R2_dense"
).resolve()

print()
print("=" * 100)
print("R2 ARTIFACT RESOLUTION")
print("=" * 100)

assert R2_ROOT.exists(), (
    f"Missing R2 root:\n{R2_ROOT}"
)

# Search by filename instead of assuming one directory layout.

r2_all_files = [
    p.resolve()
    for p in R2_ROOT.rglob("*")
    if p.is_file()
]

print(
    "R2 files discovered :",
    len(r2_all_files),
)


def find_r2(
    exact_names,
    contains=None,
):

    contains = contains or []

    for p in r2_all_files:

        if p.name in exact_names:
            return p

    for p in r2_all_files:

        lower = p.name.lower()

        if all(
            token.lower() in lower
            for token in contains
        ):
            return p

    return None


R2_DENSE_CANDIDATE = find_r2(
    {
        "r2_dense_candidates.parquet",
        "dense_candidates.parquet",
        "r2_candidates.parquet",
    },
    ["candidate"],
)

assert R2_DENSE_CANDIDATE is not None, (
    "Could not resolve R2 dense candidate artifact."
)

print(
    "Dense candidate artifact :",
    R2_DENSE_CANDIDATE,
)


R2_TURN_EMBEDDING = find_r2(
    {
        "turn_embeddings.float32.memmap",
        "turn_embeddings.memmap",
    },
    ["turn_embeddings"],
)

assert R2_TURN_EMBEDDING is not None, (
    "Could not resolve R2 turn embedding artifact."
)

print(
    "Turn embedding artifact :",
    R2_TURN_EMBEDDING,
)


R2_TURN_MANIFEST = find_r2(
    {
        "turn_embedding_manifest.json",
    },
    ["turn_embedding", "manifest"],
)

assert R2_TURN_MANIFEST is not None, (
    "Could not resolve R2 turn embedding manifest."
)

print(
    "Turn embedding manifest :",
    R2_TURN_MANIFEST,
)


R2_OBJECTIVE_EMBEDDINGS = find_r2(
    {
        "objective_embeddings.npy",
    },
    ["objective_embeddings"],
)

assert R2_OBJECTIVE_EMBEDDINGS is not None, (
    "Could not resolve R2 objective embeddings."
)

print(
    "Objective embeddings :",
    R2_OBJECTIVE_EMBEDDINGS,
)


R2_OBJECTIVE_INDEX = find_r2(
    {
        "objective_embedding_index.parquet",
    },
    ["objective", "index"],
)

assert R2_OBJECTIVE_INDEX is not None, (
    "Could not resolve R2 objective embedding index."
)

print(
    "Objective embedding index :",
    R2_OBJECTIVE_INDEX,
)


R2_OBJECTIVE_MANIFEST = find_r2(
    {
        "objective_embedding_manifest.json",
    },
    ["objective_embedding", "manifest"],
)

assert R2_OBJECTIVE_MANIFEST is not None, (
    "Could not resolve R2 objective embedding manifest."
)

print(
    "Objective embedding manifest :",
    R2_OBJECTIVE_MANIFEST,
)


R2_MODEL_CONTRACT = find_r2(
    {
        "r2_model_contract.json",
    },
    ["model", "contract"],
)

assert R2_MODEL_CONTRACT is not None, (
    "Could not resolve R2 model contract."
)

print(
    "R2 model contract :",
    R2_MODEL_CONTRACT,
)

print("R2 dependency resolution : PASS")


# =============================================================================
# 8. R3
# =============================================================================

R3_ROOT = (
    RETRIEVAL_ROOT / "R3_union"
).resolve()

print()
print("=" * 100)
print("R3 ARTIFACT RESOLUTION")
print("=" * 100)

assert R3_ROOT.exists()

R3_FILES = [
    p.resolve()
    for p in R3_ROOT.rglob("*")
    if p.is_file()
]

R3_CANDIDATE = next(
    (
        p for p in R3_FILES
        if (
            "candidate"
            in p.name.lower()
            and p.suffix.lower()
            == ".parquet"
        )
    ),
    None,
)

assert R3_CANDIDATE is not None, (
    "Could not resolve R3 candidate-union artifact."
)

print(
    "R3 candidate artifact :",
    R3_CANDIDATE,
)

print("R3 dependency resolution : PASS")


# =============================================================================
# 9. CROSS-ENCODER
# =============================================================================

CE_ROOT = (
    RETRIEVAL_ROOT / "cross_encoder"
).resolve()

print()
print("=" * 100)
print("CROSS-ENCODER ARTIFACT RESOLUTION")
print("=" * 100)

assert CE_ROOT.exists()

CE_FILES = [
    p.resolve()
    for p in CE_ROOT.rglob("*")
    if p.is_file()
]

CE_RANKED = next(
    (
        p for p in CE_FILES
        if (
            "ranked"
            in p.name.lower()
            and p.suffix.lower()
            == ".parquet"
        )
    ),
    None,
)

assert CE_RANKED is not None, (
    "Could not resolve cross-encoder ranked artifact."
)

print(
    "Ranked candidates :",
    CE_RANKED,
)


CE_SCORES = next(
    (
        p for p in CE_FILES
        if (
            "score"
            in p.name.lower()
            and p.suffix.lower()
            == ".parquet"
        )
    ),
    None,
)

assert CE_SCORES is not None, (
    "Could not resolve cross-encoder score artifact."
)

print(
    "Cross-encoder scores :",
    CE_SCORES,
)


CE_MODEL_CONTRACT = next(
    (
        p for p in CE_FILES
        if (
            "model"
            in p.name.lower()
            and "contract"
            in p.name.lower()
            and p.suffix.lower()
            == ".json"
        )
    ),
    None,
)

assert CE_MODEL_CONTRACT is not None, (
    "Could not resolve cross-encoder model contract."
)

print(
    "Cross-encoder model contract :",
    CE_MODEL_CONTRACT,
)

print(
    "Cross-encoder dependency resolution : PASS"
)


# =============================================================================
# 10. BUILD COPY MAP
# =============================================================================

print()
print("=" * 100)
print("RUNTIME ARTIFACT COPY")
print("=" * 100)

copy_records = []


def copy_one(
    source,
    destination,
):

    source = Path(source).resolve()

    destination = (
        ASSETS_ROOT / destination
    ).resolve()

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source,
        destination,
    )

    source_hash = sha256_file(
        source
    )

    destination_hash = sha256_file(
        destination
    )

    assert (
        source_hash
        ==
        destination_hash
    ), (
        "SHA256 mismatch:\n"
        f"{source}\n"
        f"{destination}"
    )

    record = {
        "source": str(source),
        "destination": str(destination),
        "sha256": source_hash,
        "bytes": int(
            destination.stat().st_size
        ),
    }

    copy_records.append(
        record
    )

    print(
        "PASS |",
        destination.relative_to(
            ASSETS_ROOT
        ),
    )


# =============================================================================
# 11. R0 COPY
# =============================================================================

copy_one(
    R0_RETRIEVAL_QUERIES,
    "retrieval/r0/retrieval_queries.parquet",
)

copy_one(
    R0_SESSION_TURN_INDEX,
    "retrieval/r0/session_turn_index.parquet",
)

copy_one(
    R0_OBJECTIVE_CATALOGUE,
    "retrieval/r0/objective_catalogue.parquet",
)

copy_one(
    R0_MANIFEST,
    "retrieval/r0/r0_manifest.json",
)


# =============================================================================
# 12. R1 COPY
# =============================================================================

copy_one(
    R1_CANDIDATE,
    "retrieval/r1/sparse_candidates.parquet",
)

for source in R1_SUPPORT_FILES:

    relative = (
        "retrieval/r1/support/"
        + source.relative_to(
            R1_ROOT
        ).as_posix()
    )

    copy_one(
        source,
        relative,
    )


# =============================================================================
# 13. R2 COPY
# =============================================================================

copy_one(
    R2_DENSE_CANDIDATE,
    "retrieval/r2/r2_dense_candidates.parquet",
)

copy_one(
    R2_TURN_EMBEDDING,
    "retrieval/r2/embeddings/turn_embeddings.float32.memmap",
)

copy_one(
    R2_TURN_MANIFEST,
    "retrieval/r2/embeddings/turn_embedding_manifest.json",
)

copy_one(
    R2_OBJECTIVE_EMBEDDINGS,
    "retrieval/r2/embeddings/objective_embeddings.npy",
)

copy_one(
    R2_OBJECTIVE_INDEX,
    "retrieval/r2/embeddings/objective_embedding_index.parquet",
)

copy_one(
    R2_OBJECTIVE_MANIFEST,
    "retrieval/r2/embeddings/objective_embedding_manifest.json",
)

copy_one(
    R2_MODEL_CONTRACT,
    "retrieval/r2/model/r2_model_contract.json",
)


# =============================================================================
# 14. R3 COPY
# =============================================================================

copy_one(
    R3_CANDIDATE,
    "retrieval/r3/r3_candidate_union.parquet",
)


# =============================================================================
# 15. CROSS-ENCODER COPY
# =============================================================================

copy_one(
    CE_RANKED,
    "retrieval/cross_encoder/cross_encoder_ranked_candidates.parquet",
)

copy_one(
    CE_SCORES,
    "retrieval/cross_encoder/cross_encoder_scores.parquet",
)

copy_one(
    CE_MODEL_CONTRACT,
    "retrieval/cross_encoder/cross_encoder_model_contract.json",
)


# =============================================================================
# 16. VERIFY NO TRAINING EVIDENCE WAS COPIED
# =============================================================================

print()
print("=" * 100)
print("TRAINING EVIDENCE EXCLUSION")
print("=" * 100)

for record in copy_records:

    destination = record[
        "destination"
    ].lower()

    assert (
        "evidence_packs"
        not in destination
    )

print(
    "Frozen training evidence packaged : NO"
)

print(
    "Training evidence exclusion : PASS"
)


# =============================================================================
# 17. VERIFY PACKAGE CONTENT
# =============================================================================

runtime_files = sorted(
    p for p in ASSETS_ROOT.rglob("*")
    if p.is_file()
)

total_bytes = sum(
    p.stat().st_size
    for p in runtime_files
)

print()
print("=" * 100)
print("RUNTIME PACKAGE INVENTORY")
print("=" * 100)

print(
    "Runtime files :",
    len(runtime_files),
)

print(
    "Runtime size  :",
    f"{total_bytes / (1024 ** 3):.3f} GB",
)

print(
    "Runtime asset inventory : PASS"
)


# =============================================================================
# 18. SELF-CONTAINMENT
# =============================================================================

for p in runtime_files:

    resolved = p.resolve()

    assert (
        ASSETS_ROOT
        in resolved.parents
    )


# No notebook is copied into assets.
assert not any(
    p.suffix.lower() == ".ipynb"
    for p in runtime_files
)

print(
    "Notebook inside assets : NO"
)

print(
    "Self-containment : PASS"
)


# =============================================================================
# 19. WRITE MANIFEST
# =============================================================================

manifest = {
    "cell": "9",
    "status": "PASS",
    "runtime_entrypoint": "main.py",

    "runtime_is_self_contained": True,
    "notebook_dependency": False,
    "dataset_dependency": False,
    "scratch_runtime_dependency": False,

    "test_data_accessed": False,
    "submission_generated": False,
    "submission_zip_generated": False,

    "training_runtime": False,
    "test_fitting": False,
    "network_dependency": False,

    "production_method": "raw_blend",

    "weights": {
        "modernbert": 0.419,
        "structured_prior": 0.351,
        "tfidf": 0.230,
    },

    "runtime_asset_count": len(
        runtime_files
    ),

    "runtime_asset_bytes": int(
        total_bytes
    ),

    "assets": copy_records,
}


MANIFEST_PATH = (
    CELL9_ROOT
    / "cell9_self_contained_runtime_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 20. FINAL
# =============================================================================

print()
print("=" * 100)
print("TRACE THE ACE — CELL 9 FINAL STATUS")
print("=" * 100)

print(
    "Physical artifact resolution       : PASS"
)

print(
    "R0 runtime artifacts               : PASS"
)

print(
    "R1 runtime artifacts               : PASS"
)

print(
    "R2 runtime artifacts               : PASS"
)

print(
    "R3 runtime artifacts               : PASS"
)

print(
    "Cross-encoder artifacts            : PASS"
)

print(
    "SHA-256 source → package parity    : PASS"
)

print(
    "Training evidence excluded        : PASS"
)

print(
    "Notebook packaged                  : NO"
)

print(
    "Test data accessed                 : NO"
)

print(
    "Runtime self-containment           : PASS"
)

print(
    "main.py generated                  : NO"
)

print(
    "submission.csv generated          : NO"
)

print(
    "submission.zip generated           : NO"
)

print()
print(
    "Manifest :",
    MANIFEST_PATH,
)

print()
print(
    "CELL 9 COMPLETE — PASS"
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 9 — SELF-CONTAINED RUNTIME ARTIFACT CLOSURE

PATH CONTRACT
PROJECT_ROOT   : D:\Competition\Trace-the-race-local
SCRATCH_ROOT   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs
SUBMISSION_ROOT: D:\Competition\Trace-the-race-local\submission_runtime
ASSETS_ROOT    : D:\Competition\Trace-the-race-local\submission_runtime\assets
CELL9_ROOT     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\submission_runtime\cell9
Path contract : PASS

CELL 6 ARTIFACT DISCOVERY
Cell 6 root : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\submission_runtime\cell6
Cell 6 files : 1
  - cell6_runtime_dependency_manifest.json
Cell 6 wording/schema assertion : SKIPPED
Reason : physical artifact resolution is authoritative.

RETRIEVAL ROOT DISCOVERY
Retrieval root : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval
Retrieval root : PASS

R0 ARTIFACT RESOLUTION
retrieval_queries.parquet : PASS
session_tu

In [22]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 10 — INFERENCE-ONLY SOURCE EXTRACTION + MAIN.PY BUILD
# =============================================================================

from __future__ import annotations

import ast
import json
import re
import shutil
import textwrap
from pathlib import Path


# =============================================================================
# 0. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT / "assets"
)

CELL10_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell10"
)

CELL10_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()


# =============================================================================
# 1. LOAD NOTEBOOK SOURCE
# =============================================================================

NOTEBOOK_ROOT = PROJECT_ROOT / "Notebooks"

MODERNBERT_NOTEBOOK = (
    NOTEBOOK_ROOT
    / "09b-modernbert-mastery-modalipynb.ipynb"
)

assert MODERNBERT_NOTEBOOK.exists(), (
    f"Missing ModernBERT notebook: {MODERNBERT_NOTEBOOK}"
)

with open(
    MODERNBERT_NOTEBOOK,
    "r",
    encoding="utf-8",
) as f:
    modernbert_nb = json.load(f)

assert "cells" in modernbert_nb


# =============================================================================
# 2. SOURCE CELL DISCOVERY
# =============================================================================

source_cells = []

for idx, cell in enumerate(
    modernbert_nb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if not source.strip():
        continue

    source_cells.append(
        {
            "index": idx,
            "source": source,
        }
    )


assert source_cells, (
    "No executable ModernBERT source cells found."
)

print(
    "ModernBERT code cells discovered :",
    len(source_cells),
)


# =============================================================================
# 3. LOCATE THE PRODUCTION / INFERENCE CELL
# =============================================================================

# Cell 8 was identified by the previous source-closure audit.
# Do NOT assume that every statement in that cell belongs in runtime.

TARGET_CELL_INDEX = 8

target = None

for item in source_cells:
    if item["index"] == TARGET_CELL_INDEX:
        target = item
        break

assert target is not None, (
    f"ModernBERT source cell {TARGET_CELL_INDEX} not found."
)

raw_source = target["source"]

print(
    "Selected ModernBERT source cell :",
    TARGET_CELL_INDEX,
)

print(
    "Raw source lines :",
    len(raw_source.splitlines()),
)


# =============================================================================
# 4. AST PARSE
# =============================================================================

tree = ast.parse(
    textwrap.dedent(raw_source)
)

print(
    "AST parse : PASS"
)


# =============================================================================
# 5. TRAINING-ONLY SYMBOLS
# =============================================================================

TRAINING_ONLY_FUNCTIONS = {
    "train",
    "train_model",
    "train_production_fold",
    "save_training_checkpoint",
    "training_step",
    "run_training",
    "fit_model",
}

TRAINING_ONLY_CLASSES = {
    "Trainer",
}

TRAINING_ONLY_NAMES = {
    "optimizer",
    "scheduler",
    "trainer",
    "training_args",
    "train_dataset",
    "eval_dataset",
}


# =============================================================================
# 6. FORBIDDEN EXECUTABLE OPERATIONS
# =============================================================================

# These are actual operations, not substring matches.
#
# A comment containing "train" does NOT trigger this gate.
# A function definition named train_production_fold does NOT trigger this
# gate by itself because that definition will be removed below.

FORBIDDEN_CALLS = {
    "backward",
    "step",
    "fit",
    "partial_fit",
}

FORBIDDEN_ATTRIBUTES = {
    "train",
    "backward",
    "step",
    "fit",
    "partial_fit",
}


# =============================================================================
# 7. AST HELPERS
# =============================================================================

def called_name(node):
    """
    Return a normalized function name for a Call node.
    """
    if not isinstance(node, ast.Call):
        return None

    func = node.func

    if isinstance(func, ast.Name):
        return func.id

    if isinstance(func, ast.Attribute):
        return func.attr

    return None


def attribute_name(node):
    """
    Return the final attribute component.
    """
    if isinstance(node, ast.Attribute):
        return node.attr

    return None


# =============================================================================
# 8. REMOVE TRAINING-ONLY DEFINITIONS
# =============================================================================

runtime_nodes = []

removed_training_definitions = []

for node in tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        if node.name in TRAINING_ONLY_FUNCTIONS:
            removed_training_definitions.append(
                node.name
            )
            continue

    if isinstance(
        node,
        ast.ClassDef,
    ):
        if node.name in TRAINING_ONLY_CLASSES:
            removed_training_definitions.append(
                node.name
            )
            continue

    runtime_nodes.append(node)


print(
    "Training-only definitions removed :",
    sorted(set(removed_training_definitions)),
)


# =============================================================================
# 9. AST-LEVEL RUNTIME SAFETY AUDIT
# =============================================================================

violations = []


for node in ast.walk(
    ast.Module(
        body=runtime_nodes,
        type_ignores=[],
    )
):

    if isinstance(
        node,
        ast.Call,
    ):
        name = called_name(node)

        if name in FORBIDDEN_CALLS:
            violations.append(
                {
                    "type": "forbidden_call",
                    "name": name,
                    "line": getattr(
                        node,
                        "lineno",
                        None,
                    ),
                }
            )

    if isinstance(
        node,
        ast.Attribute,
    ):
        name = attribute_name(node)

        if name in FORBIDDEN_ATTRIBUTES:

            # model.train() is forbidden.
            # tokenizer.model_max_length etc. are not.
            #
            # Only reject the attribute when it is actually part of a call.
            parent_call = False

            # Parent relationships are not present in Python AST,
            # so this is handled below using the complete source AST.

            if name in {
                "backward",
                "step",
                "fit",
                "partial_fit",
            }:
                violations.append(
                    {
                        "type": "forbidden_attribute",
                        "name": name,
                        "line": getattr(
                            node,
                            "lineno",
                            None,
                        ),
                    }
                )


# =============================================================================
# 10. SECONDARY CALL AUDIT
# =============================================================================

for node in ast.walk(
    ast.Module(
        body=runtime_nodes,
        type_ignores=[],
    )
):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue

    name = called_name(node)

    if name == "train":
        violations.append(
            {
                "type": "training_call",
                "name": "train",
                "line": getattr(
                    node,
                    "lineno",
                    None,
                ),
            }
        )


# =============================================================================
# 11. REMOVE DUPLICATES
# =============================================================================

unique_violations = []

seen = set()

for item in violations:

    key = (
        item["type"],
        item["name"],
        item["line"],
    )

    if key in seen:
        continue

    seen.add(key)
    unique_violations.append(item)

violations = unique_violations


# =============================================================================
# 12. HARD SAFETY GATE
# =============================================================================

if violations:

    print(
        "\nACTUAL EXECUTABLE TRAINING OPERATIONS FOUND:"
    )

    for item in violations:
        print(
            " ",
            item,
        )

    raise RuntimeError(
        "Inference-only AST still contains executable "
        "training operations. main.py will NOT be generated."
    )

print(
    "Inference-only AST safety : PASS"
)


# =============================================================================
# 13. INFERENCE SYMBOL INVENTORY
# =============================================================================

defined_functions = []
defined_classes = []
defined_names = []

for node in runtime_nodes:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        defined_functions.append(
            node.name
        )

    elif isinstance(
        node,
        ast.ClassDef,
    ):
        defined_classes.append(
            node.name
        )

    elif isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        ),
    ):
        targets = []

        if isinstance(
            node,
            ast.Assign,
        ):
            targets = node.targets

        else:
            targets = [node.target]

        for target_node in targets:

            if isinstance(
                target_node,
                ast.Name,
            ):
                defined_names.append(
                    target_node.id
                )


print(
    "\nRuntime functions:"
)

for name in defined_functions:
    print(
        " ",
        name,
    )

print(
    "\nRuntime classes:"
)

for name in defined_classes:
    print(
        " ",
        name,
    )


# =============================================================================
# 14. IMPORTANT: DO NOT BLINDLY SERIALIZE THE CELL
# =============================================================================

# We deliberately do not use:
#
#     ast.unparse(tree)
#
# on the entire notebook cell.
#
# The source cell may contain:
# - training definitions
# - checkpoint saving
# - notebook-only diagnostics
# - validation code
# - artifact generation
#
# Only verified inference primitives may enter main.py.


# =============================================================================
# 15. BUILD MINIMAL MODERNBERT LOADER
# =============================================================================

modernbert_runtime = r'''
# ---------------------------------------------------------------------------
# ModernBERT inference runtime
# ---------------------------------------------------------------------------

from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


MODERNBERT_ROOT = (
    Path(__file__).resolve().parent
    / "assets"
    / "modernbert"
)

MODERNBERT_FOLDS = [
    MODERNBERT_ROOT / "fold_1",
    MODERNBERT_ROOT / "fold_2",
    MODERNBERT_ROOT / "fold_3",
    MODERNBERT_ROOT / "fold_4",
]


def load_modernbert_models():
    """
    Load the four frozen production checkpoints.

    No training occurs here.
    """
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    models = []
    tokenizers = []

    for checkpoint in MODERNBERT_FOLDS:

        tokenizer = AutoTokenizer.from_pretrained(
            str(checkpoint),
            local_files_only=True,
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            str(checkpoint),
            local_files_only=True,
        )

        model.to(device)
        model.eval()

        tokenizers.append(tokenizer)
        models.append(model)

    return (
        models,
        tokenizers,
        device,
    )
'''


# =============================================================================
# 16. VERIFY GENERATED MODERNBERT RUNTIME
# =============================================================================

runtime_tree = ast.parse(
    modernbert_runtime
)

runtime_violations = []

for node in ast.walk(
    runtime_tree
):

    if isinstance(
        node,
        ast.Call,
    ):

        name = called_name(node)

        if name in FORBIDDEN_CALLS:
            runtime_violations.append(
                {
                    "name": name,
                    "line": getattr(
                        node,
                        "lineno",
                        None,
                    ),
                }
            )

if runtime_violations:

    raise RuntimeError(
        "Generated ModernBERT runtime failed "
        "inference-only validation."
    )

print(
    "Generated ModernBERT runtime safety : PASS"
)


# =============================================================================
# 17. WRITE VERIFIED MODERNBERT RUNTIME MODULE
# =============================================================================

MODERNBERT_MODULE = (
    SUBMISSION_ROOT
    / "modernbert_runtime.py"
)

MODERNBERT_MODULE.write_text(
    modernbert_runtime.strip()
    + "\n",
    encoding="utf-8",
)

assert MODERNBERT_MODULE.exists()

print(
    "ModernBERT runtime module :",
    MODERNBERT_MODULE,
)


# =============================================================================
# 18. BUILD MAIN.PY SKELETON
# =============================================================================

main_runtime = r'''
from pathlib import Path
import sys

from modernbert_runtime import load_modernbert_models


PROJECT_ROOT = Path(__file__).resolve().parent
DATA_ROOT = PROJECT_ROOT / "data"


def main():
    """
    Submission entrypoint.

    The competition runtime mounts data/ externally.
    This script never trains on test data.
    """

    test_features = DATA_ROOT / "test_features.csv"
    test_transcripts = DATA_ROOT / "test_transcripts"
    submission_format = DATA_ROOT / "submission_format.csv"

    if not test_features.exists():
        raise FileNotFoundError(
            f"Missing runtime test features: {test_features}"
        )

    if not test_transcripts.exists():
        raise FileNotFoundError(
            f"Missing runtime transcripts: {test_transcripts}"
        )

    if not submission_format.exists():
        raise FileNotFoundError(
            f"Missing submission template: {submission_format}"
        )

    # Frozen ModernBERT production models.
    models, tokenizers, device = load_modernbert_models()

    # -----------------------------------------------------------------------
    # IMPORTANT
    # -----------------------------------------------------------------------
    # The complete R0 -> R1 -> R2 -> R3 -> cross-encoder -> evidence ->
    # ModernBERT -> structured prior -> TF-IDF -> raw blend orchestration
    # must be inserted here only after each required inference implementation
    # has passed its own artifact/source closure.
    #
    # This cell intentionally refuses to fabricate approximate retrieval or
    # evidence logic.
    #
    # Therefore this skeleton is NOT yet a valid competition submission.
    # -----------------------------------------------------------------------

    raise RuntimeError(
        "main.py inference orchestration is not yet closed. "
        "Do not submit this package."
    )


if __name__ == "__main__":
    main()
'''


MAIN_PATH = (
    SUBMISSION_ROOT / "main.py"
)

MAIN_PATH.write_text(
    main_runtime.strip()
    + "\n",
    encoding="utf-8",
)

assert MAIN_PATH.exists()


# =============================================================================
# 19. MAIN.PY AST VALIDATION
# =============================================================================

main_source = MAIN_PATH.read_text(
    encoding="utf-8"
)

main_tree = ast.parse(
    main_source
)

main_violations = []

for node in ast.walk(
    main_tree
):

    if isinstance(
        node,
        ast.Call,
    ):

        name = called_name(node)

        if name in FORBIDDEN_CALLS:
            main_violations.append(
                {
                    "name": name,
                    "line": getattr(
                        node,
                        "lineno",
                        None,
                    ),
                }
            )

if main_violations:

    raise RuntimeError(
        f"main.py failed runtime safety audit: "
        f"{main_violations}"
    )

print(
    "main.py AST safety : PASS"
)


# =============================================================================
# 20. MANIFEST
# =============================================================================

manifest = {
    "cell": 10,
    "status": "INFERENCE_SOURCE_EXTRACTION_PASS",
    "training_definitions_removed": sorted(
        set(removed_training_definitions)
    ),
    "raw_source_cell": TARGET_CELL_INDEX,
    "runtime_functions": sorted(
        defined_functions
    ),
    "runtime_classes": sorted(
        defined_classes
    ),
    "runtime_training_operations": [],
    "main_py_generated": True,
    "main_py_is_submission_ready": False,
    "reason_not_submission_ready": (
        "Full R0-R3, cross-encoder, evidence, "
        "ModernBERT, structured-prior and TF-IDF "
        "orchestration has not yet been assembled."
    ),
}

MANIFEST_PATH = (
    CELL10_ROOT
    / "cell10_inference_source_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 21. FINAL STATUS
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRACE THE ACE — CELL 10 FINAL STATUS"
)

print(
    "=" * 100
)

print(
    "Raw ModernBERT source parsed       : PASS"
)

print(
    "Training definitions removed       : PASS"
)

print(
    "Executable training scan           : PASS"
)

print(
    "Generated ModernBERT loader         : PASS"
)

print(
    "main.py generated                   : PASS"
)

print(
    "Full inference orchestration       : NOT YET"
)

print(
    "Submission-ready                    : NO"
)

print(
    "Submission ZIP                      : NOT GENERATED"
)

print(
    "Manifest :",
    MANIFEST_PATH,
)

print(
    "=" * 100
)

ModernBERT code cells discovered : 8
Selected ModernBERT source cell : 8
Raw source lines : 350
AST parse : PASS
Training-only definitions removed : []
Inference-only AST safety : PASS

Runtime functions:

Runtime classes:
Generated ModernBERT runtime safety : PASS
ModernBERT runtime module : D:\Competition\Trace-the-race-local\submission_runtime\modernbert_runtime.py
main.py AST safety : PASS

TRACE THE ACE — CELL 10 FINAL STATUS
Raw ModernBERT source parsed       : PASS
Training definitions removed       : PASS
Executable training scan           : PASS
Generated ModernBERT loader         : PASS
main.py generated                   : PASS
Full inference orchestration       : NOT YET
Submission-ready                    : NO
Submission ZIP                      : NOT GENERATED
Manifest : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\submission_runtime\cell10\cell10_inference_source_manifest.json


In [23]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 11 — STANDALONE MAIN.PY RUNTIME ASSEMBLY
# =============================================================================
#
# PURPOSE
# -------
# Build a genuinely self-contained submission_runtime/main.py.
#
# IMPORTANT:
#   * Notebooks are BUILD-TIME sources only.
#   * No notebook is imported/executed by main.py.
#   * No Dataset/ dependency is allowed.
#   * No scratch_mastery_outputs dependency is allowed.
#   * No training is allowed.
#   * No test fitting / pseudo-labeling is allowed.
#   * No approximate retrieval/evidence implementation is allowed.
#
# If exact runtime closure cannot be established, this cell FAILS and does not
# produce a submission-ready main.py.
# =============================================================================

from __future__ import annotations

import ast
import hashlib
import json
import re
import shutil
import textwrap
from collections import defaultdict, deque
from pathlib import Path


# =============================================================================
# 0. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT / "Notebooks"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT / "assets"
)

CELL11_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell11"
)

CELL11_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists()
assert NOTEBOOK_ROOT.exists()
assert SUBMISSION_ROOT.exists()
assert ASSETS_ROOT.exists()


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 11 — STANDALONE MAIN.PY RUNTIME ASSEMBLY"
)
print("=" * 100)


# =============================================================================
# 1. LOCKED PRODUCTION CONTRACT
# =============================================================================

BLEND_WEIGHTS = {
    "modernbert": 0.419,
    "structured_prior": 0.351,
    "tfidf": 0.230,
}

assert abs(
    sum(BLEND_WEIGHTS.values()) - 1.0
) < 1e-12

print()
print("LOCKED PRODUCTION METHOD")
print("-" * 80)
print("ModernBERT       :", BLEND_WEIGHTS["modernbert"])
print("Structured+Prior :", BLEND_WEIGHTS["structured_prior"])
print("TF-IDF           :", BLEND_WEIGHTS["tfidf"])
print("Weight sum       :", sum(BLEND_WEIGHTS.values()))
print("Production method: raw_blend")


# =============================================================================
# 2. REQUIRED NOTEBOOK SOURCES
# =============================================================================

NOTEBOOKS = {
    "r0": NOTEBOOK_ROOT
    / "04_R0_retrieval_input_builder_FIXED.ipynb",

    "r1": NOTEBOOK_ROOT
    / "05_R1_sparse_retrieval.ipynb",

    "r2": NOTEBOOK_ROOT
    / "06_R2_dense_retrieval.ipynb",

    "r3": NOTEBOOK_ROOT
    / "07_R3 _Candidate_Union_Sparse_Dense_Fusion.ipynb",

    "cross_encoder": NOTEBOOK_ROOT
    / "08_cross_encoder_reranking.ipynb",

    "evidence": NOTEBOOK_ROOT
    / "09_evidence_pack_builder.ipynb",

    "modernbert": NOTEBOOK_ROOT
    / "09b-modernbert-mastery-modalipynb.ipynb",

    "structured": NOTEBOOK_ROOT
    / "10_structured_prior_oof.ipynb",
}


print()
print("SOURCE NOTEBOOK CONTRACT")
print("-" * 80)

missing_notebooks = []

for name, path in NOTEBOOKS.items():

    if not path.exists():
        missing_notebooks.append(
            (name, str(path))
        )
        print(
            "MISSING:",
            name,
            "|",
            path,
        )
    else:
        print(
            "PASS   :",
            name,
            "|",
            path.name,
        )

if missing_notebooks:
    raise RuntimeError(
        "Required source notebooks are missing. "
        "Exact runtime assembly cannot continue."
    )


# =============================================================================
# 3. NOTEBOOK LOADER
# =============================================================================

def load_notebook(path: Path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        obj = json.load(f)

    assert obj.get("cells") is not None

    return obj


NOTEBOOK_OBJECTS = {
    name: load_notebook(path)
    for name, path in NOTEBOOKS.items()
}


# =============================================================================
# 4. AST SOURCE INDEX
# =============================================================================

#
# We index:
#
#   functions
#   classes
#   assignments
#   imports
#
# across all source notebooks.
#
# This is BUILD-TIME analysis.
#
# main.py will contain extracted Python source only.
#

DEFINITIONS = defaultdict(list)

IMPORT_NODES = defaultdict(list)

ASSIGNMENT_NODES = defaultdict(list)

SOURCE_RECORDS = []


def source_of_cell(cell):
    return "".join(
        cell.get("source", [])
    )


for notebook_name, notebook in NOTEBOOK_OBJECTS.items():

    for cell_index, cell in enumerate(
        notebook["cells"]
    ):

        if cell.get("cell_type") != "code":
            continue

        source = source_of_cell(cell)

        if not source.strip():
            continue

        try:
            tree = ast.parse(
                textwrap.dedent(source)
            )
        except SyntaxError:
            continue

        record = {
            "notebook": notebook_name,
            "cell_index": cell_index,
            "source": source,
            "tree": tree,
        }

        SOURCE_RECORDS.append(record)

        for node in tree.body:

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            ):

                DEFINITIONS[
                    node.name
                ].append(
                    {
                        "record": record,
                        "node": node,
                    }
                )

            elif isinstance(
                node,
                (
                    ast.Assign,
                    ast.AnnAssign,
                ),
            ):

                targets = []

                if isinstance(
                    node,
                    ast.Assign,
                ):
                    targets = node.targets
                else:
                    targets = [node.target]

                for target in targets:

                    if isinstance(
                        target,
                        ast.Name,
                    ):
                        ASSIGNMENT_NODES[
                            target.id
                        ].append(
                            {
                                "record": record,
                                "node": node,
                            }
                        )

            elif isinstance(
                node,
                (
                    ast.Import,
                    ast.ImportFrom,
                ),
            ):

                IMPORT_NODES[
                    notebook_name
                ].append(
                    {
                        "record": record,
                        "node": node,
                    }
                )


print()
print("SOURCE INDEX")
print("-" * 80)
print(
    "Source code cells indexed :",
    len(SOURCE_RECORDS),
)
print(
    "Function/class symbols     :",
    len(DEFINITIONS),
)
print(
    "Assignment symbols         :",
    len(ASSIGNMENT_NODES),
)


# =============================================================================
# 5. REQUIRED RUNTIME ENTRY SYMBOLS
# =============================================================================

#
# These are the symbols that must exist somewhere in the source closure.
#
# We deliberately use candidate lists where notebook revisions may use a
# different function name.
#

REQUIRED_SYMBOL_CANDIDATES = {

    "r0": [
        "score_one_response_session",
        "sha256_file_r0_session_turn_index",
    ],

    "r1": [
        "select_top_k_candidates",
        "score_one_response_session",
        "r1_word_score_session",
    ],

    "r2": [
        "score_r2_dense_response",
        "r2_dense_load_checkpoint",
    ],

    "r3": [
        "r3_cell7_sha256",
        "r3_cell8_sha256",
        "r3_sha256_file",
    ],

    "cross_encoder": [
        "score_chunk",
        "resolve_turn_texts",
        "build_turn_lookup_database",
    ],

    "evidence": [
        "build_pack",
        "make_section_text",
        "canonical_role",
        "normalize_text",
    ],

    "modernbert": [
        "build_model",
        "production_collate",
    ],

    "structured": [
        "build_inner_cross_fitted_prior",
    ],
}


# =============================================================================
# 6. SYMBOL RESOLUTION
# =============================================================================

resolved_symbols = {}
unresolved_symbols = []


for stage, candidates in (
    REQUIRED_SYMBOL_CANDIDATES.items()
):

    stage_resolution = []

    for symbol in candidates:

        if symbol in DEFINITIONS:

            locations = []

            for item in DEFINITIONS[
                symbol
            ]:

                locations.append(
                    {
                        "notebook": item[
                            "record"
                        ]["notebook"],
                        "cell": item[
                            "record"
                        ]["cell_index"],
                    }
                )

            stage_resolution.append(
                {
                    "symbol": symbol,
                    "locations": locations,
                }
            )

    resolved_symbols[
        stage
    ] = stage_resolution

    if not stage_resolution:

        unresolved_symbols.append(
            {
                "stage": stage,
                "candidates": candidates,
            }
        )


print()
print("REQUIRED SYMBOL RESOLUTION")
print("-" * 80)

for stage, entries in (
    resolved_symbols.items()
):

    print()
    print(stage)

    if not entries:
        print("  UNRESOLVED")
        continue

    for entry in entries:

        print(
            " ",
            entry["symbol"],
            "->",
            entry["locations"],
        )


# =============================================================================
# 7. IMPORTANT RUNTIME ARTIFACT CHECK
# =============================================================================

#
# Cell 9 already copied the runtime artifacts.
# We verify their physical presence here.
#

REQUIRED_ASSET_PATHS = [
    ASSETS_ROOT / "retrieval",
    ASSETS_ROOT / "modernbert",
    ASSETS_ROOT / "structured_prior",
    ASSETS_ROOT / "tfidf",
]


print()
print("RUNTIME ASSET ROOTS")
print("-" * 80)

missing_asset_roots = []

for path in REQUIRED_ASSET_PATHS:

    if path.exists():

        print(
            "PASS:",
            path.relative_to(
                SUBMISSION_ROOT
            ),
        )

    else:

        print(
            "MISSING:",
            path,
        )

        missing_asset_roots.append(
            str(path)
        )

if missing_asset_roots:

    raise RuntimeError(
        "Runtime asset closure is incomplete."
    )


# =============================================================================
# 8. FORBIDDEN SOURCE DETECTION
# =============================================================================

#
# IMPORTANT:
# Do not search for the substring "train".
#
# A source file containing:
#
#     train_production_fold
#
# does NOT prove runtime training.
#
# We inspect executable AST operations instead.
#

FORBIDDEN_CALL_NAMES = {
    "fit",
    "partial_fit",
    "backward",
    "step",
}

FORBIDDEN_TRAIN_METHODS = {
    "train",
}


def find_executable_training_operations(
    tree
):

    hits = []

    for node in ast.walk(tree):

        if not isinstance(
            node,
            ast.Call,
        ):
            continue

        func = node.func

        name = None

        if isinstance(
            func,
            ast.Name,
        ):
            name = func.id

        elif isinstance(
            func,
            ast.Attribute,
        ):
            name = func.attr

        if name in FORBIDDEN_CALL_NAMES:

            hits.append(
                {
                    "line": getattr(
                        node,
                        "lineno",
                        None,
                    ),
                    "operation": name,
                }
            )

        elif name in FORBIDDEN_TRAIN_METHODS:

            hits.append(
                {
                    "line": getattr(
                        node,
                        "lineno",
                        None,
                    ),
                    "operation": name,
                }
            )

    return hits


# =============================================================================
# 9. RUNTIME CLOSURE EXTRACTION
# =============================================================================

#
# We extract only:
#
#   - imports
#   - definitions
#   - required assignments
#
# from the resolved source.
#
# We DO NOT copy entire notebooks.
#

closure_nodes = []

closure_sources = []

selected_locations = set()


def add_definition_symbol(
    symbol,
):

    if symbol not in DEFINITIONS:
        return False

    # Prefer the first definition from the appropriate source.
    item = DEFINITIONS[
        symbol
    ][0]

    record = item["record"]
    node = item["node"]

    location = (
        record["notebook"],
        record["cell_index"],
        symbol,
    )

    if location in selected_locations:
        return True

    selected_locations.add(
        location
    )

    closure_nodes.append(
        (
            record,
            node,
        )
    )

    return True


# Add all candidate runtime functions/classes.
for stage_entries in (
    resolved_symbols.values()
):

    for entry in stage_entries:

        add_definition_symbol(
            entry["symbol"]
        )


# =============================================================================
# 10. TRANSITIVE FUNCTION DEPENDENCY DISCOVERY
# =============================================================================

#
# A runtime function may call another helper.
#
# We recursively resolve Name references.
#

def names_used_by_node(
    node,
):

    names = set()

    for child in ast.walk(node):

        if isinstance(
            child,
            ast.Name,
        ):
            if isinstance(
                child.ctx,
                ast.Load,
            ):
                names.add(
                    child.id
                )

    return names


queue = deque()

for record, node in closure_nodes:

    for name in names_used_by_node(
        node
    ):

        queue.append(name)


visited_names = set()

while queue:

    name = queue.popleft()

    if name in visited_names:
        continue

    visited_names.add(name)

    if name not in DEFINITIONS:
        continue

    # Avoid pulling obvious training-only functions.
    if name in {
        "train",
        "train_model",
        "train_production_fold",
        "save_training_checkpoint",
        "training_step",
        "run_training",
    }:
        continue

    before = len(
        closure_nodes
    )

    add_definition_symbol(
        name
    )

    after = len(
        closure_nodes
    )

    if after > before:

        record, node = closure_nodes[-1]

        for dep in names_used_by_node(
            node
        ):

            if dep not in visited_names:
                queue.append(dep)


print()
print("TRANSITIVE CLOSURE")
print("-" * 80)
print(
    "Runtime definitions selected :",
    len(closure_nodes),
)


# =============================================================================
# 11. SOURCE-LEVEL SAFETY AUDIT
# =============================================================================

closure_ast = ast.Module(
    body=[
        node
        for record, node in closure_nodes
    ],
    type_ignores=[],
)

training_hits = (
    find_executable_training_operations(
        closure_ast
    )
)

if training_hits:

    print()
    print(
        "FORBIDDEN EXECUTABLE OPERATIONS:"
    )

    for hit in training_hits:

        print(
            " ",
            hit,
        )

    raise RuntimeError(
        "Runtime closure contains executable "
        "training operations."
    )

print()
print(
    "Executable training operation scan : PASS"
)


# =============================================================================
# 12. EXTERNAL SYMBOL ANALYSIS
# =============================================================================

#
# Any Name loaded by runtime definitions that is not:
#
#   - defined in closure
#   - builtin
#   - imported
#
# is potentially a hidden notebook global.
#
# Those must be resolved before final assembly.
#

BUILTINS = set(
    dir(__builtins__)
)

defined_names = set()

for record, node in closure_nodes:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        ),
    ):
        defined_names.add(
            node.name
        )

    elif isinstance(
        node,
        ast.Assign,
    ):

        for target in node.targets:

            if isinstance(
                target,
                ast.Name,
            ):
                defined_names.add(
                    target.id
                )

    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        if isinstance(
            node.target,
            ast.Name,
        ):
            defined_names.add(
                node.target.id
            )


imported_names = set()

for record, node in closure_nodes:

    # Imports that are nested inside functions are also collected.
    for child in ast.walk(node):

        if isinstance(
            child,
            ast.Import,
        ):

            for alias in child.names:

                imported_names.add(
                    alias.asname
                    or alias.name.split(
                        "."
                    )[0]
                )

        elif isinstance(
            child,
            ast.ImportFrom,
        ):

            for alias in child.names:

                imported_names.add(
                    alias.asname
                    or alias.name
                )


external_names = set()

for record, node in closure_nodes:

    for name in names_used_by_node(
        node
    ):

        if name in defined_names:
            continue

        if name in imported_names:
            continue

        if name in BUILTINS:
            continue

        external_names.add(
            name
        )


# Names that are normal module-level references may be resolved through
# assignments in the original notebooks.
unresolved_external_names = []

for name in sorted(
    external_names
):

    if name in ASSIGNMENT_NODES:

        # We found a source assignment.
        continue

    # Common harmless annotations / implementation names can be ignored only
    # when they are clearly provided by Python.
    if name in {
        "np",
        "pd",
        "torch",
        "os",
        "json",
        "math",
        "re",
        "Path",
        "PathLike",
        "Optional",
        "List",
        "Dict",
        "Tuple",
        "Any",
    }:
        continue

    unresolved_external_names.append(
        name
    )


print()
print("EXTERNAL SYMBOL AUDIT")
print("-" * 80)

print(
    "External names discovered :",
    len(external_names),
)

print(
    "Unresolved external names :",
    len(unresolved_external_names),
)

if unresolved_external_names:

    print()

    for name in (
        unresolved_external_names[:100]
    ):
        print(
            "  -",
            name,
        )


# =============================================================================
# 13. DO NOT SILENTLY FABRICATE DEPENDENCIES
# =============================================================================

#
# This is deliberately strict.
#
# If unresolved notebook globals exist, we do not guess their meaning.
#

if unresolved_external_names:

    raise RuntimeError(
        "Runtime source closure contains unresolved "
        "external symbols. Resolve them before generating "
        "a submission-ready main.py."
    )


# =============================================================================
# 14. IMPORT CLOSURE
# =============================================================================

all_import_nodes = []

seen_import_source = set()

for record, node in closure_nodes:

    for child in ast.walk(node):

        if isinstance(
            child,
            (
                ast.Import,
                ast.ImportFrom,
            ),
        ):

            source = ast.unparse(
                child
            )

            if source in seen_import_source:
                continue

            seen_import_source.add(
                source
            )

            all_import_nodes.append(
                source
            )


# =============================================================================
# 15. BUILD EXTRACTED SOURCE
# =============================================================================

definition_blocks = []

seen_definition_source = set()

for record, node in closure_nodes:

    if not isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        ),
    ):
        continue

    source = ast.unparse(
        node
    )

    if source in seen_definition_source:
        continue

    seen_definition_source.add(
        source
    )

    definition_blocks.append(
        source
    )


# =============================================================================
# 16. RUNTIME HEADER
# =============================================================================

runtime_header = r'''
from __future__ import annotations

import json
import math
import os
import re
import sys
import hashlib
from pathlib import Path
from typing import Any, Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import torch


PROJECT_ROOT = Path(
    __file__
).resolve().parent

DATA_ROOT = (
    PROJECT_ROOT / "data"
)

ASSETS_ROOT = (
    PROJECT_ROOT / "assets"
)

RETRIEVAL_ROOT = (
    ASSETS_ROOT / "retrieval"
)

MODERNBERT_ROOT = (
    ASSETS_ROOT / "modernbert"
)

STRUCTURED_ROOT = (
    ASSETS_ROOT / "structured_prior"
)

TFIDF_ROOT = (
    ASSETS_ROOT / "tfidf"
)


BLEND_MODERNBERT = 0.419
BLEND_STRUCTURED = 0.351
BLEND_TFIDF = 0.230


assert abs(
    BLEND_MODERNBERT
    + BLEND_STRUCTURED
    + BLEND_TFIDF
    - 1.0
) < 1e-12
'''


# =============================================================================
# 17. ASSEMBLE MAIN.PY SOURCE
# =============================================================================

main_parts = []

main_parts.append(
    runtime_header.strip()
)

main_parts.extend(
    all_import_nodes
)

main_parts.extend(
    definition_blocks
)


# =============================================================================
# 18. ADD FROZEN MODERNBERT LOADER
# =============================================================================

modernbert_loader = r'''
def _load_production_modernbert():
    """
    Load frozen ModernBERT production checkpoints.

    Runtime-only:
        - no training
        - no optimizer
        - no fitting
        - local checkpoint loading only
    """

    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    models = []
    tokenizers = []

    for fold_id in (
        1,
        2,
        3,
        4,
    ):

        checkpoint = (
            MODERNBERT_ROOT
            / f"fold_{fold_id}"
        )

        if not checkpoint.exists():
            raise FileNotFoundError(
                f"Missing ModernBERT checkpoint: "
                f"{checkpoint}"
            )

        tokenizer = (
            AutoTokenizer.from_pretrained(
                str(checkpoint),
                local_files_only=True,
            )
        )

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                str(checkpoint),
                local_files_only=True,
            )
        )

        model.to(device)
        model.eval()

        tokenizers.append(
            tokenizer
        )

        models.append(
            model
        )

    return (
        models,
        tokenizers,
        device,
    )
'''

main_parts.append(
    modernbert_loader.strip()
)


# =============================================================================
# 19. ADD BLEND FUNCTION
# =============================================================================

blend_function = r'''
def _raw_blend(
    modernbert_probability,
    structured_probability,
    tfidf_probability,
):
    probability = (
        BLEND_MODERNBERT
        * modernbert_probability
        + BLEND_STRUCTURED
        * structured_probability
        + BLEND_TFIDF
        * tfidf_probability
    )

    return float(
        np.clip(
            probability,
            0.0,
            1.0,
        )
    )
'''

main_parts.append(
    blend_function.strip()
)


# =============================================================================
# 20. ADD RUNTIME DATA VALIDATION
# =============================================================================

data_validation = r'''
def _validate_runtime_data():

    test_features = (
        DATA_ROOT
        / "test_features.csv"
    )

    test_transcripts = (
        DATA_ROOT
        / "test_transcripts"
    )

    submission_format = (
        DATA_ROOT
        / "submission_format.csv"
    )

    if not test_features.exists():
        raise FileNotFoundError(
            f"Missing: {test_features}"
        )

    if not test_transcripts.exists():
        raise FileNotFoundError(
            f"Missing: {test_transcripts}"
        )

    if not submission_format.exists():
        raise FileNotFoundError(
            f"Missing: {submission_format}"
        )

    return (
        test_features,
        test_transcripts,
        submission_format,
    )
'''

main_parts.append(
    data_validation.strip()
)


# =============================================================================
# 21. MAIN ORCHESTRATION GUARD
# =============================================================================

#
# We intentionally do NOT fabricate the final orchestration.
#
# The source closure above must first prove that all required stage symbols
# are available. Only then should this block be populated.
#

main_entry = r'''
def main():

    (
        test_features_path,
        test_transcripts_root,
        submission_format_path,
    ) = _validate_runtime_data()

    # -----------------------------------------------------------------------
    # RUNTIME CONTRACT
    # -----------------------------------------------------------------------
    #
    # The following stages MUST execute here:
    #
    #   R0
    #   R1
    #   R2
    #   R3
    #   Cross-encoder
    #   Evidence
    #   ModernBERT
    #   Structured + Prior
    #   TF-IDF
    #   Locked raw blend
    #
    # No training or test-set fitting is permitted.
    #
    # This assembly cell will refuse to produce a submission-ready main.py
    # until those exact stage entrypoints have been resolved.
    # -----------------------------------------------------------------------

    raise RuntimeError(
        "Exact inference orchestration is not yet closed. "
        "Do not submit this package."
    )


if __name__ == "__main__":
    main()
'''

main_parts.append(
    main_entry.strip()
)


# =============================================================================
# 22. WRITE MAIN.PY
# =============================================================================

MAIN_PATH = (
    SUBMISSION_ROOT / "main.py"
)

candidate_main = (
    "\n\n".join(
        main_parts
    )
    + "\n"
)

MAIN_PATH.write_text(
    candidate_main,
    encoding="utf-8",
)


# =============================================================================
# 23. MAIN.PY AST VALIDATION
# =============================================================================

main_tree = ast.parse(
    candidate_main
)

main_training_hits = (
    find_executable_training_operations(
        main_tree
    )
)

if main_training_hits:

    raise RuntimeError(
        "Generated main.py contains executable "
        "training operations."
    )


# =============================================================================
# 24. FORBIDDEN RUNTIME PATH REFERENCES
# =============================================================================

forbidden_runtime_strings = [
    "scratch_mastery_outputs",
    "Notebooks",
    "Dataset",
    ".ipynb",
]

main_lower = candidate_main.lower()

for token in (
    forbidden_runtime_strings
):

    if token.lower() in main_lower:

        raise RuntimeError(
            f"Forbidden runtime dependency found in "
            f"main.py: {token}"
        )


# =============================================================================
# 25. HASH
# =============================================================================

main_sha256 = hashlib.sha256(
    candidate_main.encode(
        "utf-8"
    )
).hexdigest()


# =============================================================================
# 26. MANIFEST
# =============================================================================

manifest = {
    "cell": 11,
    "status": "SOURCE_ASSEMBLY_COMPLETE_BUT_ORCHESTRATION_OPEN",
    "production_method": "raw_blend",
    "blend_weights": BLEND_WEIGHTS,
    "source_notebooks": {
        name: str(path)
        for name, path in NOTEBOOKS.items()
    },
    "source_cells_indexed": len(
        SOURCE_RECORDS
    ),
    "runtime_definitions": len(
        closure_nodes
    ),
    "runtime_training_operations": (
        len(main_training_hits)
    ),
    "unresolved_required_stages": (
        unresolved_symbols
    ),
    "unresolved_external_symbols": (
        unresolved_external_names
    ),
    "main_py": str(
        MAIN_PATH
    ),
    "main_py_sha256": main_sha256,
    "main_py_submission_ready": False,
    "submission_zip_generated": False,
}


MANIFEST_PATH = (
    CELL11_ROOT
    / "cell11_runtime_assembly_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 27. FINAL STATUS
# =============================================================================

print()
print("=" * 100)
print(
    "TRACE THE ACE — CELL 11 FINAL STATUS"
)
print("=" * 100)

print(
    "Source notebook indexing          : PASS"
)

print(
    "Runtime symbol discovery          : PASS"
)

print(
    "Transitive source closure         : PASS"
)

print(
    "Executable training scan          : PASS"
)

print(
    "Forbidden runtime path scan       : PASS"
)

print(
    "main.py written                   : PASS"
)

print(
    "main.py SHA-256                   :",
    main_sha256,
)

print(
    "Full R0→R1→R2→R3 orchestration    : NOT YET"
)

print(
    "Cross-encoder orchestration       : NOT YET"
)

print(
    "Evidence orchestration            : NOT YET"
)

print(
    "ModernBERT orchestration          : NOT YET"
)

print(
    "Structured+Prior orchestration    : NOT YET"
)

print(
    "TF-IDF orchestration              : NOT YET"
)

print(
    "Final submission.csv generation   : NOT YET"
)

print(
    "Submission-ready                  : NO"
)

print(
    "Submission ZIP                    : NOT GENERATED"
)

print(
    "Manifest :",
    MANIFEST_PATH,
)

print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 11 — STANDALONE MAIN.PY RUNTIME ASSEMBLY

LOCKED PRODUCTION METHOD
--------------------------------------------------------------------------------
ModernBERT       : 0.419
Structured+Prior : 0.351
TF-IDF           : 0.23
Weight sum       : 1.0
Production method: raw_blend

SOURCE NOTEBOOK CONTRACT
--------------------------------------------------------------------------------
PASS   : r0 | 04_R0_retrieval_input_builder_FIXED.ipynb
PASS   : r1 | 05_R1_sparse_retrieval.ipynb
PASS   : r2 | 06_R2_dense_retrieval.ipynb
PASS   : r3 | 07_R3 _Candidate_Union_Sparse_Dense_Fusion.ipynb
PASS   : cross_encoder | 08_cross_encoder_reranking.ipynb
PASS   : evidence | 09_evidence_pack_builder.ipynb
PASS   : modernbert | 09b-modernbert-mastery-modalipynb.ipynb
PASS   : structured | 10_structured_prior_oof.ipynb

SOURCE INDEX
--------------------------------------------------------------------------------
Source code cells indexed : 83
Function/class symbols    

RuntimeError: Runtime source closure contains unresolved external symbols. Resolve them before generating a submission-ready main.py.

In [24]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 11A — UNRESOLVED RUNTIME SYMBOL FORENSIC
# =============================================================================

from __future__ import annotations

import ast
import json
from collections import defaultdict
from pathlib import Path


# =============================================================================
# PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT / "Notebooks"
)

CELL11_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell11"
)

CELL11_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


NOTEBOOKS = {
    "r0":
        NOTEBOOK_ROOT
        / "04_R0_retrieval_input_builder_FIXED.ipynb",

    "r1":
        NOTEBOOK_ROOT
        / "05_R1_sparse_retrieval.ipynb",

    "r2":
        NOTEBOOK_ROOT
        / "06_R2_dense_retrieval.ipynb",

    "r3":
        NOTEBOOK_ROOT
        / "07_R3 _Candidate_Union_Sparse_Dense_Fusion.ipynb",

    "cross_encoder":
        NOTEBOOK_ROOT
        / "08_cross_encoder_reranking.ipynb",

    "evidence":
        NOTEBOOK_ROOT
        / "09_evidence_pack_builder.ipynb",

    "modernbert":
        NOTEBOOK_ROOT
        / "09b-modernbert-mastery-modalipynb.ipynb",

    "structured":
        NOTEBOOK_ROOT
        / "10_structured_prior_oof.ipynb",
}


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 11A — UNRESOLVED RUNTIME SYMBOL FORENSIC"
)
print("=" * 100)


# =============================================================================
# 1. SOURCE INDEX
# =============================================================================

definitions = defaultdict(list)
assignments = defaultdict(list)
imports = defaultdict(list)

source_cells = []


for stage, notebook_path in NOTEBOOKS.items():

    assert notebook_path.exists(), (
        f"Missing notebook: {notebook_path}"
    )

    with open(
        notebook_path,
        "r",
        encoding="utf-8",
    ) as f:

        notebook = json.load(f)

    for cell_index, cell in enumerate(
        notebook["cells"]
    ):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get("source", [])
        )

        if not source.strip():
            continue

        try:
            tree = ast.parse(
                source
            )
        except SyntaxError:
            continue

        record = {
            "stage": stage,
            "notebook": str(
                notebook_path
            ),
            "cell": cell_index,
            "source": source,
            "tree": tree,
        }

        source_cells.append(
            record
        )

        for node in tree.body:

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            ):

                definitions[
                    node.name
                ].append(
                    record
                )

            elif isinstance(
                node,
                ast.Assign,
            ):

                for target in node.targets:

                    if isinstance(
                        target,
                        ast.Name,
                    ):

                        assignments[
                            target.id
                        ].append(
                            record
                        )

            elif isinstance(
                node,
                ast.AnnAssign,
            ):

                if isinstance(
                    node.target,
                    ast.Name,
                ):

                    assignments[
                        node.target.id
                    ].append(
                        record
                    )

            elif isinstance(
                node,
                ast.Import,
            ):

                for alias in node.names:

                    imports[
                        alias.asname
                        or alias.name.split(".")[0]
                    ].append(
                        record
                    )

            elif isinstance(
                node,
                ast.ImportFrom,
            ):

                for alias in node.names:

                    imports[
                        alias.asname
                        or alias.name
                    ].append(
                        record
                    )


# =============================================================================
# 2. RECONSTRUCT THE SAME INITIAL RUNTIME SYMBOL SET
# =============================================================================

REQUIRED_SYMBOLS = {

    "r0": [
        "score_one_response_session",
        "sha256_file_r0_session_turn_index",
    ],

    "r1": [
        "select_top_k_candidates",
        "score_one_response_session",
        "r1_word_score_session",
    ],

    "r2": [
        "score_r2_dense_response",
        "r2_dense_load_checkpoint",
    ],

    "r3": [
        "r3_cell7_sha256",
        "r3_cell8_sha256",
        "r3_sha256_file",
    ],

    "cross_encoder": [
        "score_chunk",
        "resolve_turn_texts",
        "build_turn_lookup_database",
    ],

    "evidence": [
        "build_pack",
        "make_section_text",
        "canonical_role",
        "normalize_text",
    ],

    "modernbert": [
        "build_model",
        "production_collate",
    ],

    "structured": [
        "build_inner_cross_fitted_prior",
    ],
}


selected = []


for stage, symbols in REQUIRED_SYMBOLS.items():

    for symbol in symbols:

        if symbol not in definitions:
            continue

        # Prefer definitions from the requested stage.
        candidates = [
            x
            for x in definitions[symbol]
            if x["stage"] == stage
        ]

        if not candidates:
            candidates = definitions[
                symbol
            ]

        selected.append(
            (
                candidates[0],
                symbol,
            )
        )


# =============================================================================
# 3. TRANSITIVE NAME COLLECTION
# =============================================================================

def loaded_names(node):

    result = set()

    for child in ast.walk(node):

        if isinstance(
            child,
            ast.Name,
        ):

            if isinstance(
                child.ctx,
                ast.Load,
            ):

                result.add(
                    child.id
                )

    return result


selected_definition_names = set()

for record, symbol in selected:

    selected_definition_names.add(
        symbol
    )


queue = []

for record, symbol in selected:

    for node in record["tree"].body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ) and node.name == symbol:

            queue.extend(
                loaded_names(node)
            )


visited = set()

while queue:

    name = queue.pop()

    if name in visited:
        continue

    visited.add(name)

    if name not in definitions:
        continue

    # Do not pull training entrypoints.
    if name in {
        "train",
        "train_model",
        "train_production_fold",
        "save_training_checkpoint",
        "training_step",
        "run_training",
    }:
        continue

    candidates = definitions[name]

    # Prefer the first definition.
    record = candidates[0]

    if name not in selected_definition_names:

        selected.append(
            (
                record,
                name,
            )
        )

        selected_definition_names.add(
            name
        )

    for node in record["tree"].body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ) and node.name == name:

            queue.extend(
                loaded_names(node)
            )


# =============================================================================
# 4. IMPORT / DEFINITION / ASSIGNMENT RESOLUTION
# =============================================================================

builtin_names = set(
    dir(__builtins__)
)

defined = set(
    selected_definition_names
)

imported = set()

for record, symbol in selected:

    tree = record["tree"]

    for node in ast.walk(tree):

        if isinstance(
            node,
            ast.Import,
        ):

            for alias in node.names:

                imported.add(
                    alias.asname
                    or alias.name.split(".")[0]
                )

        elif isinstance(
            node,
            ast.ImportFrom,
        ):

            for alias in node.names:

                imported.add(
                    alias.asname
                    or alias.name
                )


used = set()

for record, symbol in selected:

    for node in record["tree"].body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ) and node.name == symbol:

            used.update(
                loaded_names(node)
            )


external = sorted(
    name
    for name in used
    if name not in defined
    and name not in imported
    and name not in builtin_names
)


# =============================================================================
# 5. CLASSIFY EACH EXTERNAL SYMBOL
# =============================================================================

classification = []

for name in external:

    if name in assignments:

        classification.append(
            {
                "symbol": name,
                "classification":
                    "NOTEBOOK_GLOBAL_ASSIGNMENT",
                "locations": [
                    {
                        "stage":
                            r["stage"],
                        "cell":
                            r["cell"],
                    }
                    for r in assignments[name]
                ],
            }
        )

    elif name in definitions:

        classification.append(
            {
                "symbol": name,
                "classification":
                    "RUNTIME_FUNCTION_AVAILABLE",
                "locations": [
                    {
                        "stage":
                            r["stage"],
                        "cell":
                            r["cell"],
                    }
                    for r in definitions[name]
                ],
            }
        )

    elif name in imports:

        classification.append(
            {
                "symbol": name,
                "classification":
                    "NOTEBOOK_IMPORT_AVAILABLE",
                "locations": [
                    {
                        "stage":
                            r["stage"],
                        "cell":
                            r["cell"],
                    }
                    for r in imports[name]
                ],
            }
        )

    else:

        classification.append(
            {
                "symbol": name,
                "classification":
                    "TRULY_UNRESOLVED",
                "locations": [],
            }
        )


# =============================================================================
# 6. PRINT EXACT FORENSIC RESULT
# =============================================================================

print()
print("RUNTIME SYMBOL FORENSIC")
print("-" * 100)

for item in classification:

    print(
        f"{item['symbol']:<45}"
        f" | {item['classification']}"
    )

    for location in item[
        "locations"
    ]:

        print(
            " " * 4,
            location,
        )


# =============================================================================
# 7. SOURCE SNIPPET FOR EACH TRULY UNRESOLVED SYMBOL
# =============================================================================

truly_unresolved = [
    x
    for x in classification
    if x["classification"]
    == "TRULY_UNRESOLVED"
]


print()
print("=" * 100)
print(
    "TRULY UNRESOLVED SYMBOLS"
)
print("=" * 100)

if not truly_unresolved:

    print(
        "NONE"
    )

else:

    for item in truly_unresolved:

        print()
        print(
            "SYMBOL:",
            item["symbol"],
        )

        print(
            "No exact definition/import/assignment "
            "was found in the selected source notebooks."
        )


# =============================================================================
# 8. WRITE FORENSIC JSON
# =============================================================================

forensic = {
    "cell": 11,
    "selected_runtime_symbols":
        sorted(
            selected_definition_names
        ),
    "external_symbols":
        classification,
    "truly_unresolved":
        [
            x["symbol"]
            for x in truly_unresolved
        ],
    "status":
        (
            "RESOLUTION_REQUIRED"
            if truly_unresolved
            else "NO_UNRESOLVED_SYMBOLS"
        ),
}


FORENSIC_PATH = (
    CELL11_ROOT
    / "cell11a_unresolved_symbol_forensic.json"
)


with open(
    FORENSIC_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        forensic,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 9. FINAL STATUS
# =============================================================================

print()
print("=" * 100)
print(
    "TRACE THE ACE — CELL 11A FINAL STATUS"
)
print("=" * 100)

print(
    "Source indexing                 : PASS"
)

print(
    "Runtime symbol reconstruction   : PASS"
)

print(
    "Dependency classification       : PASS"
)

print(
    "Truly unresolved symbols        :",
    len(truly_unresolved),
)

print(
    "main.py generated               : NO"
)

print(
    "Submission-ready                : NO"
)

print(
    "Forensic JSON                   :",
    FORENSIC_PATH,
)

print("=" * 100)

if truly_unresolved:

    raise RuntimeError(
        "Exact unresolved runtime symbols found. "
        "Use the printed symbol list to resolve the "
        "actual notebook dependencies before generating "
        "main.py."
    )

TRACE THE ACE — SUBMISSION RUNTIME
CELL 11A — UNRESOLVED RUNTIME SYMBOL FORENSIC

RUNTIME SYMBOL FORENSIC
----------------------------------------------------------------------------------------------------
CHAR_WEIGHT                                   | NOTEBOOK_GLOBAL_ASSIGNMENT
     {'stage': 'r1', 'cell': 8}
DEVICE                                        | NOTEBOOK_GLOBAL_ASSIGNMENT
     {'stage': 'modernbert', 'cell': 2}
     {'stage': 'modernbert', 'cell': 6}
LOOKUP_ROOT                                   | NOTEBOOK_GLOBAL_ASSIGNMENT
     {'stage': 'cross_encoder', 'cell': 3}
MATH_WEIGHT                                   | NOTEBOOK_GLOBAL_ASSIGNMENT
     {'stage': 'r1', 'cell': 8}
MAX_EVIDENCE_TOKENS                           | NOTEBOOK_GLOBAL_ASSIGNMENT
     {'stage': 'evidence', 'cell': 5}
MODEL_BATCH_SIZE                              | NOTEBOOK_GLOBAL_ASSIGNMENT
     {'stage': 'cross_encoder', 'cell': 3}
MODEL_NAME                                    | NOTEBOOK_GLOBAL_ASSIGNMENT


RuntimeError: Exact unresolved runtime symbols found. Use the printed symbol list to resolve the actual notebook dependencies before generating main.py.

In [25]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 11B — GLOBAL CONSTANT + DEPENDENCY CLOSURE FORENSIC
# =============================================================================

from __future__ import annotations

import ast
import json
from pathlib import Path


# =============================================================================
# PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT / "Notebooks"
)

CELL11_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell11"
)

CELL11_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

FORENSIC_PATH = (
    CELL11_ROOT
    / "cell11a_unresolved_symbol_forensic.json"
)


NOTEBOOKS = {
    "r0":
        NOTEBOOK_ROOT
        / "04_R0_retrieval_input_builder_FIXED.ipynb",

    "r1":
        NOTEBOOK_ROOT
        / "05_R1_sparse_retrieval.ipynb",

    "r2":
        NOTEBOOK_ROOT
        / "06_R2_dense_retrieval.ipynb",

    "r3":
        NOTEBOOK_ROOT
        / "07_R3 _Candidate_Union_Sparse_Dense_Fusion.ipynb",

    "cross_encoder":
        NOTEBOOK_ROOT
        / "08_cross_encoder_reranking.ipynb",

    "evidence":
        NOTEBOOK_ROOT
        / "09_evidence_pack_builder.ipynb",

    "modernbert":
        NOTEBOOK_ROOT
        / "09b-modernbert-mastery-modalipynb.ipynb",

    "structured":
        NOTEBOOK_ROOT
        / "10_structured_prior_oof.ipynb",
}


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 11B — GLOBAL CONSTANT + DEPENDENCY CLOSURE FORENSIC"
)
print("=" * 100)


# =============================================================================
# 1. LOAD CELL 11A FORENSIC
# =============================================================================

assert FORENSIC_PATH.exists(), (
    f"Missing Cell 11A forensic artifact:\n"
    f"{FORENSIC_PATH}"
)

with open(
    FORENSIC_PATH,
    "r",
    encoding="utf-8",
) as f:

    forensic = json.load(f)


external_symbols = forensic.get(
    "external_symbols",
    []
)

unresolved_names = [
    item["symbol"]
    for item in external_symbols
    if item.get("classification")
    == "NOTEBOOK_GLOBAL_ASSIGNMENT"
]


print()
print(
    "Cell 11A forensic load : PASS"
)

print(
    "Notebook-global symbols :",
    len(unresolved_names)
)


# =============================================================================
# 2. AST SOURCE INDEX
# =============================================================================

assignment_index = {}
definition_index = {}
import_index = {}


for stage, notebook_path in NOTEBOOKS.items():

    assert notebook_path.exists(), (
        f"Missing notebook:\n{notebook_path}"
    )

    with open(
        notebook_path,
        "r",
        encoding="utf-8",
    ) as f:

        notebook = json.load(f)

    for cell_index, cell in enumerate(
        notebook.get("cells", [])
    ):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get("source", [])
        )

        if not source.strip():
            continue

        try:
            tree = ast.parse(
                source
            )
        except SyntaxError:
            continue

        for node in tree.body:

            # -------------------------------------------------------------
            # Assignment
            # -------------------------------------------------------------

            if isinstance(
                node,
                ast.Assign,
            ):

                for target in node.targets:

                    if isinstance(
                        target,
                        ast.Name,
                    ):

                        assignment_index[
                            target.id
                        ] = {
                            "stage": stage,
                            "cell": cell_index,
                            "path": str(
                                notebook_path
                            ),
                            "source": source,
                            "node": node,
                        }

            elif isinstance(
                node,
                ast.AnnAssign,
            ):

                if isinstance(
                    node.target,
                    ast.Name,
                ):

                    assignment_index[
                        node.target.id
                    ] = {
                        "stage": stage,
                        "cell": cell_index,
                        "path": str(
                            notebook_path
                        ),
                        "source": source,
                        "node": node,
                    }

            # -------------------------------------------------------------
            # Definitions
            # -------------------------------------------------------------

            elif isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            ):

                definition_index[
                    node.name
                ] = {
                    "stage": stage,
                    "cell": cell_index,
                    "path": str(
                        notebook_path
                    ),
                    "source": source,
                    "node": node,
                }

            # -------------------------------------------------------------
            # Imports
            # -------------------------------------------------------------

            elif isinstance(
                node,
                ast.Import,
            ):

                for alias in node.names:

                    name = (
                        alias.asname
                        or alias.name.split(".")[0]
                    )

                    import_index[
                        name
                    ] = {
                        "stage": stage,
                        "cell": cell_index,
                        "path": str(
                            notebook_path
                        ),
                        "source": source,
                    }

            elif isinstance(
                node,
                ast.ImportFrom,
            ):

                for alias in node.names:

                    name = (
                        alias.asname
                        or alias.name
                    )

                    import_index[
                        name
                    ] = {
                        "stage": stage,
                        "cell": cell_index,
                        "path": str(
                            notebook_path
                        ),
                        "source": source,
                    }


print(
    "AST source index : PASS"
)


# =============================================================================
# 3. AST LITERAL DETECTION
# =============================================================================

def is_literal_expression(node):

    if node is None:
        return False

    if isinstance(
        node,
        ast.Constant,
    ):
        return True

    if isinstance(
        node,
        (
            ast.List,
            ast.Tuple,
            ast.Set,
        ),
    ):

        return all(
            is_literal_expression(x)
            for x in node.elts
        )

    if isinstance(
        node,
        ast.Dict,
    ):

        return all(
            (
                key is None
                or is_literal_expression(key)
            )
            and is_literal_expression(value)
            for key, value in zip(
                node.keys,
                node.values,
            )
        )

    if isinstance(
        node,
        ast.UnaryOp,
    ):

        return is_literal_expression(
            node.operand
        )

    return False


# =============================================================================
# 4. SYMBOL CLASSIFICATION
# =============================================================================

rows = []

for name in unresolved_names:

    record = assignment_index.get(
        name
    )

    if record is None:

        rows.append(
            {
                "symbol": name,
                "status": "ASSIGNMENT_NOT_FOUND",
                "stage": None,
                "cell": None,
                "literal_assignment": False,
                "source": None,
            }
        )

        continue

    node = record["node"]

    value_node = None

    if isinstance(
        node,
        ast.Assign,
    ):

        value_node = node.value

    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        value_node = node.value

    literal = is_literal_expression(
        value_node
    )

    if literal:

        status = "RUNTIME_CONSTANT"

    else:

        status = (
            "NON_LITERAL_GLOBAL"
        )

    rows.append(
        {
            "symbol": name,
            "status": status,
            "stage": record["stage"],
            "cell": record["cell"],
            "literal_assignment": literal,
            "source": record["source"],
        }
    )


# =============================================================================
# 5. PRINT EVERY SYMBOL — NO TRUNCATION
# =============================================================================

print()
print("=" * 100)
print(
    "FULL NOTEBOOK GLOBAL ASSIGNMENT INVENTORY"
)
print("=" * 100)

for row in rows:

    print()
    print(
        f"SYMBOL : {row['symbol']}"
    )

    print(
        f"STATUS : {row['status']}"
    )

    print(
        f"SOURCE : "
        f"{row['stage']} / cell {row['cell']}"
    )

    print(
        "-" * 80
    )

    print(
        row["source"]
    )


# =============================================================================
# 6. NON-LITERAL GLOBALS
# =============================================================================

non_literal = [
    row
    for row in rows
    if row["status"]
    == "NON_LITERAL_GLOBAL"
]


literal_globals = [
    row
    for row in rows
    if row["status"]
    == "RUNTIME_CONSTANT"
]


missing_assignments = [
    row
    for row in rows
    if row["status"]
    == "ASSIGNMENT_NOT_FOUND"
]


print()
print("=" * 100)
print(
    "GLOBAL CLASSIFICATION"
)
print("=" * 100)

print(
    "Runtime constants       :",
    len(literal_globals),
)

print(
    "Non-literal globals     :",
    len(non_literal),
)

print(
    "Missing assignments     :",
    len(missing_assignments),
)


# =============================================================================
# 7. WRITE FULL JSON
# =============================================================================

OUTPUT_JSON = (
    CELL11_ROOT
    / "cell11b_global_dependency_closure.json"
)

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "cell": "11B",
            "source": "cell11A",
            "runtime_constants": literal_globals,
            "non_literal_globals": non_literal,
            "missing_assignments":
                missing_assignments,
            "status": (
                "READY_FOR_GLOBAL_CLOSURE"
                if not missing_assignments
                else "ASSIGNMENT_DISCOVERY_INCOMPLETE"
            ),
        },
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# =============================================================================
# 8. WRITE HUMAN-READABLE REPORT
# =============================================================================

OUTPUT_TXT = (
    CELL11_ROOT
    / "cell11b_global_dependency_closure.txt"
)

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "TRACE THE ACE — CELL 11B\n"
    )
    f.write(
        "GLOBAL CONSTANT + DEPENDENCY CLOSURE\n"
    )
    f.write("=" * 100)
    f.write("\n\n")

    for row in rows:

        f.write(
            f"SYMBOL : {row['symbol']}\n"
        )

        f.write(
            f"STATUS : {row['status']}\n"
        )

        f.write(
            f"SOURCE : "
            f"{row['stage']} / "
            f"cell {row['cell']}\n"
        )

        f.write(
            "-" * 80
            + "\n"
        )

        f.write(
            row["source"]
            or ""
        )

        f.write(
            "\n\n"
        )


# =============================================================================
# 9. FINAL STATUS
# =============================================================================

print()
print("=" * 100)
print(
    "TRACE THE ACE — CELL 11B FINAL STATUS"
)
print("=" * 100)

print(
    "Cell 11A forensic load       : PASS"
)

print(
    "Notebook AST index           : PASS"
)

print(
    "Global assignment discovery  : PASS"
)

print(
    "Runtime constants discovered :",
    len(literal_globals),
)

print(
    "Non-literal globals          :",
    len(non_literal),
)

print(
    "Missing assignments          :",
    len(missing_assignments),
)

print(
    "main.py generated            : NO"
)

print(
    "Submission ZIP generated     : NO"
)

print(
    "JSON report                  :",
    OUTPUT_JSON,
)

print(
    "Text report                  :",
    OUTPUT_TXT,
)

print("=" * 100)


# =============================================================================
# IMPORTANT:
# Do NOT raise here.
# =============================================================================

print()
print(
    "CELL 11B COMPLETE — FORENSIC ONLY"
)
print(
    "No runtime code was generated."
)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 11B — GLOBAL CONSTANT + DEPENDENCY CLOSURE FORENSIC

Cell 11A forensic load : PASS
Notebook-global symbols : 51
AST source index : PASS

FULL NOTEBOOK GLOBAL ASSIGNMENT INVENTORY

SYMBOL : CHAR_WEIGHT
STATUS : RUNTIME_CONSTANT
SOURCE : r1 / cell 8
--------------------------------------------------------------------------------
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 5 — SESSION-LOCAL RETRIEVAL
# ==============================================================================

import gc
import json
import hashlib
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 5 — SESSION-LOCAL RETRIEVAL")
print("=" * 80)


# =====================================================================

In [26]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 12 — SAFE SUBMISSION ASSET PRUNING
# =============================================================================

from __future__ import annotations

import ast
import json
import os
import shutil
from pathlib import Path


# =============================================================================
# PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission_runtime"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT / "assets"
)

MAIN_PATH = (
    SUBMISSION_ROOT / "main.py"
)

CELL12_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell12"
)

CELL12_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 12 — SAFE SUBMISSION ASSET PRUNING"
)
print("=" * 100)


# =============================================================================
# 1. BASIC CONTRACT
# =============================================================================

assert SUBMISSION_ROOT.exists(), (
    f"Missing submission root:\n{SUBMISSION_ROOT}"
)

assert ASSETS_ROOT.exists(), (
    f"Missing assets root:\n{ASSETS_ROOT}"
)

assert MAIN_PATH.exists(), (
    f"Missing main.py:\n{MAIN_PATH}"
)

print()
print("=" * 100)
print("PATH CONTRACT")
print("=" * 100)

print(
    "Submission root :",
    SUBMISSION_ROOT,
)

print(
    "Assets root     :",
    ASSETS_ROOT,
)

print(
    "main.py         :",
    MAIN_PATH,
)

print(
    "Path contract : PASS"
)


# =============================================================================
# 2. READ MAIN.PY
# =============================================================================

main_text = MAIN_PATH.read_text(
    encoding="utf-8"
)

tree = ast.parse(
    main_text
)

print()
print("=" * 100)
print("MAIN.PY STATIC AUDIT")
print("=" * 100)

print(
    "main.py bytes :",
    MAIN_PATH.stat().st_size,
)

print(
    "AST parse : PASS"
)


# =============================================================================
# 3. EXTRACT EXPLICIT ASSET REFERENCES
# =============================================================================

literal_paths = set()

for node in ast.walk(tree):

    if isinstance(
        node,
        ast.Constant,
    ) and isinstance(
        node.value,
        str,
    ):

        value = node.value.replace(
            "\\",
            "/",
        )

        if (
            "assets" in value.lower()
            or "retrieval" in value.lower()
            or "modernbert" in value.lower()
            or "structured" in value.lower()
            or "tfidf" in value.lower()
        ):
            literal_paths.add(value)


print()
print(
    "Explicit asset-related string literals :",
    len(literal_paths),
)

for value in sorted(literal_paths):
    print(
        " ",
        value,
    )


# =============================================================================
# 4. IMPORT / MODULE AUDIT
# =============================================================================

imports = []

for node in ast.walk(tree):

    if isinstance(
        node,
        ast.Import,
    ):

        for alias in node.names:
            imports.append(
                alias.name
            )

    elif isinstance(
        node,
        ast.ImportFrom,
    ):

        imports.append(
            node.module or ""
        )

print()
print(
    "Python imports :"
)

for item in sorted(
    set(imports)
):
    print(
        " ",
        item,
    )


# =============================================================================
# 5. ACTUAL ASSET TREE INVENTORY
# =============================================================================

def file_size(path: Path) -> int:
    return int(
        path.stat().st_size
    )


all_files = [
    p
    for p in ASSETS_ROOT.rglob("*")
    if p.is_file()
]

total_bytes = sum(
    file_size(p)
    for p in all_files
)

print()
print("=" * 100)
print("CURRENT ASSET INVENTORY")
print("=" * 100)

print(
    "Files :",
    len(all_files),
)

print(
    "Size  :",
    f"{total_bytes / (1024 ** 3):.3f} GB",
)


# =============================================================================
# 6. TOP-LEVEL ASSET SIZE BREAKDOWN
# =============================================================================

top_level = {}

for path in all_files:

    try:
        relative = path.relative_to(
            ASSETS_ROOT
        )
    except ValueError:
        continue

    if len(relative.parts) == 0:
        key = "<root-file>"
    else:
        key = relative.parts[0]

    top_level[key] = (
        top_level.get(key, 0)
        + file_size(path)
    )


print()
print(
    "TOP-LEVEL ASSET SIZE"
)

for name, size in sorted(
    top_level.items(),
    key=lambda x: x[1],
    reverse=True,
):

    print(
        f"{name:30s}"
        f"{size / (1024 ** 3):10.3f} GB"
    )


# =============================================================================
# 7. VERIFY CURRENT MAIN.PY REQUIRED ASSETS
# =============================================================================

required_assets = {
    "structured_model":
        ASSETS_ROOT
        / "structured_prior"
        / "structured_prior_model.joblib",

    "objective_prior":
        ASSETS_ROOT
        / "structured_prior"
        / "objective_prior_stats.parquet",

    "structured_schema":
        ASSETS_ROOT
        / "structured_prior"
        / "structured_feature_schema.json",

    "tfidf_vectorizer":
        ASSETS_ROOT
        / "tfidf"
        / "tfidf_vectorizer.joblib",

    "tfidf_model":
        ASSETS_ROOT
        / "tfidf"
        / "tfidf_model.joblib",
}


for fold in (1, 2, 3, 4):

    required_assets[
        f"modernbert_fold_{fold}"
    ] = (
        ASSETS_ROOT
        / "modernbert"
        / f"production_fold_{fold}"
    )


print()
print("=" * 100)
print("MAIN.PY REQUIRED ASSET CONTRACT")
print("=" * 100)

missing_required = []

for name, path in required_assets.items():

    exists = path.exists()

    print(
        f"{name:30s} : "
        f"{'PASS' if exists else 'MISSING'}"
    )

    if not exists:
        missing_required.append(
            name
        )

assert not missing_required, (
    "Required asset(s) missing: "
    + ", ".join(missing_required)
)

print(
    "Required asset contract : PASS"
)


# =============================================================================
# 8. DETERMINE SAFE PRUNING TARGET
# =============================================================================
#
# IMPORTANT:
#
# The current main.py DOES NOT reference the packaged retrieval tree.
#
# It constructs evidence directly from each runtime transcript.
#
# Therefore the following tree is NOT needed by this main.py:
#
#   assets/retrieval/
#
# We do NOT touch:
#
#   assets/modernbert/
#   assets/structured_prior/
#   assets/tfidf/
#
# =============================================================================

RETRIEVAL_ROOT = (
    ASSETS_ROOT / "retrieval"
)

safe_to_remove = []

if RETRIEVAL_ROOT.exists():

    safe_to_remove.append(
        RETRIEVAL_ROOT
    )


print()
print("=" * 100)
print("SAFE PRUNING DECISION")
print("=" * 100)

if safe_to_remove:

    for path in safe_to_remove:

        size = sum(
            p.stat().st_size
            for p in path.rglob("*")
            if p.is_file()
        )

        print(
            "Candidate:",
            path,
        )

        print(
            "Candidate size:",
            f"{size / (1024 ** 3):.3f} GB",
        )

else:

    print(
        "No retrieval asset directory found."
    )


# =============================================================================
# 9. SAFETY ASSERTION
# =============================================================================

for path in safe_to_remove:

    assert (
        path.name == "retrieval"
    ), (
        "Safety gate refused unexpected "
        f"directory: {path}"
    )

    assert (
        path.parent == ASSETS_ROOT
    ), (
        "Safety gate refused non-direct "
        f"asset directory: {path}"
    )


print()
print(
    "Pruning safety gate : PASS"
)


# =============================================================================
# 10. PERFORM PRUNING
# =============================================================================

removed = []

for path in safe_to_remove:

    shutil.rmtree(
        path
    )

    removed.append(
        str(path)
    )

print()
print("=" * 100)
print("ASSET PRUNING")
print("=" * 100)

if removed:

    for item in removed:
        print(
            "REMOVED:",
            item,
        )

else:

    print(
        "Nothing removed."
    )


# =============================================================================
# 11. POST-PRUNING REQUIRED-ASSET CHECK
# =============================================================================

post_missing = []

for name, path in required_assets.items():

    if not path.exists():

        post_missing.append(
            name
        )

assert not post_missing, (
    "Pruning accidentally removed "
    "a required model asset: "
    + ", ".join(post_missing)
)

print()
print(
    "Post-pruning model assets : PASS"
)


# =============================================================================
# 12. VERIFY RETRIEVAL IS NOT REQUIRED BY MAIN.PY
# =============================================================================

retrieval_references = []

for value in literal_paths:

    normalized = value.lower()

    if "retrieval" in normalized:
        retrieval_references.append(
            value
        )


print()
print(
    "Retrieval references in main.py :",
    len(retrieval_references),
)

for item in retrieval_references:

    print(
        " ",
        item,
    )

assert not retrieval_references, (
    "main.py contains an explicit "
    "retrieval asset reference. "
    "Do not prune retrieval assets."
)

print(
    "Retrieval-reference safety : PASS"
)


# =============================================================================
# 13. FINAL SIZE
# =============================================================================

remaining_files = [
    p
    for p in ASSETS_ROOT.rglob("*")
    if p.is_file()
]

remaining_bytes = sum(
    p.stat().st_size
    for p in remaining_files
)

saved_bytes = (
    total_bytes
    - remaining_bytes
)

print()
print("=" * 100)
print("FINAL ASSET SIZE")
print("=" * 100)

print(
    "Before :",
    f"{total_bytes / (1024 ** 3):.3f} GB",
)

print(
    "After  :",
    f"{remaining_bytes / (1024 ** 3):.3f} GB",
)

print(
    "Saved  :",
    f"{saved_bytes / (1024 ** 3):.3f} GB",
)

print(
    "Remaining files :",
    len(remaining_files),
)


# =============================================================================
# 14. WRITE MANIFEST
# =============================================================================

manifest = {
    "cell": "12",
    "main_py": str(MAIN_PATH),
    "before_bytes": total_bytes,
    "after_bytes": remaining_bytes,
    "saved_bytes": saved_bytes,
    "removed_paths": removed,
    "required_assets": {
        name: str(path)
        for name, path
        in required_assets.items()
    },
    "retrieval_references_in_main": (
        retrieval_references
    ),
    "submission_runtime_model": (
        "manual_self_contained_runtime"
    ),
    "status": "PASS",
}

manifest_path = (
    CELL12_ROOT
    / "cell12_asset_pruning_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 15. FINAL STATUS
# =============================================================================

print()
print("=" * 100)
print(
    "TRACE THE ACE — CELL 12 FINAL STATUS"
)
print("=" * 100)

print(
    "main.py static audit          : PASS"
)

print(
    "Required ModernBERT assets    : PASS"
)

print(
    "Structured model assets       : PASS"
)

print(
    "TF-IDF assets                 : PASS"
)

print(
    "Retrieval reference audit     : PASS"
)

print(
    "Safe retrieval pruning        : PASS"
)

print(
    "Post-pruning asset integrity  : PASS"
)

print(
    "Before size                   :",
    f"{total_bytes / (1024 ** 3):.3f} GB",
)

print(
    "After size                    :",
    f"{remaining_bytes / (1024 ** 3):.3f} GB",
)

print(
    "Saved                         :",
    f"{saved_bytes / (1024 ** 3):.3f} GB",
)

print(
    "Manifest :",
    manifest_path,
)

print("=" * 100)
print(
    "CELL 12 COMPLETE — SAFE PRUNING PASS"
)
print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 12 — SAFE SUBMISSION ASSET PRUNING

PATH CONTRACT
Submission root : D:\Competition\Trace-the-race-local\submission_runtime
Assets root     : D:\Competition\Trace-the-race-local\submission_runtime\assets
main.py         : D:\Competition\Trace-the-race-local\submission_runtime\main.py
Path contract : PASS

MAIN.PY STATIC AUDIT
main.py bytes : 18298
AST parse : PASS

Explicit asset-related string literals : 10
  
    Reconstruct the 27 structural columns used by the saved structured model.

    The exact frozen training feature set is:
      20 aggregation features + 7 derived features.

    CE-specific quantities cannot be reconstructed from the model assets alone,
    so their runtime values are deterministic neutral values (0 where no
    frozen CE score exists).
    
  assets
  modernbert
  structured_feature_columns
  structured_feature_schema.json
  structured_prior
  structured_prior_model.joblib
  tfidf
  tfidf_model.joblib
  tfidf_vectorize

In [27]:
# =============================================================================
# TRACE THE ACE — SUBMISSION RUNTIME
# CELL 13 — SMOKE-TEST ZIP PACKAGING
# =============================================================================

from pathlib import Path
import json
import shutil
import zipfile
import hashlib


PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission_runtime"
)

MAIN_PATH = (
    SUBMISSION_ROOT / "main.py"
)

ASSETS_ROOT = (
    SUBMISSION_ROOT / "assets"
)

ZIP_PATH = (
    PROJECT_ROOT / "submission_smoke_test.zip"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "09C"
    / "submission_runtime"
    / "cell13"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 100)
print(
    "TRACE THE ACE — SUBMISSION RUNTIME"
)
print(
    "CELL 13 — SMOKE-TEST ZIP PACKAGING"
)
print("=" * 100)


# =============================================================================
# 1. BASIC CONTRACT
# =============================================================================

assert SUBMISSION_ROOT.exists()
assert MAIN_PATH.exists()
assert ASSETS_ROOT.exists()

print()
print("=" * 100)
print("PACKAGE INPUT CONTRACT")
print("=" * 100)

print(
    "Submission root :",
    SUBMISSION_ROOT,
)

print(
    "main.py         :",
    MAIN_PATH,
)

print(
    "assets/         :",
    ASSETS_ROOT,
)

print(
    "Input contract : PASS"
)


# =============================================================================
# 2. FORBIDDEN CONTENT CHECK
# =============================================================================

forbidden_names = {
    "data",
    "__pycache__",
    ".ipynb_checkpoints",
}

forbidden_extensions = {
    ".ipynb",
}

forbidden_paths = []

for path in SUBMISSION_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.name in forbidden_names:
        forbidden_paths.append(
            str(path)
        )

    if path.suffix.lower() in forbidden_extensions:
        forbidden_paths.append(
            str(path)
        )


print()
print("=" * 100)
print("FORBIDDEN PACKAGE CONTENT AUDIT")
print("=" * 100)

if forbidden_paths:

    for path in forbidden_paths:
        print(
            "FORBIDDEN:",
            path,
        )

else:

    print(
        "No forbidden files found."
    )

assert not forbidden_paths, (
    "Forbidden files exist inside submission_runtime."
)


# =============================================================================
# 3. REQUIRED MODEL ASSETS
# =============================================================================

required_paths = [
    ASSETS_ROOT
    / "structured_prior"
    / "structured_prior_model.joblib",

    ASSETS_ROOT
    / "structured_prior"
    / "objective_prior_stats.parquet",

    ASSETS_ROOT
    / "structured_prior"
    / "structured_feature_schema.json",

    ASSETS_ROOT
    / "tfidf"
    / "tfidf_vectorizer.joblib",

    ASSETS_ROOT
    / "tfidf"
    / "tfidf_model.joblib",
]


for fold in range(1, 5):

    required_paths.append(
        ASSETS_ROOT
        / "modernbert"
        / f"production_fold_{fold}"
        / "model.safetensors"
    )

    required_paths.append(
        ASSETS_ROOT
        / "modernbert"
        / f"production_fold_{fold}"
        / "config.json"
    )

    required_paths.append(
        ASSETS_ROOT
        / "modernbert"
        / f"production_fold_{fold}"
        / "tokenizer.json"
    )

    required_paths.append(
        ASSETS_ROOT
        / "modernbert"
        / f"production_fold_{fold}"
        / "tokenizer_config.json"
    )

    required_paths.append(
        ASSETS_ROOT
        / "modernbert"
        / f"production_fold_{fold}"
        / "special_tokens_map.json"
    )


missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]


print()
print("=" * 100)
print("REQUIRED ASSET AUDIT")
print("=" * 100)

if missing:

    for path in missing:
        print(
            "MISSING:",
            path,
        )

else:

    print(
        "All required model assets found."
    )

assert not missing, (
    "Required submission assets are missing."
)


# =============================================================================
# 4. ZIP ROOT CONTRACT
# =============================================================================
#
# IMPORTANT:
#
# ZIP ROOT MUST BE:
#
# main.py
# assets/...
#
# NOT:
#
# submission_runtime/main.py
#
# =============================================================================

print()
print("=" * 100)
print("ZIP ROOT CONTRACT")
print("=" * 100)

print(
    "Expected root entries:"
)

print(
    "  main.py"
)

print(
    "  assets/"
)


# =============================================================================
# 5. REMOVE OLD ZIP
# =============================================================================

if ZIP_PATH.exists():

    ZIP_PATH.unlink()

    print()
    print(
        "Removed previous ZIP:",
        ZIP_PATH,
    )


# =============================================================================
# 6. BUILD ZIP
# =============================================================================

print()
print("=" * 100)
print("BUILDING SMOKE-TEST ZIP")
print("=" * 100)

file_count = 0
total_bytes = 0

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True,
) as zf:

    # -------------------------------------------------------------------------
    # main.py
    # -------------------------------------------------------------------------

    zf.write(
        MAIN_PATH,
        arcname="main.py",
    )

    file_count += 1
    total_bytes += MAIN_PATH.stat().st_size

    # -------------------------------------------------------------------------
    # assets
    # -------------------------------------------------------------------------

    for path in ASSETS_ROOT.rglob("*"):

        if not path.is_file():
            continue

        relative = path.relative_to(
            SUBMISSION_ROOT
        )

        zf.write(
            path,
            arcname=str(
                relative
            ),
        )

        file_count += 1
        total_bytes += path.stat().st_size


print(
    "ZIP creation : PASS"
)

print(
    "Files packed :",
    file_count,
)

print(
    "Uncompressed input size :",
    f"{total_bytes / (1024 ** 3):.3f} GB",
)

print(
    "ZIP size :",
    f"{ZIP_PATH.stat().st_size / (1024 ** 3):.3f} GB",
)


# =============================================================================
# 7. ZIP STRUCTURE VALIDATION
# =============================================================================

print()
print("=" * 100)
print("ZIP STRUCTURE VALIDATION")
print("=" * 100)

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as zf:

    names = zf.namelist()

    assert "main.py" in names, (
        "main.py is not at ZIP root."
    )

    assert any(
        name.startswith("assets/")
        for name in names
    ), (
        "assets/ directory missing from ZIP."
    )

    assert not any(
        name.startswith("submission_runtime/")
        for name in names
    ), (
        "Invalid nested submission_runtime/ "
        "directory detected."
    )

    assert not any(
        name.startswith("data/")
        for name in names
    ), (
        "Runtime data was accidentally packaged."
    )

    assert not any(
        name.endswith(".ipynb")
        for name in names
    ), (
        "Notebook accidentally packaged."
    )

    assert not any(
        name.startswith("assets/retrieval/")
        for name in names
    ), (
        "Pruned retrieval assets unexpectedly "
        "present in ZIP."
    )


print(
    "ZIP root contract : PASS"
)

print(
    "main.py at root   : PASS"
)

print(
    "assets/ present   : PASS"
)

print(
    "data/ excluded    : PASS"
)

print(
    "notebooks excluded: PASS"
)

print(
    "retrieval excluded: PASS"
)


# =============================================================================
# 8. ZIP INTEGRITY
# =============================================================================

print()
print("=" * 100)
print("ZIP INTEGRITY TEST")
print("=" * 100)

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as zf:

    bad_file = zf.testzip()

assert bad_file is None, (
    f"Corrupt ZIP member: {bad_file}"
)

print(
    "ZIP CRC/integrity : PASS"
)


# =============================================================================
# 9. MANIFEST
# =============================================================================

manifest = {
    "cell": "13",
    "purpose": "smoke_test_submission_package",
    "zip_path": str(ZIP_PATH),
    "main_py_at_root": True,
    "assets_at_root": True,
    "data_packaged": False,
    "notebooks_packaged": False,
    "retrieval_packaged": False,
    "file_count": file_count,
    "uncompressed_bytes": total_bytes,
    "zip_bytes": ZIP_PATH.stat().st_size,
    "status": "PASS",
}


manifest_path = (
    AUDIT_ROOT
    / "cell13_smoke_test_zip_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# 10. FINAL
# =============================================================================

print()
print("=" * 100)
print(
    "TRACE THE ACE — CELL 13 FINAL STATUS"
)
print("=" * 100)

print(
    "main.py root placement       : PASS"
)

print(
    "Required assets              : PASS"
)

print(
    "Runtime data excluded        : PASS"
)

print(
    "Notebook exclusion           : PASS"
)

print(
    "Retrieval exclusion          : PASS"
)

print(
    "ZIP integrity                : PASS"
)

print(
    "Smoke-test ZIP               :",
    ZIP_PATH,
)

print(
    "ZIP size                     :",
    f"{ZIP_PATH.stat().st_size / (1024 ** 3):.3f} GB",
)

print(
    "Manifest                     :",
    manifest_path,
)

print("=" * 100)
print(
    "CELL 13 COMPLETE — SMOKE-TEST ZIP READY"
)
print("=" * 100)

TRACE THE ACE — SUBMISSION RUNTIME
CELL 13 — SMOKE-TEST ZIP PACKAGING

PACKAGE INPUT CONTRACT
Submission root : D:\Competition\Trace-the-race-local\submission_runtime
main.py         : D:\Competition\Trace-the-race-local\submission_runtime\main.py
assets/         : D:\Competition\Trace-the-race-local\submission_runtime\assets
Input contract : PASS

FORBIDDEN PACKAGE CONTENT AUDIT
No forbidden files found.

REQUIRED ASSET AUDIT
All required model assets found.

ZIP ROOT CONTRACT
Expected root entries:
  main.py
  assets/

BUILDING SMOKE-TEST ZIP
ZIP creation : PASS
Files packed : 31
Uncompressed input size : 2.252 GB
ZIP size : 2.083 GB

ZIP STRUCTURE VALIDATION
ZIP root contract : PASS
main.py at root   : PASS
assets/ present   : PASS
data/ excluded    : PASS
notebooks excluded: PASS
retrieval excluded: PASS

ZIP INTEGRITY TEST
ZIP CRC/integrity : PASS

TRACE THE ACE — CELL 13 FINAL STATUS
main.py root placement       : PASS
Required assets              : PASS
Runtime data excluded    